# NSW Electricity Demand and Spot Price Forecasting

Forecasting the next four 15-minute observations of TOTALDEMAND and RRP.
See README.md for data requirements, methodology, and execution limitations.


In [ ]:
import numpy as np
import scipy
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import os
from statsmodels.tsa.statespace.sarimax import SARIMAX
import os, random, time, copy, warnings
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
warnings.filterwarnings('ignore')
import torch, numpy as np, pandas as pd, matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf


os.makedirs('output/figures', exist_ok=True)

print(f'Package versions: \n')

print(f'numpy {np.__version__}')
print(f'scipy {scipy.__version__}')
print(f'pandas {pd.__version__}')
print(f'seaborn {sns.__version__}')
print(f'scikit-learn {sklearn.__version__}')

# Notebook tested on:

# numpy 1.26.4
# scipy 1.15.1
# pandas 2.2.3
# seaborn 0.13.2
# scikit-learn 1.6.1

In [ ]:
# Functions, place all functions you need here together
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score,  mean_absolute_error
from statsmodels.tsa.stattools import acf
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.graphics.tsaplots import plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from itertools import product
from statsmodels.tsa.statespace.sarimax import SARIMAX
import math
import itertools
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.layers import Dense, Input, Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf

Load data, there are two types of versions here, run the one which is convinient to you.

In [ ]:
# from google.colab import files
# uploaded = files.upload()

Since TOTALDEMAND and RRP share the same SETTLEMENTDATE index, timestamp continuity and duplicated timestamp checks are conducted once at the dataset level. **Would add another method (IQR or z score based on distribution) to clean and compare with subjective threshold.**

In [ ]:
# Load data
df_raw = pd.read_csv("PRICE_AND_DEMAND_train_15min.csv")
df_raw['SETTLEMENTDATE'] = pd.to_datetime(df_raw['SETTLEMENTDATE'])
df_raw = df_raw.set_index('SETTLEMENTDATE').sort_index()

In [ ]:
# Sanity check: off-grid timestamps and duplicates
ts = df_raw.index.to_series()

off_grid = (~ts.dt.minute.isin([0, 15, 30, 45])) | (ts.dt.second != 0)
off_grid = off_grid.to_numpy()

print(f'Off-grid rows: {off_grid.sum()}')
print(df_raw.index[off_grid][:20].tolist())

dup = ts.duplicated(keep=False).to_numpy()
print(f'\nDuplicate timestamps: {dup.sum()}')
print(df_raw.loc[dup].sort_index().head(20))

In [ ]:
# 1) Drop off-grid rows (e.g. 02:20, 06:50)
df_raw = df_raw.loc[~off_grid].copy()

# 2) Drop exact duplicate timestamps if any remain
df_raw = df_raw[~df_raw.index.duplicated(keep='first')]

# 3) Reindex onto the canonical 15-min grid — missing slots become NaN
grid = pd.date_range(df_raw.index.min(), df_raw.index.max(), freq='15min')
df_raw = df_raw.reindex(grid)
df_raw.index.name = 'SETTLEMENTDATE'

n_missing_before = df_raw.isna().any(axis=1).sum()
print(f'Missing slots after reindex: {n_missing_before}')

# 4) Time-based interpolation to fill the missing slots
df_raw = df_raw.interpolate(method='time')

n_missing_after = df_raw.isna().any(axis=1).sum()
print(f'Missing slots after interpolation: {n_missing_after}')

In [ ]:
# Extract variables
y_demand = df_raw['TOTALDEMAND'].astype(float)
y_price  = df_raw['RRP'].astype(float)

# Parameters
STEPS_PER_DAY = 96
HORIZON = 4

TRAIN_DAYS = 28
VAL_DAYS   = 10
TEST_DAYS  = 2

TRAIN_END = TRAIN_DAYS * STEPS_PER_DAY
VAL_END   = (TRAIN_DAYS + VAL_DAYS) * STEPS_PER_DAY

# Split
y_demand_train = y_demand.iloc[:TRAIN_END].copy()
y_demand_val   = y_demand.iloc[TRAIN_END:VAL_END].copy()
y_demand_test  = y_demand.iloc[VAL_END:].copy()

y_price_train = y_price.iloc[:TRAIN_END].copy()
y_price_val   = y_price.iloc[TRAIN_END:VAL_END].copy()
y_price_test  = y_price.iloc[VAL_END:].copy()

print(f'Train:      {len(y_demand_train)} steps ({len(y_demand_train)//96} days)')
print(f'Validation: {len(y_demand_val)} steps ({len(y_demand_val)//96} days)')
print(f'Test:       {len(y_demand_test)} steps ({len(y_demand_test)//96} days)')
print(f'Total:      {len(y_demand_train) + len(y_demand_val) + len(y_demand_test)} steps')

In [ ]:
demand_summary = pd.DataFrame({
    'Mean': [y_demand_train.mean()],
    'Median': [y_demand_train.median()],
    'Std': [y_demand_train.std()],
    'Variance': [y_demand_train.var()],
    'Min': [y_demand_train.min()],
    'Q1': [y_demand_train.quantile(0.25)],
    'Q3': [y_demand_train.quantile(0.75)],
    'Max': [y_demand_train.max()],
    'Skewness': [y_demand_train.skew()],
    'Kurtosis': [y_demand_train.kurtosis()]
}).T

demand_summary.columns = ['TOTALDEMAND']

# Display with plain decimal format (no scientific notation)
display(demand_summary.style.format('{:,.3f}'))

In [ ]:
print(f'Raw demand stats:')
print(f'  99th percentile: {y_demand_train.quantile(0.99):.0f}')
print(f'  Values between 15k–50k: {((y_demand_train > 15000) & (y_demand_train < 50000)).sum()}')
print(f'  Values > 50k: {(y_demand_train > 50000).sum()}') ###important###

Visualising an outlier threshold detection

In [ ]:
# Zoom around the outlier — single plot
mask = y_demand_train > 50000
outlier_time = y_demand_train.index[mask][0]
window = pd.Timedelta(hours=3)
sl = slice(outlier_time - window, outlier_time + window)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(y_demand_train.loc[sl].index, y_demand_train.loc[sl].values,
        'o-', color='tab:blue', markersize=5, linewidth=1.2)
ax.scatter([outlier_time], [y_demand_train.loc[outlier_time]],
           color='red', marker='o', s=100, zorder=5, label='Outlier')
ax.set_title(f'Zoom: ±3h around {outlier_time}')
ax.set_ylabel('TOTALDEMAND (MW)')
ax.set_xlabel('SETTLEMENTDATE')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
plt.setp(ax.get_xticklabels(), rotation=20, ha='right')

plt.tight_layout()
plt.show()


Outlier detection attempt: Tukey IQR rule

In [ ]:
q1, q3 = y_demand_train.quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
iqr_mask = (y_demand_train < lower) | (y_demand_train > upper)
print(f'IQR rule flags {iqr_mask.sum()} points '
      f'({100*iqr_mask.sum()/len(y_demand_train):.1f}% of training set) — too many.')
print(f'Lower fence: {lower:.0f} MW    Upper fence: {upper:.0f} MW')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: full-scale boxplot — the bad value dominates the axis ---
ax = axes[0]
ax.boxplot(y_demand_train.dropna().values, vert=True,
           flierprops=dict(marker='o', markerfacecolor='red',
                          markeredgecolor='red', markersize=4, alpha=0.6))
DEMAND_UPPER = 50000

ax.axhline(DEMAND_UPPER if 'DEMAND_UPPER' in dir() else 50000,
           color='orange', linestyle='--', alpha=0.7, label='Hard cap = 50,000')
ax.set_title('Boxplot of training demand (full scale)')
ax.set_ylabel('TOTALDEMAND (MW)')
ax.set_xticklabels(['y_demand_train'])
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

# --- Right: zoomed boxplot — shows the many "false" outliers near the peaks ---
ax = axes[1]
ax.boxplot(y_demand_train.dropna().values, vert=True,
           flierprops=dict(marker='o', markerfacecolor='red',
                           markeredgecolor='red', markersize=4, alpha=0.6))
ax.set_ylim(y_demand_train.min() * 0.95, 16000)  # zoom to plausible range
ax.set_title('Boxplot zoomed in — IQR over-flags normal daily peaks')
ax.set_ylabel('TOTALDEMAND (MW)')
ax.set_xticklabels(['y_demand_train'])
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()



# Final approach: hard cap at 50,000 MW
#
# NSW1 demand historically ranges ~5,500–14,500 MW. 50,000 MW is far above
# any plausible real peak and far below the magnitude of obvious sensor
# errors, so it isolates only the one truly impossible value.

In [ ]:
DEMAND_UPPER = 50000

def clean_demand(y, upper=DEMAND_UPPER):
    y_clean = y.copy()
    mask = y_clean > upper
    print(f'Demand implausible values > {upper}: {mask.sum()}')
    y_clean = y_clean.mask(mask, np.nan)
    y_clean = y_clean.interpolate(method='time')
    return y_clean

yc_demand_train = clean_demand(y_demand_train)
yc_demand_val   = clean_demand(y_demand_val)
yc_demand_test  = clean_demand(y_demand_test)

Some observations of RRP before cleaning. Including distribution, volatility clustering and outlier analysis.

Visualizations for Original RRP.
The following plots shows the raw and symlog RRP over time.
Symlog scale is used because RRP may contain large positive spikes and negative prices.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(y_price_train)
axes[0].set_title("Original RRP")

axes[1].plot(y_price_train)
axes[1].set_yscale("symlog", linthresh=100)
axes[1].set_title("Original RRP on Symlog Y-axis")

plt.tight_layout()
plt.show()

I suspect there is volatility clustering in the price because of the spikes are relatively closed to each other, so I did some tests to find out. Since RRP can be negative, I used squared return on price. **Could only use the last graph below.** Plan is to do this observation, together with distribution (above) and ACF (later) before data cleaning to identify whether large spikes should be removed.

In [ ]:
# # Still trying to justify the arcsine transformation although it has nothing to
# # do with the result.
# rrp_transformed = np.arcsinh(y_price_train / 100)


# Calculate return
rrp_change = y_price_train.diff().dropna()

# Squared return/change
rrp_squared_return = rrp_change ** 2

# Absolute return/change
rrp_abs_return = np.abs(rrp_change)

# Plot Original RRP, return, squared return

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(y_price_train)
axes[0].set_title("Original RRP")
axes[0].set_ylabel("Original RRP")

axes[1].plot(rrp_change)
axes[1].set_title("Change in Original RRP")
axes[1].set_ylabel("Change")

axes[2].plot(rrp_squared_return)
axes[2].set_title("Squared Change in Original RRP")
axes[2].set_ylabel("Squared Change")

plt.tight_layout()
plt.show()

From the grapgh we can see some isolated spikes sit relatively close to each other, such as between 13/01/26 and 17/01/26, not likely to be volatility clustering. However, we do ACF to this period to further confirm if there is volitility clustering.

In [ ]:
# Calculate return of RRP
rrp_change = y_price_train.diff().dropna()

# Square and absolute the series
squared_change = rrp_change ** 2


fig, ax = plt.subplots(figsize=(14, 5))

plot_acf(
    squared_change,
    lags=1632,
    ax=ax,
    alpha=0.05,   # 95% confidence interval
    zero=True
)

ax.set_xlim(0, 1632)
ax.set_title("ACF of Squared Return: Lags 0 to 1632")
ax.set_xlabel("Lag")
ax.set_ylabel("ACF")

plt.show()

ACF does not dies off gradually, so no volatility clustering is observed.

In [ ]:
# RRP summary statistics table

# This table summarises the central tendency, volatility, tail behaviour and extreme values of RRP.

rrp_summary = pd.DataFrame({
    'Mean': [y_price_train.mean()],
    'Median': [y_price_train.median()],
    'Std': [y_price_train.std()],
    'Variance': [y_price_train.var()],
    'Min': [y_price_train.min()],
    'Q1': [y_price_train.quantile(0.25)],
    'Q3': [y_price_train.quantile(0.75)],
    'Max': [y_price_train.max()],
    'Skewness': [y_price_train.skew()],
    'Kurtosis': [y_price_train.kurtosis()]
}).T

rrp_summary.columns = ['RRP']
rrp_summary = rrp_summary.round(3)

display(rrp_summary)

In [ ]:
#RRP histogram / distribution

plt.figure(figsize=(12, 5))
plt.hist(y_price_train, bins=50, edgecolor='black')
plt.title('RRP Distribution')
plt.xlabel('RRP ($/MWh)')
plt.ylabel('Frequency')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'RRP skewness = {y_price_train.skew():.3f}')
print(f'RRP kurtosis = {y_price_train.kurtosis():.3f}')

# # Clipped histogram for visualisation only
# lower_clip = yc_price.quantile(0.01)
# upper_clip = yc_price.quantile(0.99)
# yc_price_clipped = yc_price.clip(lower=lower_clip, upper=upper_clip)

# plt.figure(figsize=(12, 5))
# plt.hist(yc_price_clipped, bins=50, edgecolor='black')
# plt.title('RRP Distribution after 1%–99% Clipping for Visualisation')
# plt.xlabel('RRP ($/MWh)')
# plt.ylabel('Frequency')
# plt.grid(alpha=0.3)
# plt.tight_layout()
# plt.show()

# print(f'1% quantile = {lower_clip:.3f}')
# print(f'99% quantile = {upper_clip:.3f}')

In [ ]:
# RRP spike threshold and spike count

# IQR rule is used to define high price spikes and low price outliers.
# Also because the distribution is not a normal distribution so z score may not work well
# Negative price observations are counted separately because they are meaningful in electricity markets.

Q1_p = y_price_train.quantile(0.25)
Q3_p = y_price_train.quantile(0.75)
IQR_p = Q3_p - Q1_p

upper_spike_threshold = Q3_p + 1.5 * IQR_p
lower_spike_threshold = Q1_p - 1.5 * IQR_p

high_spike_mask = y_price_train > upper_spike_threshold
low_spike_mask = y_price_train < lower_spike_threshold
negative_price_mask = y_price_train < 0

rrp_spike_summary = pd.DataFrame({
    'Metric': [
        'Q1',
        'Q3',
        'IQR',
        'Upper spike threshold',
        'Lower spike threshold',
        'Number of high price spikes',
        'Number of low price outliers',
        'Number of negative price observations',
        'High spike percentage',
        'Low outlier percentage',
        'Negative price percentage'
    ],
    'Value': [
        Q1_p,
        Q3_p,
        IQR_p,
        upper_spike_threshold,
        lower_spike_threshold,
        high_spike_mask.sum(),
        low_spike_mask.sum(),
        negative_price_mask.sum(),
        high_spike_mask.mean() * 100,
        low_spike_mask.mean() * 100,
        negative_price_mask.mean() * 100
    ]
})

rrp_spike_summary['Value'] = rrp_spike_summary['Value'].round(3)

display(rrp_spike_summary)

The RRP distribution is highly right-skewed, with a maximum of 20,299.98 compared with a median of 59.845. The IQR rule flags large positive RRP values as extreme observations. However, because electricity prices can naturally spike, these observations were not automatically treated as data errors. Instead, an inverse hyperbolic sine transformation was used for modelling to reduce the influence of extreme values while preserving their information.

In [ ]:
# Transformed RRP, both testing and val sets are consistent with training set.
yc_price_train = np.arcsinh(y_price_train / 100)
yc_price_val = np.arcsinh(y_price_val / 100)
yc_price_test = np.arcsinh(y_price_test / 100)

# # Below not necessary
# # adding clean-price
# y_price_train = y_price_train.copy()
# y_price_val   = y_price_val.copy()
# y_price_test  = y_price_test.copy()

# print('\nDemand NaNs after cleaning:')
# print(f'  Train: {yc_demand_train.isna().sum()}')
# print(f'  Val:   {yc_demand_val.isna().sum()}')
# print(f'  Test:  {yc_demand_test.isna().sum()}')

# print('\nRRP NaNs:')
# print(f'  Train: {yc_price_train.isna().sum()}')
# print(f'  Val:   {yc_price_val.isna().sum()}')
# print(f'  Test:  {yc_price_test.isna().sum()}')

Use yc_price_train for "cleaned" RRP data. Later in the modelling section we can use lower = y_train.quantile(0.005)
upper = y_train.quantile(0.995)

yc_train_capped = y_train.clip(lower, upper)
yc_val_capped = y_val.clip(lower, upper) to compare results with transformed RRP.

Comparison between transformed and Original RRP, both on symlog scale. Characters are preserved. Note the scales are different.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(y_price_train)
axes[0].set_yscale("symlog", linthresh=100)
axes[0].set_title("Original RRP on Symlog Scale")
axes[0].set_ylabel("RRP")

axes[1].plot(yc_price_train)
axes[1].set_title("Transformed RRP: asinh(RRP / 100)")
axes[1].set_ylabel("asinh(RRP / 100)")

plt.tight_layout()
plt.show()

In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(16, 8))

# --- (0,0) Raw Demand ---
ax = axes[0, 0]
ax.plot(y_demand_train.index, y_demand_train.values,
        color='tab:red', alpha=0.8, linewidth=1.0)
ax.set_title('y_demand_train (raw)')
ax.set_ylabel('TOTALDEMAND (MW)')
ax.grid(True, alpha=0.3)

# --- (0,1) Cleaned Demand ---
ax = axes[0, 1]
ax.plot(yc_demand_train.index, yc_demand_train.values,
        color='tab:blue', alpha=0.8, linewidth=1.0)
ax.set_title('yc_demand_train (cleaned)')
ax.set_ylabel('TOTALDEMAND (MW)')
ax.grid(True, alpha=0.3)

# --- (1,0) Raw Price ---
ax = axes[1, 0]
ax.plot(y_price_train.index, y_price_train.values,
        color='tab:red', alpha=0.8, linewidth=1.0)
ax.set_title('y_price_train (raw)')
ax.set_ylabel('RRP ($/MWh)')
ax.set_xlabel('SETTLEMENTDATE')
ax.grid(True, alpha=0.3)

# --- (1,1) Transformed Price ---
ax = axes[1, 1]
ax.plot(yc_price_train.index, yc_price_train.values,
        color='tab:blue', alpha=0.8, linewidth=1.0)
ax.set_title('yc_price_train (arcsinh transformed)')
ax.set_ylabel('arcsinh(RRP/100)')
ax.set_xlabel('SETTLEMENTDATE')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Visualizations for demand（after cleaning）- on training data**

In [ ]:
# Create some features

TOD = yc_demand_train.index.hour * 4 + yc_demand_train.index.minute // 15
TOD_TICKS  = range(0, 96, 8)
TOD_LABELS = [f'{h}:00' for h in range(0, 24, 2)]
DOW = yc_demand_train.index.dayofweek
is_weekend = DOW.isin([5, 6])

In [ ]:
# Demand rolling stability plot to investigate demand fluctuations
# Rolling mean captures average demand changes, while rolling standard deviation captures changing volatility

rolling_window = 96

demand_rolling_mean = yc_demand_train.rolling(window=rolling_window).mean()
demand_rolling_std  = yc_demand_train.rolling(window=rolling_window).std()

plt.figure(figsize=(12, 4))
plt.plot(yc_demand_train.index, yc_demand_train, linewidth=0.5, alpha=0.4, label='Total Demand')
plt.plot(yc_demand_train.index, demand_rolling_mean, linewidth=1.5, label='1-day rolling mean')
plt.yscale('linear')
plt.title('Total Demand Rolling Mean')
plt.xlabel('Date / Time')
plt.ylabel('Total Demand (MW)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(yc_demand_train.index, demand_rolling_std, linewidth=1.2)
plt.yscale('linear')
plt.title('Total Demand 1-Day Rolling Standard Deviation')
plt.xlabel('Date / Time')
plt.ylabel('Rolling Std of Demand (MW)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# The average demand changes in a day with one standard deviation interval

fig, ax = plt.subplots(figsize=(8, 5))

prof = yc_demand_train.groupby(TOD).agg(['mean', 'std'])

ax.plot(prof.index, prof['mean'], color='tab:blue', lw=1.5, label='Mean')

ax.fill_between(
    prof.index,
    prof['mean'] - prof['std'],
    prof['mean'] + prof['std'],
    alpha=0.2,
    color='tab:blue',
    label='±1 Std'
)

ax.set(
    xlabel='Time of Day',
    ylabel='Demand (MW)',
    title='Intraday Demand Pattern',
    xlim=(0, 95)
)

ax.set_xticks(TOD_TICKS)
ax.set_xticklabels(TOD_LABELS)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.show()

In [ ]:
#The average demand changes on weekday and weekend respectively with one standard deviation interval

fig, ax = plt.subplots(figsize=(8, 5))

for mask, lab, c in [
    (~is_weekend, 'Weekday', 'tab:blue'),
    (is_weekend,  'Weekend',  'tab:red')
]:
    prof = yc_demand_train[mask].groupby(TOD[mask]).agg(['mean', 'std'])

    ax.plot(prof.index, prof['mean'], color=c, lw=1.5, label=f'{lab} mean')

    ax.fill_between(
        prof.index,
        prof['mean'] - prof['std'],
        prof['mean'] + prof['std'],
        alpha=0.1,
        color=c
    )

ax.set(
    xlabel='Time of Day',
    ylabel='Demand (MW)',
    title='Demand — Weekday vs Weekend',
    xlim=(0, 95)
)

ax.set_xticks(TOD_TICKS)
ax.set_xticklabels(TOD_LABELS)
ax.legend()
ax.grid(alpha=0.3)
plt.show()

In [ ]:
# Demand day-of-week pattern
# which helps us compare demand across different days of the week

day_name = yc_demand_train.index.day_name()

day_order = [
    'Monday', 'Tuesday', 'Wednesday',
    'Thursday', 'Friday', 'Saturday', 'Sunday'
]

demand_day_stats = (
    yc_demand_train.groupby(day_name)
    .agg(['mean', 'median', 'std', 'min', 'max'])
    .reindex(day_order)
)

display(demand_day_stats.round(3))

plt.figure(figsize=(10, 5))
plt.bar(demand_day_stats.index, demand_day_stats['mean'])
plt.title('Average Demand by Day of Week')
plt.xlabel('Day of Week')
plt.ylabel('Average Demand (MW)')
plt.xticks(rotation=30)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

boxplot_demand = [
    yc_demand_train[day_name == day]
    for day in day_order
]

plt.figure(figsize=(10, 5))
plt.boxplot(boxplot_demand, labels=day_order)
plt.yscale('linear')
plt.title('Demand Distribution by Day of Week')
plt.xlabel('Day of Week')
plt.ylabel('Demand (MW)')
plt.xticks(rotation=30)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

**Visualizations for RRP（after transformed）- on training data**

The time features used below are based only on the timestamp index.

Although TOTALDEMAND and RRP are processed differently, these operations only change the data values, not the timestamps. TOTALDEMAND is cleaned by replacing an implausible value and interpolating it, while RRP is transformed using `asinh(RRP / 100)` to reduce the effect of extreme price spikes.

Because both series are split from the same aligned 15-minute dataset, cleaned TOTALDEMAND and transformed RRP still share the same training timestamp index. Therefore, the same `TOD`, `is_weekend`, and `day_name` variables are valid for the demand and RRP visualisations.

In [ ]:
# RRP rolling stability plot on transformed training data
# yc_price_train is transformed as asinh(RRP / 100), so it is not in the original $/MWh scale.

rolling_window = 96  # 1 day = 96 observations at 15-minute frequency

rrp_rolling_mean = yc_price_train.rolling(window=rolling_window).mean()
rrp_rolling_std  = yc_price_train.rolling(window=rolling_window).std()

plt.figure(figsize=(12, 4))
plt.plot(
    yc_price_train.index,
    yc_price_train,
    linewidth=0.5,
    alpha=0.35,
    label='Transformed RRP'
)
plt.plot(
    yc_price_train.index,
    rrp_rolling_mean,
    linewidth=1.5,
    label='1-day rolling mean'
)
plt.title('Transformed RRP Rolling Mean on Training Data')
plt.xlabel('Date / Time')
plt.ylabel('Transformed RRP: asinh(RRP / 100)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(
    'output/figures/transformed_rrp_rolling_mean_train.png',
    dpi=300,
    bbox_inches='tight'
)
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(
    yc_price_train.index,
    rrp_rolling_std,
    linewidth=1.2
)
plt.title('Transformed RRP 1-Day Rolling Standard Deviation on Training Data')
plt.xlabel('Date / Time')
plt.ylabel('Rolling Std of asinh(RRP / 100)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(
    'output/figures/transformed_rrp_rolling_std_train.png',
    dpi=300,
    bbox_inches='tight'
)
plt.show()

In [ ]:
# RRP intraday pattern on transformed training data
# This plot shows the average transformed RRP at each 15-minute time-of-day slot.

fig, ax = plt.subplots(figsize=(8, 5))

rrp_intraday_prof = yc_price_train.groupby(TOD).agg(['mean', 'std'])

ax.plot(
    rrp_intraday_prof.index,
    rrp_intraday_prof['mean'],
    lw=1.5,
    label='Mean'
)

ax.fill_between(
    rrp_intraday_prof.index,
    rrp_intraday_prof['mean'] - rrp_intraday_prof['std'],
    rrp_intraday_prof['mean'] + rrp_intraday_prof['std'],
    alpha=0.2,
    label='±1 Std'
)

ax.set(
    xlabel='Time of Day',
    ylabel='Transformed RRP: asinh(RRP / 100)',
    title='Intraday Transformed RRP Pattern on Training Data',
    xlim=(0, 95)
)

ax.set_xticks(TOD_TICKS)
ax.set_xticklabels(TOD_LABELS)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(
    'output/figures/transformed_rrp_intraday_pattern_train.png',
    dpi=300,
    bbox_inches='tight'
)
plt.show()

In [ ]:
# RRP weekday vs weekend pattern on transformed training data
# This compares whether the intraday transformed price pattern differs between weekdays and weekends.

fig, ax = plt.subplots(figsize=(8, 5))

for mask, lab in [
    (~is_weekend, 'Weekday'),
    (is_weekend,  'Weekend')
]:
    prof = yc_price_train[mask].groupby(TOD[mask]).agg(['mean', 'std'])

    ax.plot(
        prof.index,
        prof['mean'],
        lw=1.5,
        label=f'{lab} mean'
    )

    ax.fill_between(
        prof.index,
        prof['mean'] - prof['std'],
        prof['mean'] + prof['std'],
        alpha=0.1
    )

ax.set(
    xlabel='Time of Day',
    ylabel='Transformed RRP: asinh(RRP / 100)',
    title='Transformed RRP — Weekday vs Weekend Pattern on Training Data',
    xlim=(0, 95)
)

ax.set_xticks(TOD_TICKS)
ax.set_xticklabels(TOD_LABELS)
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(
    'output/figures/transformed_rrp_weekday_weekend_pattern_train.png',
    dpi=300,
    bbox_inches='tight'
)
plt.show()

In [ ]:
# RRP day-of-week pattern on transformed training data
# This table and plots compare the level and dispersion of transformed RRP across different days of the week.

rrp_day_stats = (
    yc_price_train.groupby(day_name)
    .agg(['mean', 'median', 'std', 'min', 'max'])
    .reindex(day_order)
)

display(rrp_day_stats.round(3))

plt.figure(figsize=(10, 5))
plt.bar(rrp_day_stats.index, rrp_day_stats['mean'])
plt.title('Average Transformed RRP by Day of Week on Training Data')
plt.xlabel('Day of Week')
plt.ylabel('Average Transformed RRP: asinh(RRP / 100)')
plt.xticks(rotation=30)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(
    'output/figures/transformed_rrp_average_by_day_of_week_train.png',
    dpi=300,
    bbox_inches='tight'
)
plt.show()

boxplot_rrp = [
    yc_price_train[day_name == day]
    for day in day_order
]

plt.figure(figsize=(10, 5))
plt.boxplot(boxplot_rrp, tick_labels=day_order)
plt.title('Transformed RRP Distribution by Day of Week on Training Data')
plt.xlabel('Day of Week')
plt.ylabel('Transformed RRP: asinh(RRP / 100)')
plt.xticks(rotation=30)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(
    'output/figures/transformed_rrp_distribution_by_day_of_week_train.png',
    dpi=300,
    bbox_inches='tight'
)
plt.show()

**Cross-series relationship (on training data) - Cleaned Demand vs Transformed RRP**

In [ ]:
# Cross-series relationship: demand-price scatterplot (training data)
# yc_demand_train is cleaned demand.
# yc_price_train is transformed RRP = asinh(RRP / 100), not the original $/MWh price.

plt.figure(figsize=(12, 5))
plt.scatter(yc_demand_train, yc_price_train, alpha=0.3, s=5)

plt.title('Cleaned Demand vs Transformed RRP — Scatterplot on Training Data')
plt.xlabel('Cleaned Demand (MW)')
plt.ylabel('Transformed RRP: asinh(RRP / 100)')

plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(
    'output/figures/cleaned_demand_vs_transformed_rrp_scatter_train.png',
    dpi=300,
    bbox_inches='tight'
)
plt.show()

corr = np.corrcoef(yc_demand_train, yc_price_train)[0, 1]
print(f'Pearson correlation between cleaned demand and transformed RRP = {corr:.3f}')

In [ ]:
# Cross-series relationship: lagged cleaned demand and current transformed RRP correlation
# Positive lag means demand is lagged: corr(Demand_{t-lag}, Transformed RRP_t).

max_lag = 96  # up to 1 day

lag_corrs = []

for lag in range(0, max_lag + 1):
    lagged_demand = yc_demand_train.shift(lag)

    aligned = pd.concat(
        [
            lagged_demand.rename('lagged_demand'),
            yc_price_train.rename('transformed_rrp')
        ],
        axis=1
    ).dropna()

    corr_value = aligned['lagged_demand'].corr(aligned['transformed_rrp'])
    lag_corrs.append(corr_value)

lag_corrs = pd.Series(
    lag_corrs,
    index=range(0, max_lag + 1),
    name='correlation'
)

plt.figure(figsize=(12, 4))
plt.plot(lag_corrs.index, lag_corrs.values, linewidth=1.5)
plt.axhline(0, linestyle='--', linewidth=1)

plt.title('Lagged Cleaned Demand and Current Transformed RRP Correlation on Training Data')
plt.xlabel('Demand Lag')
plt.ylabel('Correlation: corr(Demand$_{t-lag}$, Transformed RRP$_t$)')

plt.axvline(4, linestyle=':', linewidth=1, label='1 hour')
plt.axvline(96, linestyle=':', linewidth=1, label='1 day')

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(
    'output/figures/lagged_cleaned_demand_current_transformed_rrp_correlation_train.png',
    dpi=300,
    bbox_inches='tight'
)
plt.show()

lag_corr_table = pd.DataFrame({
    'Lag': lag_corrs.index,
    'Lag in hours': lag_corrs.index / 4,
    'Correlation': lag_corrs.values
})

display(
    lag_corr_table
    .assign(abs_corr=lambda x: x['Correlation'].abs())
    .sort_values('abs_corr', ascending=False)
    .drop(columns='abs_corr')
    .head(10)
    .round(3)
)

In [ ]:
# Cross-series relationship: rolling correlation between cleaned demand and transformed RRP
# This checks whether the demand-price relationship is stable over time.

rolling_corr_window = 96  # 1 day

rolling_corr = yc_demand_train.rolling(window=rolling_corr_window).corr(yc_price_train)

plt.figure(figsize=(12, 4))
plt.plot(rolling_corr.index, rolling_corr, linewidth=1.2)
plt.axhline(0, linestyle='--', linewidth=1)

plt.title('1-Day Rolling Correlation Between Cleaned Demand and Transformed RRP on Training Data')
plt.xlabel('Date / Time')
plt.ylabel('Rolling Correlation')

plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(
    'output/figures/rolling_correlation_cleaned_demand_transformed_rrp_train.png',
    dpi=300,
    bbox_inches='tight'
)
plt.show()

rolling_corr_summary = pd.DataFrame({
    'Mean rolling correlation': [rolling_corr.mean()],
    'Median rolling correlation': [rolling_corr.median()],
    'Std of rolling correlation': [rolling_corr.std()],
    'Min rolling correlation': [rolling_corr.min()],
    'Max rolling correlation': [rolling_corr.max()]
}).T

rolling_corr_summary.columns = ['Value']

display(rolling_corr_summary.round(3))

This is the part to determine seasonality of TOTALDEMAND and RRP by observing ACF, after getting the cleaned data, PACF is also plotted for future use.

Do Totaldemand check first.

In [ ]:
# 96 lags = 1 day
# 672 lags = 1 week
# 1344 lags = 2 weeks
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

plot_acf(yc_demand_train, lags=96, ax=axes[0])
axes[0].set_title("ACF of Cleaned TOTALDEMAND up to 1 day")

plot_pacf(yc_demand_train, lags=96, ax=axes[1], method="ywm")
axes[1].set_title("PACF of Cleaned TOTALDEMAND up to 1 day")

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

plot_acf(yc_demand_train, lags=672, ax=axes[0])
axes[0].set_title("ACF of Cleaned TOTALDEMAND up to 1 week")

plot_pacf(yc_demand_train, lags=672, ax=axes[1], method="ywm")
axes[1].set_title("PACF of Cleaned TOTALDEMAND up to 1 week")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

plot_acf(yc_demand_train, lags=1344, ax=axes[0])
axes[0].set_title("ACF of Cleaned TOTALDEMAND up to 2 weeks")

plot_pacf(yc_demand_train, lags=1344, ax=axes[1], method="ywm")
axes[1].set_title("PACF of Cleaned TOTALDEMAND up to 2 weeks")

plt.tight_layout()
plt.show()


Conclusion: ACF chart clearly shows spikes at 96 in up to 1 day plot and m, 2m, 3m... where m=96 in up to 1 week plot and dies down, this suggested daily seasonality for TOTALDEMAND. No spikes at lag 672 and 1344 so no weekly seasonality was observed.

Now check seasonality in RRP.

In [ ]:
# 96 lags = 1 day
# 672 lags = 1 week
# 1344 lags = 2 weeks
fig, axes = plt.subplots(6, 1, figsize=(14, 8))

plot_acf(yc_price_train, lags=96, ax=axes[0])
axes[0].set_title("ACF of Cleaned RRP up to 1 day")

plot_pacf(yc_price_train, lags=96, ax=axes[1], method="ywm")
axes[1].set_title("PACF of Cleaned RRP up to 1 day")

plot_acf(yc_price_train, lags=672, ax=axes[2])
axes[2].set_title("ACF of Cleaned RRP up to 1 week")

plot_pacf(yc_price_train, lags=672, ax=axes[3], method="ywm")
axes[3].set_title("PACF of Cleaned RRP up to 1 week")

plot_acf(yc_price_train, lags=1344, ax=axes[4])
axes[4].set_title("ACF of Cleaned RRP up to 2 weeks")

plot_pacf(yc_price_train, lags=1344, ax=axes[5], method="ywm")
axes[5].set_title("PACF of Cleaned RRP up to 2 weeks")
plt.tight_layout()
plt.show()

The results are exactly same as TOTALDEMAND, which have spikes at 96 in up to 1 day plot and m, 2m, 3m... where m=96 in up to 1 week plot and dies down, this suggested daily seasonality in RRP. No spikes at lag 672 and 1344 so no weekly seasonality was observed. We can also find out that the transformation did a good job to avoid large spikes disturbing ACF results.

Note in future parts, for PACF, shorter ranges such as 96 or 192 are usually inspected first.

In [ ]:
# Seasonal naive with M=96 (daily). At each origin t, forecast y[t+h]=y[t+h-M] for h=1..4.
M = 96
H = 4
all_demand = pd.concat([yc_demand_train, yc_demand_val, yc_demand_test])

# val
val_errs = []
snaive_h4_val = pd.Series(index=yc_demand_val.index, dtype=float)
for i in range(len(yc_demand_val) - H):
    origin = TRAIN_END + i
    fc  = np.array([all_demand.iloc[origin + h - M] for h in range(1, H+1)])
    act = np.array([all_demand.iloc[origin + h]     for h in range(1, H+1)])
    val_errs.append(np.sqrt(np.mean((fc - act) ** 2)))
    snaive_h4_val.iloc[i + H] = fc[H - 1]

# test
test_errs = []
snaive_h4_test = pd.Series(index=yc_demand_test.index, dtype=float)
for i in range(len(yc_demand_test) - H):
    origin = VAL_END + i
    fc  = np.array([all_demand.iloc[origin + h - M] for h in range(1, H+1)])
    act = np.array([all_demand.iloc[origin + h]     for h in range(1, H+1)])
    test_errs.append(np.sqrt(np.mean((fc - act) ** 2)))
    snaive_h4_test.iloc[i + H] = fc[H - 1]

snaive_val_rmse  = np.mean(val_errs)
snaive_test_rmse = np.mean(test_errs)
print('SN val RMSE :', round(snaive_val_rmse, 2))
print('SN test RMSE:', round(snaive_test_rmse, 2))

In [ ]:
# SN competition forecast: next 4 timestamps = same time-of-day from 1 day ago
future_idx = pd.date_range(yc_demand_test.index[-1] + pd.Timedelta(minutes=15),
                            periods=4, freq='15min')
sn_future = all_demand.loc[future_idx - pd.Timedelta(days=1)].values
print(pd.DataFrame({'time': future_idx, 'SN forecast (MW)': np.round(sn_future, 2)}))


In [ ]:
# ETS / Holt-Winters with additive seasonality (M=96).
# We extract (l, b, s, alpha, beta, gamma, phi) from the fit, then update the state
# manually as new observations arrive. This avoids re-fitting at every origin.
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import copy
import warnings
warnings.filterwarnings('ignore')

def fit_hw(y, trend, damped):

    model = ExponentialSmoothing(y, trend=trend, damped_trend=damped,
                                  seasonal='add', seasonal_periods=M,
                                  initialization_method='heuristic')
    return model.fit()

def _safe(x, default):

    try:
        v = float(x)
        return default if np.isnan(v) else v
    except (TypeError, ValueError):
        return default

def get_state(fit):
    p = fit.params
    b = _safe(fit.trend.iloc[-1], 0.0) if fit.trend is not None and len(fit.trend) > 0 else 0.0
    return {
        'l': _safe(fit.level.iloc[-1], 0.0),
        'b': b,
        's': [_safe(x, 0.0) for x in fit.season.iloc[-M:].values],
        'alpha': _safe(p.get('smoothing_level'),    0.0),
        'beta':  _safe(p.get('smoothing_trend'),    0.0),
        'gamma': _safe(p.get('smoothing_seasonal'), 0.0),
        'phi':   _safe(p.get('damping_trend'),      1.0),
    }

def hw_pred(s, h_max=4):
    l, b, ss, phi = s['l'], s['b'], s['s'], s['phi']
    out = []
    for h in range(1, h_max + 1):
        cum = h if phi >= 1.0 else sum(phi**i for i in range(1, h + 1))
        out.append(l + cum * b + ss[(h - 1) % M])
    return out

def hw_step(s, y):
    l, b, ss = s['l'], s['b'], s['s']
    a, beta, g, phi = s['alpha'], s['beta'], s['gamma'], s['phi']
    s0 = ss[0]
    new_l = a * (y - s0) + (1 - a) * (l + phi * b)
    new_b = beta * (new_l - l) + (1 - beta) * phi * b
    new_s = g * (y - l - phi * b) + (1 - g) * s0
    s['l'], s['b'], s['s'] = new_l, new_b, ss[1:] + [new_s]
    return s

configs = {
    'add':    ('add', False),
    'damped': ('add', True),
    'none':   (None,  False),
}

fits = {}
states = {}
for name, (trend, damped) in configs.items():
    fits[name] = fit_hw(yc_demand_train, trend, damped)
    states[name] = get_state(fits[name])
    print(name, 'AIC =', round(fits[name].aic, 1))

In [ ]:
# Rolling 4-step RMSE on val for each config. Update first then forecast so the
# origin t = y_eval[t], matching the SN convention.
def rolling_rmse(init_state, y_eval, H=4):
    s = copy.deepcopy(init_state)
    errs = []
    n = len(y_eval) - H
    for t in range(n):
        s = hw_step(s, y_eval.iloc[t])
        fc = hw_pred(s, H)
        act = y_eval.iloc[t + 1: t + 1 + H].values
        errs.append(np.sqrt(np.mean((np.array(fc) - act) ** 2)))
    for t in range(n, len(y_eval)):
        s = hw_step(s, y_eval.iloc[t])
    return errs, s

val_rmses = {}
for name in configs:
    errs, _ = rolling_rmse(states[name], yc_demand_val, H=H)
    val_rmses[name] = np.mean(errs)
    print(name, 'val RMSE:', round(val_rmses[name], 2))

# Skip configs whose fit failed (val RMSE = NaN)
valid = {k: v for k, v in val_rmses.items() if not np.isnan(v)}
if not valid:
    raise RuntimeError('All ETS configs failed to converge')
best_name = min(valid, key=valid.get)
best_state = states[best_name]
print('winner:', best_name)


In [ ]:
# VAL: same loop, but also record the h=4 forecast at each step for plotting later.
ets_h4_val = pd.Series(index=yc_demand_val.index, dtype=float)
s = copy.deepcopy(best_state)
for t in range(len(yc_demand_val) - H):
    s = hw_step(s, yc_demand_val.iloc[t])
    ets_h4_val.iloc[t + H] = hw_pred(s, H)[H - 1]
for t in range(len(yc_demand_val) - H, len(yc_demand_val)):
    s = hw_step(s, yc_demand_val.iloc[t])
state_after_val = s

In [ ]:
# TEST: same protocol, start from state_after_val. Tail loop advances state to the
# very last observation in the dataset so the competition forecast can use it.
ets_h4_test = pd.Series(index=yc_demand_test.index, dtype=float)
s = copy.deepcopy(state_after_val)
test_preds = []
for t in range(len(yc_demand_test) - H):
    s = hw_step(s, yc_demand_test.iloc[t])
    fc = hw_pred(s, H)
    ets_h4_test.iloc[t + H] = fc[H - 1]
    test_preds.append(fc)
for t in range(len(yc_demand_test) - H, len(yc_demand_test)):
    s = hw_step(s, yc_demand_test.iloc[t])
state_after_test = s

test_errs_ets = []
for t, fc in enumerate(test_preds):
    act = yc_demand_test.iloc[t + 1: t + 1 + H].values
    test_errs_ets.append(np.sqrt(np.mean((np.array(fc) - act) ** 2)))
ets_test_rmse = np.mean(test_errs_ets)
print('ETS test RMSE:', round(ets_test_rmse, 2))


residual plot check


In [ ]:
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf


actuals = yc_demand_val.iloc[H : len(yc_demand_val)]
predictions = ets_h4_val.iloc[H : len(yc_demand_val)]
residuals = (actuals - predictions).dropna()


fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# scatter plot of residual
axes[0].scatter(range(len(residuals)), residuals.values,
                color='#1f77b4', alpha=0.6, s=15, label='Residuals')
axes[0].axhline(0, color='red', linestyle='--', linewidth=1.5, label='Zero Error')
axes[0].set_title(f'{H}-Step Ahead Residuals (Scatter)')
axes[0].set_xlabel('Observation Index')
axes[0].set_ylabel('Residual (Actual - Predicted)')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

plot_acf(residuals, ax=axes[1], lags=24, color='#2ca02c', vlines_kwargs={"colors": '#2ca02c'})
axes[1].set_title('Residuals Autocorrelation (ACF)')
axes[1].set_xlabel('Lags')
axes[1].set_ylabel('ACF Value')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ETS competition forecast: 4 future steps from end-of-dataset state
ets_future = hw_pred(state_after_test, h_max=4)
print(pd.DataFrame({'time': future_idx, 'ETS forecast (MW)': np.round(ets_future, 2)}))

In [ ]:
comp = pd.DataFrame({
    'method': ['Seasonal Naive', f'ETS-{best_name}'],
    'val RMSE (MW)':  [round(snaive_val_rmse, 2),  round(val_rmses[best_name], 2)],
    'test RMSE (MW)': [round(snaive_test_rmse, 2), round(ets_test_rmse, 2)],
})
print(comp)


In [ ]:
# Test forecasts: SN and ETS side by side
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

ax1.plot(yc_demand_test, color='black', linewidth=1.2, label='actual')
ax1.plot(snaive_h4_test, color='red',   linewidth=1.2, linestyle='--', label='SN h=4')
ax1.set_title(f'Seasonal Naive  —  test RMSE = {snaive_test_rmse:.0f} MW')
ax1.set_ylabel('demand (MW)')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(yc_demand_test, color='black', linewidth=1.2, label='actual')
ax2.plot(ets_h4_test,    color='blue',  linewidth=1.2, linestyle='--', label=f'ETS-{best_name} h=4')
ax2.set_title(f'ETS  —  test RMSE = {ets_test_rmse:.0f} MW')
ax2.set_xlabel('time')
ax2.set_ylabel('demand (MW)')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Zoom: last 6 hours of test + the 4 future steps from both methods
ctx = yc_demand_test.iloc[-24:]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(ctx.index, ctx.values, color='black', marker='o', markersize=4, label='actual (last 6 h)')
ax.plot(future_idx, sn_future,  color='red',  linestyle='--', marker='o', markersize=10, label='SN future')
ax.plot(future_idx, ets_future, color='blue', linestyle='--', marker='s', markersize=10, label='ETS future')
ax.axvline(ctx.index[-1], color='gray', linestyle=':')
ax.set_title('4-step ahead future forecast (the competition deliverable)')
ax.set_xlabel('time')
ax.set_ylabel('demand (MW)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
final = pd.DataFrame({
    'timestamp': future_idx,
    'h':         [1, 2, 3, 4],
    'SN (MW)':   np.round(sn_future, 2),
    'ETS (MW)':  np.round(ets_future, 2),
})
print(final)


**TASK B: SARIMA (demand)**

In [ ]:
#define a function to check stationarity visually

def plot_curve(timeseries):
    # Determine rolling statistics
    rolmean = timeseries.rolling(96,center=True).mean()
    rolstd = timeseries.rolling(96,center=True).std()

    # Plot rolling statistics:
    plt.figure(figsize=(13,6))
    plt.plot(timeseries, color='blue',label='Original')
    plt.plot(rolmean, color='red', label='Rolling Mean')
    plt.plot(rolstd, color='black', label = 'Rolling Std')
    plt.legend(loc='best')
    plt.title('Rolling Mean & Standard Deviation')
    plt.grid()
    plt.show()

In [ ]:
#it seems the demand time series is not stationary
#first differencing may be needed

plot_curve(yc_demand_train)

In [ ]:
#demand time series still shows non-stationarity after seasonal differencing

plot_curve(yc_demand_train.diff(96).dropna())

In [ ]:
#According to the ACF plot, it has positive values to a high number of lags, further suggesting a first differencing may be needed.

demand_diff = yc_demand_train.diff(96).dropna()

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
plot_acf(demand_diff, lags=192, alpha=0.05, ax=axes[0])
axes[0].set_title('ACF after diff(96)')
axes[0].axvline(96, color='red', linestyle='--', alpha=0.5, label='lag=96')
axes[0].axvline(192, color='red', linestyle='--', alpha=0.5, label='lag=192')
axes[0].legend()

plot_pacf(demand_diff, lags=192, alpha=0.05, ax=axes[1])
axes[1].set_title('PACF after diff(96)')
axes[1].axvline(96, color='red', linestyle='--', alpha=0.5, label='lag=96')
axes[1].axvline(192, color='red', linestyle='--', alpha=0.5, label='lag=192')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
#after first differencing and seasonal differencing, the demand time series looks statioanry

plot_curve(yc_demand_train.diff(1).diff(96).dropna())

In [ ]:
#ACF and PACF suggests the range of order of the SARIMA model
#p = 1 q = [1,5] P = 0 Q = 1

demand_diff = yc_demand_train.diff(1).diff(96).dropna()

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
plot_acf(demand_diff, lags=192, alpha=0.05, ax=axes[0])
axes[0].set_title('ACF after diff(1) and diff(96)')
axes[0].axvline(96, color='red', linestyle='--', alpha=0.5, label='lag=96')
axes[0].axvline(192, color='red', linestyle='--', alpha=0.5, label='lag=192')
axes[0].legend()

plot_pacf(demand_diff, lags=192, alpha=0.05, ax=axes[1])
axes[1].set_title('PACF after diff(1) and diff(96)')
axes[1].axvline(96, color='red', linestyle='--', alpha=0.5, label='lag=96')
axes[1].axvline(192, color='red', linestyle='--', alpha=0.5, label='lag=192')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
#use grid search to find the best combination with the lowest AIC

p_values = [1]
q_values = [1, 2, 3, 4, 5]
P_values = [0]
Q_values = [1]

d, D, m = 1, 1, 96

results = []

for p in p_values:
    for q in q_values:
        for P in P_values:
            for Q in Q_values:
                try:
                    print(f'Fitting SARIMA{(p,d,q)}{(P,D,Q,m)}...')

                    model = SARIMAX(
                        yc_demand_train,
                        order=(p, d, q),
                        seasonal_order=(P, D, Q, m),
                        enforce_stationarity=False,
                        enforce_invertibility=False,
                        simple_differencing=True
                    )

                    result = model.fit()

                    results.append({
                        'order':          (p, d, q),
                        'seasonal_order': (P, D, Q, m),
                        'AIC':            result.aic,
                        'BIC':            result.bic,
                        'converged':      result.mle_retvals.get('converged', None),
                        'result':         result
                    })

                    print(f'✔ SARIMA{(p,d,q)}{(P,D,Q,m)} '
                          f'AIC={result.aic:.2f} BIC={result.bic:.2f}')

                except Exception as e:
                    print(f'✘ FAILED SARIMA{(p,d,q)}{(P,D,Q,m)}: {e}')
                    continue

results_df = pd.DataFrame([{k:v for k,v in r.items() if k != 'result'}
                            for r in results])
results_df = results_df.sort_values('AIC').reset_index(drop=True)
results_df['ΔAIC'] = (results_df['AIC'] - results_df['AIC'].min()).round(2)
display(results_df)

results_df_converged = results_df[results_df['converged'] == True]
best_row = results_df_converged.loc[results_df_converged['AIC'].idxmin()]

best_order          = best_row['order']
best_seasonal_order = best_row['seasonal_order']

print(f'Best converged model: SARIMA{best_order}{best_seasonal_order}')
print(f'AIC: {best_row["AIC"]:.2f}')
print(f'BIC: {best_row["BIC"]:.2f}')

In [ ]:
#fit the SARIMA model with the best combination

demand_model = SARIMAX(yc_demand_train,
                order = (1,1,4),
                seasonal_order = (0,1,1,96),
                trend = 'c',
                enforce_stationarity = False,
                enforce_invertibility = False)

demand_model_fit = demand_model.fit()

In [ ]:
#show the SARIMA model summary for demand

print(demand_model_fit.summary())

In [ ]:
#do a rolling evaluation on the validation set using the fitted SARIMA model

import warnings
warnings.filterwarnings('ignore')

H = 4

res_full = demand_model_fit.apply(
    pd.concat([yc_demand_train, yc_demand_val]),
    refit=False
)

n_train = len(yc_demand_train)
n_val   = len(yc_demand_val)

rmses = []
sarima_forecast = pd.Series(index=yc_demand_val.index, dtype=float)

for t in range(n_val - H):
    start_idx = n_train + t + 1
    end_idx   = n_train + t + H

    fc = res_full.get_prediction(
        start   = start_idx,
        end     = end_idx,
        dynamic = start_idx
    ).predicted_mean.values

    act = yc_demand_val.iloc[t + 1: t + 1 + H].values

    if len(fc) == H and len(act) == H:
        sarima_forecast.iloc[t + H] = fc[-1]
        rmses.append(np.sqrt(np.mean((fc - act) ** 2)))

sarima_rmse = np.mean(rmses)
print(f"4-step RMSE: {sarima_rmse:.2f} MW")

In [ ]:
#check the training RMSE and validation RMSE, and training RMSE is just slightly higher than validation RMSE

res_train = demand_model_fit.apply(
    yc_demand_train,
    refit=False
)

rmses_train = []

for t in range(len(yc_demand_train) - H):
    start_idx = t + 1
    end_idx   = t + H

    fc = res_train.get_prediction(
        start   = start_idx,
        end     = end_idx,
        dynamic = start_idx
    ).predicted_mean.values

    act = yc_demand_train.iloc[t + 1: t + 1 + H].values

    if len(fc) == H and len(act) == H:
        rmses_train.append(np.sqrt(np.mean((fc - act) ** 2)))

sarima_train_rmse = np.mean(rmses_train)

summary = pd.DataFrame({
    'Split':     ['Train', 'Validation'],
    'Days':      [TRAIN_DAYS, VAL_DAYS],
    'RMSE (MW)': [round(sarima_train_rmse, 2),
                  round(sarima_rmse,       2)]
})

print('\n── SARIMA Performance Summary ───────────────────────────────────────')
print(summary.to_string(index=False))

In [ ]:
#plot the training set and validation set as well as the forecasts on the validation set for visual comparison
#plot only the validation set and the forecasts on the validation set for clearer and further comparison

fig, axes = plt.subplots(2, 1, figsize=(15, 10))

ax = axes[0]
ax.plot(yc_demand_train,  label='Training Demand',
        linewidth=1.5, color='tab:blue')
ax.plot(yc_demand_val,    label='Validation Demand (Actual)',
        linewidth=1.5, color='tab:orange')
ax.plot(sarima_forecast,  label='4-step Forecast (SARIMA)',
        linewidth=1.2, linestyle='--', color='tab:green')
ax.axvline(yc_demand_val.index[0], color='black',
           linestyle=':', linewidth=1.2, label='Train/Val Split')
ax.set_title(f'SARIMA Demand Forecast — Full View  |  Val RMSE: {sarima_rmse:.2f} MW')
ax.set_xlabel('Time')
ax.set_ylabel('Demand (MW)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

ax = axes[1]
ax.plot(yc_demand_val,   label='Actual Demand',
        linewidth=1.5, color='tab:orange')
ax.plot(sarima_forecast, label='4-step Forecast (SARIMA)',
        linewidth=1.2, linestyle='--', color='tab:green')
ax.set_title('Validation Period: Actual vs SARIMA 4-Step-Ahead Forecast')
ax.set_xlabel('Time')
ax.set_ylabel('Demand (MW)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.savefig('output/figures/sarima_demand_forecast.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

In [ ]:
#make a residual diagnostics
#residuals fluctuates around 0, and variance is relatively stable
#distribution of the residuals is roughly symmetric and centered around 0
#points stays tightly with the diagonal across the full demand range
#all other lags almost fall within the confidence band except lag 0
residuals_sarima = yc_demand_val - sarima_forecast
residuals_clean  = residuals_sarima.dropna()

print(f'Residual stats (validation set):')
print(f'  Mean:     {residuals_clean.mean():.2f} MW')
print(f'  Std:      {residuals_clean.std():.2f} MW')
print(f'  Max:      {residuals_clean.max():.2f} MW')
print(f'  Min:      {residuals_clean.min():.2f} MW')
print(f'  Kurtosis: {residuals_clean.kurtosis():.2f}')

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# ── Plot 1: Residuals over time ────────────────────────────────────────────────
ax = axes[0, 0]
ax.plot(residuals_clean.index, residuals_clean.values,
        color='tab:red', linewidth=0.8, alpha=0.8)
ax.axhline(0, color='black', linestyle='--', linewidth=1)
ax.set_title('Residuals over Time')
ax.set_xlabel('Time')
ax.set_ylabel('Residual (MW)')
ax.grid(True, alpha=0.3)
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

# ── Plot 2: Distribution of residuals ─────────────────────────────────────────
ax = axes[0, 1]
ax.hist(residuals_clean.values, bins=50,
        color='tab:blue', edgecolor='white', alpha=0.8)
ax.axvline(0, color='black', linestyle='--', linewidth=1)
ax.axvline(residuals_clean.mean(), color='tab:red', linestyle='--',
           linewidth=1.5, label=f'Mean = {residuals_clean.mean():.1f} MW')
ax.set_title('Distribution of Residuals')
ax.set_xlabel('Residual (MW)')
ax.set_ylabel('Frequency')
ax.legend()
ax.grid(True, alpha=0.3)

# ── Plot 3: Scatterplot — actual vs predicted ─────────────────────────────────────
ax = axes[1, 0]
common_idx  = residuals_clean.index
actual_vals = yc_demand_val.loc[common_idx].values
pred_vals   = sarima_forecast.loc[common_idx].values
ax.scatter(pred_vals, actual_vals,
           alpha=0.3, s=8, color='tab:purple')
min_v = min(pred_vals.min(), actual_vals.min())
max_v = max(pred_vals.max(), actual_vals.max())
ax.plot([min_v, max_v], [min_v, max_v],
        'k--', linewidth=1.5, label='Perfect forecast')
ax.set_title('Predicted vs Actual (Validation)')
ax.set_xlabel('Predicted Demand (MW)')
ax.set_ylabel('Actual Demand (MW)')
ax.legend()
ax.grid(True, alpha=0.3)

# ── Plot 4: ACF of residuals ───────────────────────────────────────────────────
ax = axes[1, 1]
plot_acf(residuals_clean.values, lags=96, ax=ax, alpha=0.05)
ax.set_title('ACF of Residuals')
ax.set_xlabel('Lag')
ax.set_ylabel('ACF')
ax.grid(True, alpha=0.3)

plt.suptitle('SARIMA Residual Diagnostics — Validation Set', fontsize=14)
plt.tight_layout()
plt.savefig('output/figures/sarima_residual_diagnostics_val.png',
            dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

In [ ]:
#forecast on the test set to simulate true performance

res_test = demand_model_fit.apply(
    pd.concat([yc_demand_train, yc_demand_val, yc_demand_test]),
    refit=False
)

n_train_val = len(yc_demand_train) + len(yc_demand_val)
n_test      = len(yc_demand_test)

rmses_test = []
sarima_forecast_test = pd.Series(index=yc_demand_test.index, dtype=float)

for t in range(n_test - H):
    start_idx = n_train_val + t + 1
    end_idx   = n_train_val + t + H

    fc = res_test.get_prediction(
        start   = start_idx,
        end     = end_idx,
        dynamic = start_idx
    ).predicted_mean.values

    act = yc_demand_test.iloc[t + 1: t + 1 + H].values

    if len(fc) == H and len(act) == H:
        sarima_forecast_test.iloc[t + H] = fc[-1]
        rmses_test.append(np.sqrt(np.mean((fc - act) ** 2)))

sarima_test_rmse = np.mean(rmses_test)
print(f"Test RMSE:       {sarima_test_rmse:.2f} MW")
print(f"Validation RMSE: {sarima_rmse:.2f} MW")

In [ ]:
#plot the training, validation and test sets as well as the forecasts on the validation and test sets for visual comparison
#plot only the test set and the forecasts on the test set for clearer and further comparison

fig, axes = plt.subplots(2, 1, figsize=(15, 10))

ax = axes[0]
ax.plot(yc_demand_train,           label='Train Demand',
        linewidth=1.5, color='tab:red')
ax.plot(yc_demand_val,           label='Validation Demand (Actual)',
        linewidth=1.5, color='tab:orange')
ax.plot(sarima_forecast,  label='4-step val Forecast (SARIMA)',
        linewidth=1.2, linestyle='--', color='tab:blue')
ax.plot(yc_demand_test,          label='Test Demand (Actual)',
        linewidth=1.5, color='tab:purple')
ax.plot(sarima_forecast_test,    label='4-step test Forecast (SARIMA)',
        linewidth=1.2, linestyle='--', color='tab:green')
ax.axvline(yc_demand_val.index[0], color='black',
           linestyle=':', linewidth=1.2, label='Train/Val Split')
ax.axvline(yc_demand_test.index[0], color='black',
           linestyle=':', linewidth=1.2, label='Val/Test Split')
ax.set_title(f'SARIMA Demand Forecast — Test Set  |  Test RMSE: {sarima_test_rmse:.2f} MW')
ax.set_xlabel('Time')
ax.set_ylabel('Demand (MW)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

ax = axes[1]
ax.plot(yc_demand_test,       label='Actual Demand (Test)',
        linewidth=1.5, color='tab:red')
ax.plot(sarima_forecast_test, label='4-step Forecast (SARIMA)',
        linewidth=1.2, linestyle='--', color='tab:green')
ax.set_title('Test Period: Actual vs SARIMA 4-Step-Ahead Forecast')
ax.set_xlabel('Time')
ax.set_ylabel('Demand (MW)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.savefig('output/figures/sarima_demand_forecast_test.png',
            dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

In [ ]:
#make a summary table including validation RMSE and test RMSE

summary = pd.DataFrame({
    'Split':     ['Validation', 'Test'],
    'Days':      [VAL_DAYS, TEST_DAYS],
    'RMSE (MW)': [round(sarima_rmse, 2), round(sarima_test_rmse, 2)]
})
print('\n── SARIMA Final Performance Summary ────────────────────────────────')
print(summary.to_string(index=False))

In [ ]:
#make a 4-step-ahead forecast

final_fc = res_test.forecast(steps=4).values

final_timestamps = pd.date_range(
    start='2026-02-10 00:00:00',
    periods=4,
    freq='15min'
)

sarima_finaldemand_forecast_df = pd.DataFrame({
    'SETTLEMENTDATE': final_timestamps,
    'TOTALDEMAND':    np.round(final_fc, 2)
})

print('\nSARIMA Forecast (Demand):')
print(sarima_finaldemand_forecast_df.to_string(index=False))

# **Task B: Price Forecasting**
This section builds price (RRP) forecasts.

All modelling is done on the arcsinh-transformed price yc_price = arcsinh(RRP / 100) produced in Task A, because RRP contains spikes and negative values. Forecasts are inverse-transformed via 100 * sinh(yc) for evaluation on the original $/MWh scale.

Test set is not used here — final test evaluation is left until the group's best model is chosen. Model selection in this section uses the validation set only.

The following part is designed to isolate the **price spike** section and validate models for that section.

In [ ]:
# Tunable parameter
IQR_MULTIPLIER = 1.5

# Compute spike thresholds using TRAINING RRP only
Q1_price = y_price_train.quantile(0.25)
Q3_price = y_price_train.quantile(0.75)
IQR_price = Q3_price - Q1_price

lower_spike_threshold = Q1_price - IQR_MULTIPLIER * IQR_price
upper_spike_threshold = Q3_price + IQR_MULTIPLIER * IQR_price

# Classify validation observations on ORIGINAL RRP scale
spike_mask_val_raw = (
    (y_price_val < lower_spike_threshold) |
    (y_price_val > upper_spike_threshold)
)

non_spike_mask_val_raw = ~spike_mask_val_raw

print("Lower spike threshold:", round(lower_spike_threshold, 3))
print("Upper spike threshold:", round(upper_spike_threshold, 3))
print("Validation spike count:", int(spike_mask_val_raw.sum()))
print("Validation non-spike count:", int(non_spike_mask_val_raw.sum()))

**1. Price 4-step-ahead Forecast:  Naive and Seasonal Naive Benchmarks**

**Naive:** predicts the next 4 steps as a constant equal to the last observed transformed RRP. This is the simplest no-information benchmark.

**Seasonal Naive:** predicts each future step as the value from the same time-of-day one day earlier. The seasonal period M = 96 is justified by:

  The data has 15-minute frequency, so one full day contains 24 * 4 = 96 observations.

  Task A EDA showed a clear intraday pattern in transformed RRP, with low overnight prices and pronounced morning/evening behaviour.

Modelling is performed on yc_price = arcsinh(RRP / 100), and the forecast table reports both the transformed-scale forecast and the back-transformed RRP forecast.

1.1 Build Naive and Seasonal Naive

In [ ]:
# Parameters
HORIZON = 4
STEPS_PER_DAY = 96
M = STEPS_PER_DAY  # daily seasonal period for 15-minute data

# Helper 1: inverse arcsinh transform back to RRP scale
def inverse_price_transform(yc):
    return 100 * np.sinh(yc)

# Helper 2: RMSE
def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

# Build the 4 future 15-min timestamps starting right after the training end
last_train_time = yc_price_train.index[-1]
forecast_index = pd.date_range(
    start=last_train_time + pd.Timedelta(minutes=15),
    periods=HORIZON,
    freq='15min'
)

# Naive: repeat the last training value HORIZON times
yc_naive_forecast = pd.Series(
    [yc_price_train.iloc[-1]] * HORIZON,
    index=forecast_index,
    name='naive'
)

# Seasonal Naive: same 4 time-of-day slots from one day earlier in training
yc_snaive_forecast = pd.Series(
    yc_price_train.iloc[-M : -M + HORIZON].values,
    index=forecast_index,
    name='snaive'
)

1.2 Naive and Seasonal Naive 4-step-ahead forecasts for price (RRP)

In [ ]:
# Naive and Seasonal Naive 4-step-ahead forecasts table
forecast_table_b1 = pd.DataFrame({
    'naive (transformed)':  yc_naive_forecast.values,
    'snaive (transformed)': yc_snaive_forecast.values,
    'naive (RRP $/MWh)':    inverse_price_transform(yc_naive_forecast).values,
    'snaive (RRP $/MWh)':   inverse_price_transform(yc_snaive_forecast).values,
}, index=forecast_index)

print("Naive and Seasonal Naive 4-step-ahead forecasts for price (RRP)")
forecast_table_b1.round(3)

1.3 Rolling 4-step-ahead validation for Naive and Seasonal Naive

We evaluate these two benchmarks under the **rolling 4-step-ahead block protocol**, which matches the assignment's 4-step-ahead forecasting requirement:

1. The first forecast origin is the **last training timestamp**.
2. From each origin, we forecast the next `HORIZON = 4` validation observations as a block.
3. The origin then advances by 4 steps; the just-forecasted block becomes observed history for the next block.
4. We repeat until the entire validation set is covered.

Rules:

- **Naive**: all 4 forecasts in a block equal the last observed transformed RRP at the block origin.
- **Seasonal Naive**: for each future timestamp `t+h`, forecast = transformed RRP at `t+h − 96`. Because `M = 96 ≫ HORIZON = 4`, the lookup is always strictly before block start, so it uses only training + already-observed validation history.

In [ ]:
# Shared validation setup
val_index   = yc_price_val.index
n_val       = len(val_index)
yc_combined = pd.concat([yc_price_train, yc_price_val]).sort_index()

# Align spike / non-spike masks to the validation index. This avoids silent
# bugs from length mismatches or stray NaN entries.
spike_mask_val     = (spike_mask_val_raw
                      .reindex(y_price_val.index)
                      .fillna(False)
                      .astype(bool))
non_spike_mask_val = (non_spike_mask_val_raw
                      .reindex(y_price_val.index)
                      .fillna(False)
                      .astype(bool))

assert (spike_mask_val | non_spike_mask_val).all(), \
    'Spike/non-spike masks do not cover the full validation index'
assert not (spike_mask_val & non_spike_mask_val).any(), \
    'Spike and non-spike masks overlap'

# Shared subset-RMSE helper
def _aligned_rmse(actual, predicted, mask=None):
    a = actual if mask is None else actual[mask]
    a = a.dropna()
    p = predicted.reindex(a.index).dropna()
    common = a.index.intersection(p.index)
    if len(common) == 0:
        return np.nan
    return rmse(a.loc[common].values, p.loc[common].values)

print(f"Validation length            : {n_val}")
print(f"Validation spike count       : {int(spike_mask_val.sum())}")
print(f"Validation non-spike count   : {int(non_spike_mask_val.sum())}")

# Rolling 4-step-ahead validation forecasts for Naive and Seasonal Naive
yc_naive_val  = pd.Series(index=val_index, dtype=float, name='naive_val')
yc_snaive_val = pd.Series(index=val_index, dtype=float, name='snaive_val')

for start in range(0, n_val, HORIZON):
    end       = min(start + HORIZON, n_val)
    block_idx = val_index[start:end]
    first_block_time = block_idx[0]

    # Block origin: last observed timestamp strictly before block start
    origin_pos = yc_combined.index.get_loc(first_block_time) - 1
    y_origin   = yc_combined.iloc[origin_pos]

    # Naive: 4 constant predictions equal to y_origin
    yc_naive_val.loc[block_idx] = y_origin

    # Seasonal Naive: each forecast = value 96 steps earlier
    for ts in block_idx:
        lookup_ts = ts - pd.Timedelta(minutes=15) * M
        yc_snaive_val.loc[ts] = yc_combined.loc[lookup_ts]

assert yc_naive_val.notna().all(),  'Naive forecast has missing values'
assert yc_snaive_val.notna().all(), 'Seasonal Naive forecast has missing values'

# Plot 1: Overview
# Full training + validation with both rolling 4-step-ahead forecasts overlaid.
# Plotted on the modelling (transformed) scale.
plt.figure(figsize=(16, 6))
plt.plot(yc_price_train.index, yc_price_train.values,
         color='gray', alpha=0.4, label='Training actual (transformed RRP)')
plt.plot(yc_price_val.index, yc_price_val.values,
         color='black', linewidth=1, label='Validation actual (transformed RRP)')
plt.plot(yc_naive_val.index, yc_naive_val.values,
         color='tab:orange', linewidth=1, label='Naive rolling 4-step-ahead forecast')
plt.plot(yc_snaive_val.index, yc_snaive_val.values,
         color='tab:blue', linewidth=1, label='Seasonal Naive rolling 4-step-ahead forecast')
plt.axvline(yc_price_val.index[0], color='red', linestyle='--', alpha=0.5,
            label='Train / Val boundary')
plt.title('B1: Overview - Rolling 4-step-ahead Naive vs Seasonal Naive')
plt.xlabel('Time')
plt.ylabel('Transformed RRP: arcsinh(RRP / 100)')
plt.legend()
plt.tight_layout()
plt.show()

# Plot 2: Zoomed validation view
# Last 3 training days + first 3 validation days for clearer comparison.
# Visualisation only; RMSE in section 1.4 uses the full validation set.
# Temporary variables (train_plot / val_plot / naive_zoom / snaive_zoom) are
# used so yc_price_train / yc_price_val / y_price_train / y_price_val are not modified.
ZOOM_TRAIN_DAYS = 3
ZOOM_VAL_DAYS   = 3
train_plot  = yc_price_train.iloc[-ZOOM_TRAIN_DAYS * STEPS_PER_DAY:]
val_plot    = yc_price_val.iloc[:ZOOM_VAL_DAYS * STEPS_PER_DAY]
naive_zoom  = yc_naive_val.reindex(val_plot.index)
snaive_zoom = yc_snaive_val.reindex(val_plot.index)

plt.figure(figsize=(16, 6))
plt.plot(train_plot.index, train_plot.values,
         color='gray', alpha=0.5,
         label=f'Training actual (last {ZOOM_TRAIN_DAYS} days)')
plt.plot(val_plot.index, val_plot.values,
         color='black', linewidth=1,
         label=f'Validation actual (first {ZOOM_VAL_DAYS} days)')
plt.plot(naive_zoom.index, naive_zoom.values,
         color='tab:orange', linewidth=1.4, marker='.', markersize=4,
         label='Naive 4-step-ahead forecast')
plt.plot(snaive_zoom.index, snaive_zoom.values,
         color='tab:blue', linewidth=1.4, marker='.', markersize=4,
         label='Seasonal Naive 4-step-ahead forecast')
plt.axvline(yc_price_val.index[0], color='red', linestyle='--', alpha=0.5,
            label='Train / Val boundary')
plt.title('B1: Zoomed validation view - Naive vs Seasonal Naive (Rolling 4-step-ahead)')
plt.xlabel('Time')
plt.ylabel('Transformed RRP: arcsinh(RRP / 100)')
plt.legend()
plt.tight_layout()
plt.show()

1.4 RMSE table for Naive and Seasonal Naive（Spike and Non spike）

In [ ]:
# Robust RMSE helper (aligned by common index, optional mask, drops NaN).
def compute_rmse_by_mask(actual, predicted, mask=None):
    a = actual.dropna()
    p = predicted.reindex(a.index).dropna()
    common = a.index.intersection(p.index)
    if mask is not None:
        m_aligned = mask.reindex(common).fillna(False).astype(bool)
        common    = common[m_aligned]
    if len(common) == 0:
        return float('nan'), 0
    err = a.loc[common].values - p.loc[common].values
    return float(np.sqrt(np.mean(err ** 2))), int(len(common))


# Training spike / non-spike masks (reuse existing training thresholds) ----
training_spike_mask = (
    (y_price_train < lower_spike_threshold) |
    (y_price_train > upper_spike_threshold)
)
training_non_spike_mask = ~training_spike_mask

# Training in-sample forecasts on the original RRP scale ----
yc_naive_train    = yc_price_train.shift(1)
naive_train_pred  = inverse_price_transform(yc_naive_train)

yc_snaive_train   = yc_price_train.shift(STEPS_PER_DAY)
snaive_train_pred = inverse_price_transform(yc_snaive_train)

# Validation forecasts on the original RRP scale ----
# (forecast logic in cell above is preserved; we only inverse-transform here.)
naive_val_rrp  = inverse_price_transform(yc_naive_val)
snaive_val_rrp = inverse_price_transform(yc_snaive_val)


# Table 1: Training RMSE diagnostic for Naive and Seasonal Naive ----
b1_train_methods = [
    ('Naive',          naive_train_pred),
    ('Seasonal Naive', snaive_train_pred),
]
train_rows = []
for name, train_pred in b1_train_methods:
    overall_rmse, _  = compute_rmse_by_mask(y_price_train, train_pred)
    spike_rmse,   sn = compute_rmse_by_mask(y_price_train, train_pred, training_spike_mask)
    ns_rmse,      nn = compute_rmse_by_mask(y_price_train, train_pred, training_non_spike_mask)
    train_rows.append({
        'Method'                  : name,
        'Training Overall RMSE'   : overall_rmse,
        'Training Spike RMSE'     : spike_rmse,
        'Training Non-spike RMSE' : ns_rmse,
        'Training spike count'    : sn,
        'Training non-spike count': nn,
    })
training_rmse_table_b1 = pd.DataFrame(train_rows)
print("Table 1: Training RMSE diagnostic for Naive and Seasonal Naive (original RRP, $/MWh)")
display(training_rmse_table_b1.round(3))


# Table 2: Validation RMSE evaluation for Naive and Seasonal Naive ----
b1_val_methods = [
    ('Naive',          naive_val_rrp),
    ('Seasonal Naive', snaive_val_rrp),
]
val_rows = []
for name, val_pred in b1_val_methods:
    overall_rmse, _  = compute_rmse_by_mask(y_price_val, val_pred)
    spike_rmse,   sn = compute_rmse_by_mask(y_price_val, val_pred, spike_mask_val)
    ns_rmse,      nn = compute_rmse_by_mask(y_price_val, val_pred, non_spike_mask_val)
    val_rows.append({
        'Method'                    : name,
        'Validation Overall RMSE'   : overall_rmse,
        'Validation Spike RMSE'     : spike_rmse,
        'Validation Non-spike RMSE' : ns_rmse,
        'Validation spike count'    : sn,
        'Validation non-spike count': nn,
    })
validation_rmse_table_b1 = pd.DataFrame(val_rows)
print("Table 2: Validation RMSE evaluation for Naive and Seasonal Naive (original RRP, $/MWh)")
display(validation_rmse_table_b1.round(3))

**2. Price decomposition for Transformed RRP**

This follows the **classical additive decomposition** applied to the transformed price:

`yc_price_t = trend_t + seasonal_t + remainder_t`

We use an **additive** decomposition because the transformed series `arcsinh(RRP/100)` can take both positive and negative values, so additive components match the data range naturally.

We follow the decomposition workflow:

(1) Estimate an initial trend-cycle using a 2×96 centred moving average.

(2) Detrend the transformed RRP.

(3) Estimate and normalise the daily seasonal component.

(4) Construct the seasonally adjusted series.

(5) Re-estimate the trend using a linear trend model.

(6) Visually inspect the remainder.

2.1 Build price decomposition

Step 1 — Initial trend-cycle by 2×96 centered moving average

In [ ]:
# Working copy of transformed training RRP
y_decomp = yc_price_train.copy()

# 2x96 centered moving average -> initial trend-cycle
ma_96         = y_decomp.rolling(window=M, center=True).mean()
trend_initial = ma_96.rolling(window=2, center=True).mean()

# Detrended series (used in Step 2 to estimate the daily seasonal component)
detrended = y_decomp - trend_initial

# Figure 1: Transformed RRP and Initial 2x96 CMA Trend
plt.figure(figsize=(14, 4))
plt.plot(y_decomp.index, y_decomp.values,
         label='Transformed RRP', alpha=0.4)
plt.plot(trend_initial.index, trend_initial.values,
         label='Initial 2x96 CMA trend', color='red', linewidth=1.5)
plt.title('Figure 1: Transformed RRP and Initial 2x96 CMA Trend')
plt.xlabel('Time')
plt.ylabel('Transformed RRP: arcsinh(RRP / 100)')
plt.legend()
plt.tight_layout()
plt.show()

# Figure 2: Detrended Transformed RRP
plt.figure(figsize=(14, 4))
plt.plot(detrended.index, detrended.values,
         color='darkorange', label='Detrended transformed RRP')
plt.axhline(0, color='black', linewidth=0.8)
plt.title('Figure 2: Detrended Transformed RRP')
plt.xlabel('Time')
plt.ylabel('Detrended transformed RRP')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

Step 2 — Daily seasonal component from the detrended series

In [ ]:
# Step 2: estimate 96 seasonal effects by time-of-day averaging
tod_train = yc_price_train.index.hour * 4 + yc_price_train.index.minute // 15

# Group detrended values by 0..95 TOD slot and take the mean
seasonal_pattern_raw = detrended.groupby(tod_train).mean()
seasonal_pattern_raw = seasonal_pattern_raw.sort_index()

# Normalize so the 96 seasonal effects sum to zero over a full day
seasonal_pattern = seasonal_pattern_raw - seasonal_pattern_raw.mean()

# Build the full-length seasonal component by timestamp mapping: each
# observation's time-of-day slot is mapped to its seasonal effect. This is
# safer than np.tile because it does not assume the training data starts at
# TOD = 0 or contains a whole number of full days.
seasonal_component = pd.Series(
    seasonal_pattern.reindex(tod_train).values,
    index=y_decomp.index,
    name="seasonal_component"
)

# Seasonally adjusted series (used in Step 3 to re-estimate trend)
seasonally_adjusted = y_decomp - seasonal_component

# Plot 1: the estimated 96-slot seasonal pattern
plt.figure(figsize=(10, 4))
plt.plot(seasonal_pattern.index, seasonal_pattern.values, marker='.')
plt.axhline(0, color='black', linewidth=0.8)
plt.title('Step 2: Estimated Daily Seasonal Pattern (96 TOD slots, normalized)')
plt.xlabel('Time-of-day slot (0 = 00:00, 95 = 23:45)')
plt.ylabel('Seasonal effect (transformed scale)')
plt.tight_layout()
plt.show()

# Plot 2: seasonally adjusted transformed RRP
plt.figure(figsize=(14, 4))
plt.plot(seasonally_adjusted.index, seasonally_adjusted.values, color='steelblue')
plt.title('Step 2: Seasonally Adjusted Transformed RRP')
plt.xlabel('Time')
plt.ylabel('arcsinh(RRP / 100) - seasonal_component')
plt.tight_layout()
plt.show()

print(f"Daily mean of seasonal_pattern        : {seasonal_pattern.mean():.6f}")

Step 3 — Re-estimate the trend by linear regression on seasonally adjusted data

This models the trend component (a deterministic linear time trend), not the remainder.

In [ ]:
# Fit a simple linear trend on seasonally adjusted training data
t_train     = np.arange(len(seasonally_adjusted)).reshape(-1, 1)
trend_model = LinearRegression()
trend_model.fit(t_train, seasonally_adjusted.values)

trend_refined = pd.Series(
    trend_model.predict(t_train),
    index=y_decomp.index,
    name="refined_trend"
)

# Figure 5: Re-estimated Trend from Seasonally Adjusted Transformed RRP
plt.figure(figsize=(14, 4))
plt.plot(seasonally_adjusted.index, seasonally_adjusted.values,
         label='Seasonally adjusted transformed RRP',
         color='steelblue', alpha=0.6)
plt.plot(trend_refined.index, trend_refined.values,
         label='Refined linear trend', color='red', linewidth=2)
plt.title('Figure 5: Re-estimated Trend from Seasonally Adjusted Transformed RRP')
plt.xlabel('Time')
plt.ylabel('Transformed RRP: arcsinh(RRP / 100)')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Linear trend slope (per 15-min step): {trend_model.coef_[0]:.6f}")
print(f"Linear trend intercept             : {trend_model.intercept_:.6f}")

Step 4 — Remainder diagnostic (visual only, no modelling)

In [ ]:
# Remainder = transformed RRP - refined trend - seasonal component
remainder = y_decomp - trend_refined - seasonal_component

# Figure 6: combined remainder diagnostic (time series + histogram + ACF)
from statsmodels.graphics.tsaplots import plot_acf

fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Panel 1: remainder over time
axes[0].plot(remainder.index, remainder.values, color='gray')
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('Remainder over time (transformed RRP scale)')
axes[0].set_xlabel('Time')
axes[0].set_ylabel('Remainder of transformed RRP')

# Panel 2: remainder histogram
axes[1].hist(remainder.dropna().values, bins=60,
             color='gray', edgecolor='black')
axes[1].set_title('Remainder histogram (transformed RRP scale)')
axes[1].set_xlabel('Remainder of transformed RRP')
axes[1].set_ylabel('Frequency')

# Panel 3: ACF of remainder
plot_acf(remainder.dropna(), lags=192, ax=axes[2])
axes[2].set_title('ACF of remainder (lags = 192 = 2 days)')
axes[2].set_xlabel('Lag (15-min steps)')
axes[2].set_ylabel('Autocorrelation')

fig.suptitle('Figure 6: Remainder Diagnostic', fontsize=14, y=1.00)
plt.tight_layout()
plt.show()

print(f"Remainder mean: {remainder.mean():.4f}")
print(f"Remainder std : {remainder.std():.4f}")

**2.2 Decomposition-based 4-step-ahead Forecast**

Using the additive decomposition estimated above, the forecast is:

`yc_forecast = trend_forecast + seasonal_forecast + remainder_forecast`

with:
- `trend_forecast` is generated from the refined linear trend model fitted on seasonally adjusted training data.
- `seasonal_forecast` = `seasonal_pattern` looked up by the time-of-day of each future timestamp
- `remainder_forecast = 0` (remainder is treated as unexplained noise; not modelled)

Forecasts are then inverse-transformed via `100 * sinh(·)` to obtain the RRP $/MWh scale.

In [ ]:
# 4-step-ahead decomposition forecast
n_train  = len(yc_price_train)
future_t = np.arange(n_train, n_train + HORIZON).reshape(-1, 1)

trend_forecast    = trend_model.predict(future_t)
future_tod        = forecast_index.hour * 4 + forecast_index.minute // 15
seasonal_forecast = seasonal_pattern.reindex(future_tod).values

yc_decomp_forecast = pd.Series(
    trend_forecast + seasonal_forecast,
    index=forecast_index,
    name='decomp'
)

decomp_forecast_table = pd.DataFrame({
    'trend forecast':              trend_forecast,
    'seasonal forecast':           seasonal_forecast,
    'decomposition (transformed)': yc_decomp_forecast.values,
    'decomposition (RRP $/MWh)':   inverse_price_transform(yc_decomp_forecast).values,
}, index=forecast_index)

print("Classical decomposition 4-step-ahead forecast for price (RRP)")
decomp_forecast_table.round(3)

**2.3 Rolling 4-step-ahead validation for Decomposition**

Validation uses the **same rolling 4-step-ahead block protocol**. The decomposition components (`trend_model` and `seasonal_pattern`) are estimated from **training only** and are **never re-fit** on validation data.

In [ ]:
# Continuing integer time index for each validation timestamp (n_train, n_train+1, ...)
pos_in_combined = np.array([yc_combined.index.get_loc(ts) for ts in val_index])
t_val_full      = pos_in_combined.reshape(-1, 1)

yc_decomp_val = pd.Series(index=val_index, dtype=float, name='decomp_val')

for start in range(0, n_val, HORIZON):
    end       = min(start + HORIZON, n_val)
    block_idx = val_index[start:end]
    block_pos = np.arange(start, end)

    # Trend: continuing linear regression prediction (training-only model)
    t_block    = t_val_full[block_pos]
    trend_part = trend_model.predict(t_block)

    # Seasonal: training-estimated seasonal pattern, looked up by TOD
    tod_block     = block_idx.hour * 4 + block_idx.minute // 15
    seasonal_part = seasonal_pattern.reindex(tod_block).values

    # Remainder forecast = 0
    yc_decomp_val.loc[block_idx] = trend_part + seasonal_part

assert yc_decomp_val.notna().all(), 'Decomposition forecast has missing values'

# ---- Plot 1: Overview ----
# Full training + validation with the Classical Decomposition rolling
# 4-step-ahead forecast overlaid on the validation window.
plt.figure(figsize=(16, 6))
plt.plot(yc_price_train.index, yc_price_train.values,
         color='gray', alpha=0.4, label='Training actual (transformed RRP)')
plt.plot(yc_price_val.index, yc_price_val.values,
         color='black', linewidth=1, label='Validation actual (transformed RRP)')
plt.plot(yc_decomp_val.index, yc_decomp_val.values,
         color='tab:green', linewidth=1.2,
         label='Decomposition rolling 4-step-ahead forecast')
plt.axvline(yc_price_val.index[0], color='red', linestyle='--', alpha=0.5,
            label='Train / Val boundary')
plt.title('B3: Overview - Classical Decomposition 4-step-ahead validation')
plt.xlabel('Time')
plt.ylabel('Transformed RRP: arcsinh(RRP / 100)')
plt.legend()
plt.tight_layout()
plt.show()

# ---- Plot 2: Zoomed validation view ----
# Last 3 training days + first 3 validation days. Visualisation only;
# RMSE in section 2.4 uses the full validation set. Temporary variables
# (train_plot_decomp / val_plot_decomp / decomp_zoom) avoid touching
# yc_price_train / yc_price_val / y_price_train / y_price_val.
ZOOM_TRAIN_DAYS_B3 = 3
ZOOM_VAL_DAYS_B3   = 3
train_plot_decomp = yc_price_train.iloc[-ZOOM_TRAIN_DAYS_B3 * STEPS_PER_DAY:]
val_plot_decomp   = yc_price_val.iloc[:ZOOM_VAL_DAYS_B3 * STEPS_PER_DAY]
decomp_zoom       = yc_decomp_val.reindex(val_plot_decomp.index)

plt.figure(figsize=(16, 6))
plt.plot(train_plot_decomp.index, train_plot_decomp.values,
         color='gray', alpha=0.5,
         label=f'Training actual (last {ZOOM_TRAIN_DAYS_B3} days)')
plt.plot(val_plot_decomp.index, val_plot_decomp.values,
         color='black', linewidth=1,
         label=f'Validation actual (first {ZOOM_VAL_DAYS_B3} days)')
plt.plot(decomp_zoom.index, decomp_zoom.values,
         color='tab:green', linewidth=1.4, marker='.', markersize=4,
         label='Decomposition 4-step-ahead forecast')
plt.axvline(yc_price_val.index[0], color='red', linestyle='--', alpha=0.5,
            label='Train / Val boundary')
plt.title('B3: Zoomed validation view - Classical Decomposition (Rolling 4-step-ahead)')
plt.xlabel('Time')
plt.ylabel('Transformed RRP: arcsinh(RRP / 100)')
plt.legend()
plt.tight_layout()
plt.show()

2.4 RMSE table for Decomposition（Spike and Non spike）

In [ ]:
# Reuse compute_rmse_by_mask defined in section 1.4. If for any reason that
# cell has not been run yet, fall back to defining the helper here.
try:
    compute_rmse_by_mask
except NameError:
    def compute_rmse_by_mask(actual, predicted, mask=None):
        a = actual.dropna()
        p = predicted.reindex(a.index).dropna()
        common = a.index.intersection(p.index)
        if mask is not None:
            m_aligned = mask.reindex(common).fillna(False).astype(bool)
            common    = common[m_aligned]
        if len(common) == 0:
            return float('nan'), 0
        err = a.loc[common].values - p.loc[common].values
        return float(np.sqrt(np.mean(err ** 2))), int(len(common))

# Same training spike / non-spike masks as section 1.4 (reuse if available).
try:
    training_spike_mask
    training_non_spike_mask
except NameError:
    training_spike_mask = (
        (y_price_train < lower_spike_threshold) |
        (y_price_train > upper_spike_threshold)
    )
    training_non_spike_mask = ~training_spike_mask


# Training in-sample fitted values for Classical Decomposition
yc_decomp_train_fitted  = trend_refined + seasonal_component
decomp_train_fitted_rrp = inverse_price_transform(yc_decomp_train_fitted)

# Validation forecast on the original RRP scale
# (decomposition validation forecast logic in cell above is preserved; we
#  only inverse-transform here.)
decomp_val_rrp = inverse_price_transform(yc_decomp_val)


# Table 1: Training RMSE diagnostic for Classical Decomposition
overall_rmse, _  = compute_rmse_by_mask(y_price_train, decomp_train_fitted_rrp)
spike_rmse,   sn = compute_rmse_by_mask(y_price_train, decomp_train_fitted_rrp, training_spike_mask)
ns_rmse,      nn = compute_rmse_by_mask(y_price_train, decomp_train_fitted_rrp, training_non_spike_mask)

training_rmse_table_b3 = pd.DataFrame([{
    'Method'                  : 'Classical Decomposition',
    'Training Overall RMSE'   : overall_rmse,
    'Training Spike RMSE'     : spike_rmse,
    'Training Non-spike RMSE' : ns_rmse,
    'Training spike count'    : sn,
    'Training non-spike count': nn,
}])
print("Table 1: Training RMSE diagnostic for Classical Decomposition (original RRP, $/MWh)")
display(training_rmse_table_b3.round(3))


# Table 2: Validation RMSE evaluation for Classical Decomposition ----
overall_rmse, _  = compute_rmse_by_mask(y_price_val, decomp_val_rrp)
spike_rmse,   sn = compute_rmse_by_mask(y_price_val, decomp_val_rrp, spike_mask_val)
ns_rmse,      nn = compute_rmse_by_mask(y_price_val, decomp_val_rrp, non_spike_mask_val)

validation_rmse_table_b3 = pd.DataFrame([{
    'Method'                    : 'Classical Decomposition',
    'Validation Overall RMSE'   : overall_rmse,
    'Validation Spike RMSE'     : spike_rmse,
    'Validation Non-spike RMSE' : ns_rmse,
    'Validation spike count'    : sn,
    'Validation non-spike count': nn,
}])
print("Table 2: Validation RMSE evaluation for Classical Decomposition (original RRP, $/MWh)")
display(validation_rmse_table_b3.round(3))

Here I use two models:
Model 1: transformed RRP recent lags + calendar features
Model 2: transformed RRP recent lags + calendar features + demand predictors
Lags: 1,2,3,4,96,672
Calander features: daily sine/cosine, day-of-week dummies, weekend indicator
Use Ridge regression rather than plain OLS because the feature set contains correlated predictors and Ridge is often more stable.
No spike predictor was used because no volatility clustering was found.

In [ ]:

# ============================================================
# Regression models for RRP forecasting with forced core lags
# Version: forced RRP lags + BIC selection for optional variables
# ============================================================
#
# Assumed existing variables:
#   y_price_train      : original-scale RRP training series, pandas Series with DatetimeIndex
#   y_price_val        : original-scale RRP validation series, pandas Series with DatetimeIndex
#   yc_demand_train    : TOTALDEMAND training series, pandas Series with DatetimeIndex
#   yc_demand_val      : TOTALDEMAND validation series, pandas Series with DatetimeIndex
#
# What this code does:
#   Model A = forced RRP lags + BIC-selected optional RRP/calendar/trend/spike variables
#   Model B = Model A candidates + BIC-selected demand variables
#
# Key design change:
#   BIC is NOT allowed to remove core RRP lags [1,2,3,4,96].
#   BIC only removes optional variables.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_acf
from sklearn.metrics import mean_squared_error


# ============================================================
# 1. Tunable parameters
# ============================================================

FORECAST_HORIZON = 4
ASINH_SCALE = 100
SEASONAL_PERIOD = 96
IQR_MULTIPLIER = 1.5

# Core lags forced to remain in the model.
FORCED_RRP_LAGS = [1, 2, 3, 4, 5, 6, 7, 8, 96]

# Optional RRP lags tested by BIC.
# Keep 672 out by default because it costs one week of training rows.
# OPTIONAL_RRP_LAGS = [5, 6, 7, 8]
OPTIONAL_RRP_LAGS = [9, 10]

# Demand lags tested by BIC in Model B.
DEMAND_LAGS = [0, 1, 2, 3, 4, 96]


# ============================================================
# 2. Basic preparation
# ============================================================

y_price_train = y_price_train.sort_index().astype(float)
y_price_val = y_price_val.sort_index().astype(float)

yc_demand_train = yc_demand_train.sort_index().astype(float)
yc_demand_val = yc_demand_val.sort_index().astype(float)

rrp_raw_full = pd.concat([y_price_train, y_price_val]).sort_index()
demand_full = pd.concat([yc_demand_train, yc_demand_val]).sort_index()

train_end_time = y_price_train.index.max()
val_start_time = y_price_val.index.min()
val_end_time = y_price_val.index.max()

rrp_asinh_full = np.arcsinh(rrp_raw_full / ASINH_SCALE)


# ============================================================
# 3. Spike dummy and validation spike masks
# ============================================================

Q1_price = y_price_train.quantile(0.25)
Q3_price = y_price_train.quantile(0.75)
IQR_price = Q3_price - Q1_price

lower_spike_threshold = Q1_price - IQR_MULTIPLIER * IQR_price
upper_spike_threshold = Q3_price + IQR_MULTIPLIER * IQR_price

# Spike dummy at forecast origin.
spike_dummy_full = (
    (rrp_raw_full < lower_spike_threshold) |
    (rrp_raw_full > upper_spike_threshold)
).astype(int)

# Spike/non-spike masks for validation evaluation.
spike_mask_val = (
    (y_price_val < lower_spike_threshold) |
    (y_price_val > upper_spike_threshold)
)
non_spike_mask_val = ~spike_mask_val

print("Spike threshold lower:", round(lower_spike_threshold, 3))
print("Spike threshold upper:", round(upper_spike_threshold, 3))
print("Validation spike count:", int(spike_mask_val.sum()))
print("Validation non-spike count:", int(non_spike_mask_val.sum()))


# ============================================================
# 4. Build modelling table
# ============================================================

df_model = pd.DataFrame(index=rrp_raw_full.index)

# RRP lag features.
all_rrp_lags = sorted(set(FORCED_RRP_LAGS + OPTIONAL_RRP_LAGS))

for lag in all_rrp_lags:
    df_model[f"rrp_asinh_lag_{lag}"] = rrp_asinh_full.shift(lag)

# Demand lag features.
for lag in DEMAND_LAGS:
    if lag == 0:
        df_model["demand_t"] = demand_full
    else:
        df_model[f"demand_lag_{lag}"] = demand_full.shift(lag)

# Spike dummy at forecast origin.
df_model["spike_dummy_origin"] = spike_dummy_full

# Targets.
df_model["target_rrp_asinh"] = rrp_asinh_full.shift(-FORECAST_HORIZON)
df_model["target_rrp_raw"] = rrp_raw_full.shift(-FORECAST_HORIZON)

# Target timestamp: the timestamp being forecast, t+4.
target_timestamp = pd.Series(df_model.index, index=df_model.index).shift(-FORECAST_HORIZON)
df_model["target_timestamp"] = target_timestamp

target_dt = pd.to_datetime(df_model["target_timestamp"])

# Trend feature.
df_model["trend"] = np.arange(len(df_model))

# Daily seasonal terms.
slot_in_day = target_dt.dt.hour * 4 + target_dt.dt.minute // 15
df_model["slot_in_day"] = slot_in_day
df_model["sin_daily"] = np.sin(2 * np.pi * slot_in_day / SEASONAL_PERIOD)
df_model["cos_daily"] = np.cos(2 * np.pi * slot_in_day / SEASONAL_PERIOD)

# Calendar features.
df_model["hour_of_day"] = target_dt.dt.hour
df_model["day_of_week"] = target_dt.dt.dayofweek
df_model["is_weekend"] = df_model["day_of_week"].isin([5, 6]).astype(int)
df_model["pm_dummy"] = (target_dt.dt.hour >= 12).astype(int)

hour_dummies = pd.get_dummies(
    df_model["hour_of_day"],
    prefix="hour",
    drop_first=True,
    dtype=int
)

dow_dummies = pd.get_dummies(
    df_model["day_of_week"],
    prefix="dow",
    drop_first=True,
    dtype=int
)

# Flexible daily seasonality dummies.
# If this is too many variables, comment out slot_dummies and remove it below.
slot_dummies = pd.get_dummies(
    df_model["slot_in_day"],
    prefix="slot",
    drop_first=True,
    dtype=int
)

df_model = pd.concat([df_model, hour_dummies, dow_dummies, slot_dummies], axis=1)


# ============================================================
# 5. Define train and validation design rows
# ============================================================

train_mask = df_model["target_timestamp"] <= train_end_time

val_mask = (
    (df_model["target_timestamp"] >= val_start_time) &
    (df_model["target_timestamp"] <= val_end_time)
)

df_train = df_model.loc[train_mask].dropna()
df_val = df_model.loc[val_mask].dropna()

print("Training rows used:", len(df_train))
print("Validation rows used:", len(df_val))


# ============================================================
# 6. Feature sets
# ============================================================

forced_features = [f"rrp_asinh_lag_{lag}" for lag in FORCED_RRP_LAGS]

optional_rrp_features = [f"rrp_asinh_lag_{lag}" for lag in OPTIONAL_RRP_LAGS]

calendar_candidate_features = (
    ["trend", "sin_daily", "cos_daily", "is_weekend", "pm_dummy", "spike_dummy_origin"]
    + list(hour_dummies.columns)
    + list(dow_dummies.columns)
    + list(slot_dummies.columns)
)

demand_candidate_features = []
for lag in DEMAND_LAGS:
    if lag == 0:
        demand_candidate_features.append("demand_t")
    else:
        demand_candidate_features.append(f"demand_lag_{lag}")

candidate_features_A = optional_rrp_features + calendar_candidate_features
candidate_features_B = optional_rrp_features + calendar_candidate_features + demand_candidate_features


# ============================================================
# 7. BIC backward selection with forced variables
# ============================================================

def fit_ols_bic(feature_list):
    X = df_train[feature_list].apply(pd.to_numeric, errors="coerce").astype(float)
    X = sm.add_constant(X, has_constant="add")
    y = pd.to_numeric(df_train["target_rrp_asinh"], errors="coerce").astype(float)

    model = sm.OLS(y, X)
    result = model.fit()
    return result


def backward_bic_selection(forced_features, candidate_features, verbose=True):
    selected_candidates = list(candidate_features)
    current_features = list(forced_features) + selected_candidates

    current_result = fit_ols_bic(current_features)
    current_bic = current_result.bic

    if verbose:
        print("Initial BIC:", round(current_bic, 3))
        print("Initial number of features:", len(current_features))

    improved = True

    while improved and len(selected_candidates) > 0:
        improved = False
        trial_records = []

        for feature in selected_candidates:
            trial_candidates = [f for f in selected_candidates if f != feature]
            trial_features = list(forced_features) + trial_candidates

            try:
                trial_result = fit_ols_bic(trial_features)
                trial_records.append({
                    "removed_feature": feature,
                    "bic": trial_result.bic,
                    "result": trial_result,
                    "trial_candidates": trial_candidates
                })
            except Exception:
                pass

        if len(trial_records) == 0:
            break

        trial_df = pd.DataFrame(trial_records)
        best_trial = trial_df.loc[trial_df["bic"].idxmin()]

        if best_trial["bic"] < current_bic:
            current_bic = best_trial["bic"]
            selected_candidates = best_trial["trial_candidates"]
            current_result = best_trial["result"]
            improved = True

            if verbose:
                print(
                    "Removed:",
                    best_trial["removed_feature"],
                    "| New BIC:",
                    round(current_bic, 3)
                )

    final_features = list(forced_features) + selected_candidates

    if verbose:
        print("Final BIC:", round(current_bic, 3))
        print("Final number of features:", len(final_features))
        print("Forced features kept:", forced_features)

    return final_features, current_result


# ============================================================
# 8. Evaluation functions
# ============================================================

def inverse_asinh(y_asinh):
    return ASINH_SCALE * np.sinh(y_asinh)


def evaluate_model(result, final_features, model_name):
    # Training fitted values.
    X_train = df_train[final_features].apply(pd.to_numeric, errors="coerce").astype(float)
    X_train = sm.add_constant(X_train, has_constant="add")
    train_pred_asinh = result.predict(X_train)
    train_pred_raw = inverse_asinh(train_pred_asinh)

    train_actual_raw = df_train["target_rrp_raw"]
    train_residual_raw = train_actual_raw - train_pred_raw

    training_rmse = np.sqrt(mean_squared_error(train_actual_raw, train_pred_raw))

    # Validation predictions.
    X_val = df_val[final_features].apply(pd.to_numeric, errors="coerce").astype(float)
    X_val = sm.add_constant(X_val, has_constant="add")
    val_pred_asinh = result.predict(X_val)
    val_pred_raw = inverse_asinh(val_pred_asinh)

    val_actual_raw = df_val["target_rrp_raw"]
    val_error_raw = val_actual_raw - val_pred_raw

    validation_rmse = np.sqrt(mean_squared_error(val_actual_raw, val_pred_raw))

    # Spike/non-spike validation RMSE.
    target_times = pd.to_datetime(df_val["target_timestamp"])

    aligned_spike_mask = target_times.map(spike_mask_val).astype(bool)
    aligned_non_spike_mask = ~aligned_spike_mask

    if aligned_spike_mask.sum() > 0:
        spike_rmse = np.sqrt(
            mean_squared_error(
                val_actual_raw[aligned_spike_mask],
                val_pred_raw[aligned_spike_mask]
            )
        )
    else:
        spike_rmse = np.nan

    if aligned_non_spike_mask.sum() > 0:
        non_spike_rmse = np.sqrt(
            mean_squared_error(
                val_actual_raw[aligned_non_spike_mask],
                val_pred_raw[aligned_non_spike_mask]
            )
        )
    else:
        non_spike_rmse = np.nan

    train_results = pd.DataFrame({
        "target_timestamp": df_train["target_timestamp"],
        "actual_rrp_raw": train_actual_raw,
        "fitted_rrp_raw": train_pred_raw,
        "residual_raw": train_residual_raw
    }, index=df_train.index)

    val_results = pd.DataFrame({
        "target_timestamp": df_val["target_timestamp"],
        "actual_rrp_raw": val_actual_raw,
        "predicted_rrp_raw": val_pred_raw,
        "error_raw": val_error_raw,
        "is_spike": aligned_spike_mask
    }, index=df_val.index)

    metrics = {
        "Model": model_name,
        "BIC": result.bic,
        "AIC": result.aic,
        "Num Features": len(final_features),
        "Training RMSE": training_rmse,
        "Validation RMSE": validation_rmse,
        "Spike RMSE": spike_rmse,
        "Non-spike RMSE": non_spike_rmse
    }

    return metrics, train_results, val_results


# ============================================================
# 9. Fit Model A and Model B
# ============================================================

print("\n" + "=" * 90)
print("MODEL A: Forced RRP lags + BIC-selected optional variables")
print("=" * 90)

features_A, result_A = backward_bic_selection(
    forced_features=forced_features,
    candidate_features=candidate_features_A,
    verbose=True
)

metrics_A, train_results_A, val_results_A = evaluate_model(
    result=result_A,
    final_features=features_A,
    model_name="Model A: RRP lags + calendar/trend/spike"
)

print("\nSelected Model A features:")
print(features_A)


print("\n" + "=" * 90)
print("MODEL B: Model A candidates + BIC-selected demand variables")
print("=" * 90)

features_B, result_B = backward_bic_selection(
    forced_features=forced_features,
    candidate_features=candidate_features_B,
    verbose=True
)

metrics_B, train_results_B, val_results_B = evaluate_model(
    result=result_B,
    final_features=features_B,
    model_name="Model B: Model A + demand lags"
)

print("\nSelected Model B features:")
print(features_B)


# ============================================================
# 10. Compare Model A and Model B
# ============================================================

comparison_table = pd.DataFrame([metrics_A, metrics_B])

print("\n" + "=" * 110)
print("FINAL REGRESSION MODEL COMPARISON")
print("=" * 110)

display(comparison_table.round(3))


# ============================================================
# 11. Key coefficient interpretation
# ============================================================

def print_key_coefficients(result, features, model_name):
    print("\n" + "=" * 90)
    print(f"KEY COEFFICIENTS: {model_name}")
    print("=" * 90)

    params = result.params

    for key in ["spike_dummy_origin", "trend", "pm_dummy", "is_weekend"]:
        if key in params.index:
            print(f"{key}: {params[key]:.6f}")
        else:
            print(f"{key}: not selected")

    demand_selected = [f for f in features if f.startswith("demand")]
    print("Demand features selected:", demand_selected if len(demand_selected) > 0 else "None")


print_key_coefficients(result_A, features_A, "Model A")
print_key_coefficients(result_B, features_B, "Model B")


# ============================================================
# 12. Plot actual vs predicted validation forecasts
# ============================================================

plt.figure(figsize=(16, 6))

plt.plot(y_price_train.index, y_price_train, label="Training actual RRP")
plt.plot(y_price_val.index, y_price_val, label="Validation actual RRP")

plt.plot(
    val_results_A["target_timestamp"],
    val_results_A["predicted_rrp_raw"],
    label="Model A prediction",
    linestyle="--"
)

plt.plot(
    val_results_B["target_timestamp"],
    val_results_B["predicted_rrp_raw"],
    label="Model B prediction",
    linestyle="--"
)

plt.title("Regression Forecasts for RRP: Actual vs Predicted")
plt.xlabel("Time")
plt.ylabel("RRP ($/MWh)")
plt.legend()
plt.tight_layout()
plt.show()


plt.figure(figsize=(16, 6))

plt.plot(y_price_train.index, y_price_train, label="Training actual RRP")
plt.plot(y_price_val.index, y_price_val, label="Validation actual RRP")

plt.plot(
    val_results_A["target_timestamp"],
    val_results_A["predicted_rrp_raw"],
    label="Model A prediction",
    linestyle="--"
)

plt.plot(
    val_results_B["target_timestamp"],
    val_results_B["predicted_rrp_raw"],
    label="Model B prediction",
    linestyle="--"
)

plt.yscale("symlog", linthresh=100)
plt.title("Regression Forecasts for RRP: Actual vs Predicted, Symlog Scale")
plt.xlabel("Time")
plt.ylabel("RRP ($/MWh), symlog scale")
plt.legend()
plt.tight_layout()
plt.show()


# ============================================================
# 13. Residual diagnostics on original scale only
# ============================================================

def residual_diagnostics_original_scale(train_results, val_results, model_name, acf_lags=120):
    train_resid = train_results["residual_raw"].dropna()
    val_error = val_results["error_raw"].dropna()

    # Training residual scatter.
    plt.figure(figsize=(10, 5))
    plt.scatter(
        train_results["fitted_rrp_raw"],
        train_results["residual_raw"],
        alpha=0.5
    )
    plt.axhline(0, linewidth=1)
    plt.title(f"{model_name}: Training fitted RRP vs residuals")
    plt.xlabel("Fitted RRP, original scale")
    plt.ylabel("Training residual, original scale")
    plt.tight_layout()
    plt.show()

    # Training residual ACF.
    fig, ax = plt.subplots(figsize=(12, 5))
    plot_acf(train_resid, lags=acf_lags, ax=ax, zero=False)
    ax.set_title(f"{model_name}: ACF of training residuals, original scale")
    plt.tight_layout()
    plt.show()

    # Validation error scatter.
    plt.figure(figsize=(10, 5))
    plt.scatter(
        val_results["predicted_rrp_raw"],
        val_results["error_raw"],
        alpha=0.5
    )
    plt.axhline(0, linewidth=1)
    plt.title(f"{model_name}: Validation predicted RRP vs forecast errors")
    plt.xlabel("Predicted RRP, original scale")
    plt.ylabel("Validation forecast error, original scale")
    plt.tight_layout()
    plt.show()

    # Validation forecast error ACF.
    fig, ax = plt.subplots(figsize=(12, 5))
    plot_acf(val_error, lags=acf_lags, ax=ax, zero=False)
    ax.set_title(f"{model_name}: ACF of validation forecast errors, original scale")
    plt.tight_layout()
    plt.show()


residual_diagnostics_original_scale(
    train_results=train_results_A,
    val_results=val_results_A,
    model_name="Model A",
    acf_lags=120
)

residual_diagnostics_original_scale(
    train_results=train_results_B,
    val_results=val_results_B,
    model_name="Model B",
    acf_lags=120
)


# ============================================================
# 14. Notes for report
# ============================================================
#
# 1. RRP lags 1,2,3,4,96 were forced because EDA and residual ACF suggested
#    short-run and daily dependence. BIC was used only to select optional variables.
#
# 2. The spike dummy is based on RRP at the forecast origin, so it is known at
#    forecast time and does not leak future target information.
#
# 3. All RMSE values are computed on the original RRP scale after inverse-transforming
#    from asinh(RRP / 100).
#
# 4. If the spike dummy coefficient is positive, current spikes tend to persist into
#    the 1-hour-ahead forecast. If negative, current spikes tend to mean-revert.
# ============================================================


SARIMA for RRP (7,0,1),(1,0,1)96

In [ ]:

# ============================================================
# Selected SARIMA model for transformed RRP forecasting
# Final selected model + diagnostics + RMSE + plots
# ============================================================
#
# Assumed existing variables:
#   y_price_train       : original-scale RRP training series, pandas Series/DataFrame with DatetimeIndex
#   y_price_val         : original-scale RRP validation series, pandas Series/DataFrame with DatetimeIndex
#
# Optional existing variables:
#   spike_mask_val      : Boolean Series indexed by validation timestamps
#   non_spike_mask_val  : Boolean Series indexed by validation timestamps
#
# Selected SARIMA:
#   SARIMA(7,0,1)(1,0,1,96)
#
# Target:
#   asinh(RRP / ASINH_SCALE)
#
# Validation:
#   Fixed-parameter rolling-origin validation.
#   Parameters estimated once on training set.
#   New validation observations appended with refit=False.
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.stats.stattools import jarque_bera
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_squared_error


# ============================================================
# 1. Tunable parameters
# ============================================================

FORECAST_HORIZON = 4
ASINH_SCALE = 100
SEASONAL_PERIOD = 96

SELECTED_ORDER = (7, 0, 1)
SELECTED_SEASONAL_ORDER = (1, 0, 1, SEASONAL_PERIOD)

TREND_TERM = "c"
# With d=0 and D=0, constant is reasonable.
# If convergence is poor, try TREND_TERM = "n".

MAX_ITER = 200

ENFORCE_STATIONARITY = False
ENFORCE_INVERTIBILITY = False

SIMPLE_DIFFERENCING = True
LOW_MEMORY_FIT = True

IQR_MULTIPLIER = 1.5

LJUNG_BOX_LAGS = [1, 2, 3, 4, 8, 12, 24, 48, 96]
ARCH_LAGS = 12

RESIDUAL_ACF_LAGS = 120
RESIDUAL_PACF_LAGS = 120


# ============================================================
# 2. Prepare data
# ============================================================

if isinstance(y_price_train, pd.DataFrame):
    y_price_train = y_price_train.squeeze()

if isinstance(y_price_val, pd.DataFrame):
    y_price_val = y_price_val.squeeze()

y_price_train = pd.to_numeric(
    y_price_train.sort_index(),
    errors="coerce"
).astype(float).dropna()

y_price_val = pd.to_numeric(
    y_price_val.sort_index(),
    errors="coerce"
).astype(float).dropna()

y_price_train.name = "RRP"
y_price_val.name = "RRP"

train_index = y_price_train.index
val_index = y_price_val.index

y_train_asinh = np.arcsinh(y_price_train / ASINH_SCALE)
y_val_asinh = np.arcsinh(y_price_val / ASINH_SCALE)

y_train_asinh_np = np.asarray(y_train_asinh, dtype=float)
y_val_asinh_np = np.asarray(y_val_asinh, dtype=float)

y_train_raw_np = np.asarray(y_price_train, dtype=float)
y_val_raw_np = np.asarray(y_price_val, dtype=float)


# ============================================================
# 3. Define spike / non-spike masks
# ============================================================

if "spike_mask_val" not in globals() or "non_spike_mask_val" not in globals():
    Q1_price = y_price_train.quantile(0.25)
    Q3_price = y_price_train.quantile(0.75)
    IQR_price = Q3_price - Q1_price

    lower_spike_threshold = Q1_price - IQR_MULTIPLIER * IQR_price
    upper_spike_threshold = Q3_price + IQR_MULTIPLIER * IQR_price

    spike_mask_val = (
        (y_price_val < lower_spike_threshold) |
        (y_price_val > upper_spike_threshold)
    )

    non_spike_mask_val = ~spike_mask_val

    print("Spike masks created using training-set IQR thresholds.")
    print("Lower spike threshold:", round(lower_spike_threshold, 3))
    print("Upper spike threshold:", round(upper_spike_threshold, 3))

else:
    if isinstance(spike_mask_val, pd.DataFrame):
        spike_mask_val = spike_mask_val.squeeze()
    if isinstance(non_spike_mask_val, pd.DataFrame):
        non_spike_mask_val = non_spike_mask_val.squeeze()

    spike_mask_val = spike_mask_val.reindex(val_index).fillna(False).astype(bool)
    non_spike_mask_val = non_spike_mask_val.reindex(val_index).fillna(False).astype(bool)

spike_mask_val_np = np.asarray(spike_mask_val, dtype=bool)

print("Validation spike count:", int(spike_mask_val.sum()))
print("Validation non-spike count:", int((~spike_mask_val).sum()))


# ============================================================
# 4. Helper functions
# ============================================================

def inverse_asinh(y_asinh):
    return ASINH_SCALE * np.sinh(y_asinh)


def fit_selected_sarima():
    model = SARIMAX(
        y_train_asinh_np,
        order=SELECTED_ORDER,
        seasonal_order=SELECTED_SEASONAL_ORDER,
        trend=TREND_TERM,
        enforce_stationarity=ENFORCE_STATIONARITY,
        enforce_invertibility=ENFORCE_INVERTIBILITY,
        simple_differencing=SIMPLE_DIFFERENCING
    )

    result = model.fit(
        disp=False,
        maxiter=MAX_ITER,
        low_memory=LOW_MEMORY_FIT
    )

    converged = result.mle_retvals.get("converged", np.nan)

    print("=" * 90)
    print("SELECTED SARIMA MODEL FITTED")
    print("=" * 90)
    print("Order:", SELECTED_ORDER)
    print("Seasonal order:", SELECTED_SEASONAL_ORDER)
    print("Trend:", TREND_TERM)
    print("AIC:", round(result.aic, 3))
    print("BIC:", round(result.bic, 3))
    print("Converged:", converged)
    print()

    return result


def rolling_four_step_forecast_numpy(fitted_result, val_array_asinh, horizon=4):
    n_val = len(val_array_asinh)
    predictions = np.full(n_val, np.nan, dtype=float)

    initial_forecast = fitted_result.get_forecast(steps=horizon).predicted_mean
    initial_forecast = np.asarray(initial_forecast, dtype=float).squeeze()

    first_h = min(horizon, n_val)
    predictions[:first_h] = initial_forecast[:first_h]

    updated_result = fitted_result

    for i in range(n_val - horizon):
        new_observation = np.asarray([val_array_asinh[i]], dtype=float)

        updated_result = updated_result.append(
            new_observation,
            refit=False
        )

        forecast_h = updated_result.get_forecast(steps=horizon).predicted_mean
        forecast_h = np.asarray(forecast_h, dtype=float).squeeze()

        predictions[i + horizon] = forecast_h[-1]

    return predictions


def residual_tests(residuals, label):
    residuals = np.asarray(pd.Series(residuals).dropna(), dtype=float)

    print("=" * 90)
    print(f"RESIDUAL DIAGNOSTIC TESTS: {label}")
    print("=" * 90)

    lb_lags = [lag for lag in LJUNG_BOX_LAGS if lag < len(residuals)]
    if len(lb_lags) > 0:
        lb = acorr_ljungbox(
            residuals,
            lags=lb_lags,
            return_df=True
        )
        print("\nLjung-Box test:")
        print(lb.round(4))
    else:
        print("\nLjung-Box skipped: not enough residual observations.")

    try:
        jb_stat, jb_pvalue, skew, kurtosis = jarque_bera(residuals)
        print("\nJarque-Bera normality test:")
        print("JB statistic:", round(jb_stat, 4))
        print("p-value:", round(jb_pvalue, 4))
        print("skew:", round(skew, 4))
        print("kurtosis:", round(kurtosis, 4))
    except Exception as e:
        print("\nJarque-Bera failed:", e)

    try:
        arch_lags = min(ARCH_LAGS, max(1, len(residuals) // 5))
        arch_stat, arch_pvalue, _, _ = het_arch(
            residuals,
            nlags=arch_lags
        )
        print("\nARCH LM test:")
        print("ARCH statistic:", round(arch_stat, 4))
        print("p-value:", round(arch_pvalue, 4))
        print("lags:", arch_lags)
    except Exception as e:
        print("\nARCH LM failed:", e)

    print()


def evaluate_selected_sarima(fitted_result):
    # -----------------------------
    # Training fitted values
    # -----------------------------
    train_pred_asinh = np.asarray(fitted_result.fittedvalues, dtype=float).squeeze()
    train_pred_asinh = pd.Series(train_pred_asinh).dropna().to_numpy(dtype=float)

    n_fit = len(train_pred_asinh)

    train_actual_asinh = y_train_asinh_np[-n_fit:]
    train_actual_raw = y_train_raw_np[-n_fit:]
    train_results_index = train_index[-n_fit:]

    train_pred_raw = inverse_asinh(train_pred_asinh)

    train_residual_asinh = train_actual_asinh - train_pred_asinh
    train_residual_raw = train_actual_raw - train_pred_raw

    train_rmse = np.sqrt(
        mean_squared_error(train_actual_raw, train_pred_raw)
    )

    train_results = pd.DataFrame({
        "actual_rrp_raw": train_actual_raw,
        "fitted_rrp_raw": train_pred_raw,
        "residual_raw": train_residual_raw,
        "actual_rrp_asinh": train_actual_asinh,
        "fitted_rrp_asinh": train_pred_asinh,
        "residual_asinh": train_residual_asinh
    }, index=train_results_index)

    # -----------------------------
    # Rolling-origin validation
    # -----------------------------
    val_pred_asinh = rolling_four_step_forecast_numpy(
        fitted_result=fitted_result,
        val_array_asinh=y_val_asinh_np,
        horizon=FORECAST_HORIZON
    )

    valid_mask = ~np.isnan(val_pred_asinh)

    val_pred_asinh_valid = val_pred_asinh[valid_mask]
    val_actual_asinh_valid = y_val_asinh_np[valid_mask]
    val_actual_raw_valid = y_val_raw_np[valid_mask]
    val_index_valid = val_index[valid_mask]
    spike_mask_valid = spike_mask_val_np[valid_mask]

    val_pred_raw_valid = inverse_asinh(val_pred_asinh_valid)

    val_error_asinh = val_actual_asinh_valid - val_pred_asinh_valid
    val_error_raw = val_actual_raw_valid - val_pred_raw_valid

    validation_rmse = np.sqrt(
        mean_squared_error(val_actual_raw_valid, val_pred_raw_valid)
    )

    # -----------------------------
    # Spike and non-spike RMSE
    # -----------------------------
    if spike_mask_valid.sum() > 0:
        spike_rmse = np.sqrt(
            mean_squared_error(
                val_actual_raw_valid[spike_mask_valid],
                val_pred_raw_valid[spike_mask_valid]
            )
        )
    else:
        spike_rmse = np.nan

    non_spike_mask_valid = ~spike_mask_valid

    if non_spike_mask_valid.sum() > 0:
        non_spike_rmse = np.sqrt(
            mean_squared_error(
                val_actual_raw_valid[non_spike_mask_valid],
                val_pred_raw_valid[non_spike_mask_valid]
            )
        )
    else:
        non_spike_rmse = np.nan

    val_results = pd.DataFrame({
        "actual_rrp_raw": val_actual_raw_valid,
        "predicted_rrp_raw": val_pred_raw_valid,
        "error_raw": val_error_raw,
        "actual_rrp_asinh": val_actual_asinh_valid,
        "predicted_rrp_asinh": val_pred_asinh_valid,
        "error_asinh": val_error_asinh,
        "is_spike": spike_mask_valid
    }, index=val_index_valid)

    metrics = {
        "Model": f"SARIMA{SELECTED_ORDER}x{SELECTED_SEASONAL_ORDER}",
        "Order": str(SELECTED_ORDER),
        "Seasonal Order": str(SELECTED_SEASONAL_ORDER),
        "Trend": TREND_TERM,
        "AIC": fitted_result.aic,
        "BIC": fitted_result.bic,
        "Converged": fitted_result.mle_retvals.get("converged", np.nan),
        "Training RMSE": train_rmse,
        "Validation RMSE": validation_rmse,
        "Spike RMSE": spike_rmse,
        "Non-spike RMSE": non_spike_rmse
    }

    return metrics, train_results, val_results


def plot_acf_pacf_for_residuals(series, title_prefix, lags_acf=120, lags_pacf=120):
    series = pd.Series(series).dropna()

    fig, axes = plt.subplots(2, 1, figsize=(14, 7))

    plot_acf(series, lags=lags_acf, ax=axes[0], alpha=0.05, zero=False)
    axes[0].set_title(f"{title_prefix}: ACF")

    max_pacf_lags = min(lags_pacf, max(1, len(series) // 2 - 1))
    plot_pacf(series, lags=max_pacf_lags, ax=axes[1], alpha=0.05, zero=False, method="ywm")
    axes[1].set_title(f"{title_prefix}: PACF")

    plt.tight_layout()
    plt.show()


def plot_residual_time_series(train_results, val_results):
    plt.figure(figsize=(14, 4))
    plt.plot(train_results.index, train_results["residual_asinh"])
    plt.axhline(0, linewidth=1)
    plt.title("Training residuals, transformed scale")
    plt.xlabel("Time")
    plt.ylabel("Residual: asinh scale")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(14, 4))
    plt.plot(val_results.index, val_results["error_asinh"])
    plt.axhline(0, linewidth=1)
    plt.title("Validation forecast errors, transformed scale")
    plt.xlabel("Time")
    plt.ylabel("Forecast error: asinh scale")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(14, 4))
    plt.plot(train_results.index, train_results["residual_raw"])
    plt.axhline(0, linewidth=1)
    plt.title("Training residuals, original RRP scale")
    plt.xlabel("Time")
    plt.ylabel("Residual: $/MWh")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(14, 4))
    plt.plot(val_results.index, val_results["error_raw"])
    plt.axhline(0, linewidth=1)
    plt.title("Validation forecast errors, original RRP scale")
    plt.xlabel("Time")
    plt.ylabel("Forecast error: $/MWh")
    plt.tight_layout()
    plt.show()


def plot_validation_actual_vs_prediction(val_results):
    plt.figure(figsize=(16, 6))

    plt.plot(
        val_results.index,
        val_results["actual_rrp_raw"],
        label="Validation actual RRP"
    )

    plt.plot(
        val_results.index,
        val_results["predicted_rrp_raw"],
        label="SARIMA predicted RRP",
        linestyle="--"
    )

    plt.title(f"Validation Actual RRP vs SARIMA Prediction: SARIMA{SELECTED_ORDER}x{SELECTED_SEASONAL_ORDER}")
    plt.xlabel("Time")
    plt.ylabel("RRP ($/MWh)")
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(16, 6))

    plt.plot(
        val_results.index,
        val_results["actual_rrp_raw"],
        label="Validation actual RRP"
    )

    plt.plot(
        val_results.index,
        val_results["predicted_rrp_raw"],
        label="SARIMA predicted RRP",
        linestyle="--"
    )

    plt.yscale("symlog", linthresh=100)
    plt.title(f"Validation Actual vs SARIMA Prediction, Symlog Scale: SARIMA{SELECTED_ORDER}x{SELECTED_SEASONAL_ORDER}")
    plt.xlabel("Time")
    plt.ylabel("RRP ($/MWh), symlog scale")
    plt.legend()
    plt.tight_layout()
    plt.show()


# ============================================================
# 5. Fit selected SARIMA and evaluate
# ============================================================

selected_result = fit_selected_sarima()

sarima_metrics, sarima_train_results, sarima_val_results = evaluate_selected_sarima(
    fitted_result=selected_result
)

print("=" * 90)
print("SARIMA PERFORMANCE SUMMARY")
print("=" * 90)
sarima_summary_table = pd.DataFrame([sarima_metrics])
display(sarima_summary_table.round(3))


# ============================================================
# 6. Residual diagnostic tests
# ============================================================

# Transformed-scale diagnostics are primary because the model is fitted on transformed RRP.
residual_tests(
    residuals=sarima_train_results["residual_asinh"],
    label="Training residuals, transformed scale"
)

residual_tests(
    residuals=sarima_val_results["error_asinh"],
    label="Validation forecast errors, transformed scale"
)

# Original-scale diagnostics are useful for reporting error behaviour.
residual_tests(
    residuals=sarima_train_results["residual_raw"],
    label="Training residuals, original RRP scale"
)

residual_tests(
    residuals=sarima_val_results["error_raw"],
    label="Validation forecast errors, original RRP scale"
)


# ============================================================
# 7. Residual ACF/PACF plots
# ============================================================

plot_acf_pacf_for_residuals(
    series=sarima_train_results["residual_asinh"],
    title_prefix="Training residuals, transformed scale",
    lags_acf=RESIDUAL_ACF_LAGS,
    lags_pacf=RESIDUAL_PACF_LAGS
)

plot_acf_pacf_for_residuals(
    series=sarima_val_results["error_asinh"],
    title_prefix="Validation forecast errors, transformed scale",
    lags_acf=RESIDUAL_ACF_LAGS,
    lags_pacf=RESIDUAL_PACF_LAGS
)


# ============================================================
# 8. Residual time-series plots
# ============================================================

plot_residual_time_series(
    train_results=sarima_train_results,
    val_results=sarima_val_results
)


# ============================================================
# 9. Validation actual vs predicted plots
# ============================================================

plot_validation_actual_vs_prediction(
    val_results=sarima_val_results
)


# ============================================================
# 10. Suggested report notes
# ============================================================
#
# 1. The selected SARIMA(7,0,1)(1,0,1,96) model was fitted to transformed RRP,
#    asinh(RRP / 100), to reduce the influence of extreme positive spikes while
#    preserving negative prices.
#
# 2. d = 0 and D = 0 mean the final model captures persistence and daily
#    seasonality through ARMA and seasonal ARMA terms rather than differencing.
#
# 3. AIC/BIC, convergence status, validation RMSE, spike/non-spike RMSE, and
#    residual diagnostics should all be used to judge adequacy.
#
# 4. Ljung-Box p-values below 0.05 indicate remaining residual autocorrelation.
#    Jarque-Bera may reject normality because RRP contains extreme price spikes.
#    ARCH LM significance suggests remaining conditional heteroskedasticity.
# ============================================================


# **Task C: Deep Learning for Price Forecasting**

1. The **Multi-Layer Perceptron (MLP) with lagged inputs** model

1.1 Problem Setup

- The forecast horizon is `HORIZON = 4`, i.e. the next four 15-minute observations (one hour ahead).
- The MLP is a **direct multi-output** forecaster: a single forward pass predicts all four future transformed RRP values jointly.
- Training data fit the model; validation data are used for hyperparameter selection.
- All random seeds are fixed for reproducibility.

In [ ]:
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Confirm forecasting parameters
try:
    HORIZON
except NameError:
    HORIZON = 4
try:
    STEPS_PER_DAY
except NameError:
    STEPS_PER_DAY = 96

print(f"HORIZON       = {HORIZON}")
print(f"STEPS_PER_DAY = {STEPS_PER_DAY}")
print(f"len(yc_price_train) = {len(yc_price_train)}")
print(f"len(yc_price_val)   = {len(yc_price_val)}")
print(f"len(yc_price_test)  = {len(yc_price_test)}")

# Inverse transform helper (reuse if defined earlier)
try:
    inverse_price_transform
except NameError:
    def inverse_price_transform(yc):
        return 100 * np.sinh(yc)

# RMSE helper (reuse if defined earlier)
try:
    rmse
except NameError:
    def rmse(y_true, y_pred):
        y_true = np.asarray(y_true, dtype=float)
        y_pred = np.asarray(y_pred, dtype=float)
        return np.sqrt(np.mean((y_true - y_pred) ** 2))

1.2 Construct Lagged Supervised Learning Dataset

We convert the univariate transformed-RRP series into a supervised learning dataset:

- **Input vector** `x_t` = the past `L` (lookback) transformed-RRP values strictly before time `t`.
- **Target vector** `y_t` = the next `HORIZON = 4` transformed-RRP values at and after time `t`.

A baseline lookback of `L = 96` (one full day) is justified by the 15-minute frequency. Other lookbacks (e.g. 48, 192) are explored during hyperparameter tuning.

**No leakage rules.** Training windows are drawn only from `yc_price_train`. Validation windows are drawn so that **targets always lie strictly inside `yc_price_val.index`**, but their **inputs may use the last `L` observations of training history** (and later validation history that has already been observed before the forecast block). This mirrors how the model would be deployed sequentially. The test set is never touched.

In [ ]:
def make_lagged_windows(series, lookback, horizon, target_start_idx, target_end_idx):
    # Build (X, y, target_timestamps) sliding windows.
    # series           : pandas Series of transformed RRP, DatetimeIndex
    # lookback         : L (number of past observations as input)
    # horizon          : H (number of future observations as target)
    # target_start_idx : first integer position (in series) where a target block starts
    # target_end_idx   : one past the last integer position where a target block starts
    values = series.values
    index  = series.index

    X_list, y_list, ts_list = [], [], []
    for i in range(target_start_idx, target_end_idx):
        if i - lookback < 0 or i + horizon > len(series):
            continue
        X_list.append(values[i - lookback : i])
        y_list.append(values[i : i + horizon])
        ts_list.append(index[i : i + horizon])

    X  = np.asarray(X_list, dtype=np.float32)
    y  = np.asarray(y_list, dtype=np.float32)
    ts = pd.DatetimeIndex(np.concatenate(ts_list)).values.reshape(-1, horizon) \
            if ts_list else np.empty((0, horizon), dtype='datetime64[ns]')
    return X, y, ts


def build_train_val_windows(lookback, horizon):
    # Construct training windows from yc_price_train only, and validation
    # windows from the combined train+val series with targets restricted to
    # yc_price_val.index.
    n_train = len(yc_price_train)
    n_val   = len(yc_price_val)

    # Training windows
    X_train, y_train, ts_train = make_lagged_windows(
        series           = yc_price_train,
        lookback         = lookback,
        horizon          = horizon,
        target_start_idx = lookback,         # first input window fits inside training
        target_end_idx   = n_train - horizon + 1,
    )

    # Validation windows on the combined history
    combined = pd.concat([yc_price_train, yc_price_val]).sort_index()
    val_target_start = n_train                          # first val target = val[0]
    val_target_end   = n_train + n_val - horizon + 1    # last target block fits in val
    X_val, y_val, ts_val = make_lagged_windows(
        series           = combined,
        lookback         = lookback,
        horizon          = horizon,
        target_start_idx = val_target_start,
        target_end_idx   = val_target_end,
    )

    # Sanity: every validation target timestamp must lie in yc_price_val.index.
    # Use pd.DatetimeIndex.isin so numpy.datetime64 (from ts_val) and
    # pd.Timestamp (from yc_price_val.index) compare correctly.
    flat_val_ts = pd.DatetimeIndex(ts_val.flatten())
    assert flat_val_ts.isin(yc_price_val.index).all(), \
        'Validation target timestamps must all lie within yc_price_val.index'

    return X_train, y_train, ts_train, X_val, y_val, ts_val


# Quick shape check with the baseline lookback
LOOKBACK_BASELINE = 96
X_tr, y_tr, ts_tr, X_va, y_va, ts_va = build_train_val_windows(LOOKBACK_BASELINE, HORIZON)
print(f"Baseline lookback L = {LOOKBACK_BASELINE}")
print(f"X_train shape: {X_tr.shape}   y_train shape: {y_tr.shape}")
print(f"X_val   shape: {X_va.shape}   y_val   shape: {y_va.shape}")

1.3 Scaling and PyTorch DataLoaders

Neural networks train more stably when inputs are standardised. Following the Efficient BackProp guidance, we **fit the input scaler on training `X` only** and then apply it to validation `X` — no leakage.

Targets are kept on the transformed RRP scale (i.e. `arcsinh(RRP / 100)`), so the MSE loss is also on this scale. We only inverse-transform predictions when reporting RMSE on the original RRP scale.

In [ ]:
def make_dataloaders(X_train, y_train, X_val, y_val, batch_size):
    # StandardScaler fit on training X only
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train).astype(np.float32)
    X_val_s   = scaler.transform(X_val).astype(np.float32)

    # Tensors
    X_train_t = torch.from_numpy(X_train_s)
    y_train_t = torch.from_numpy(y_train.astype(np.float32))
    X_val_t   = torch.from_numpy(X_val_s)
    y_val_t   = torch.from_numpy(y_val.astype(np.float32))

    train_loader = DataLoader(
        TensorDataset(X_train_t, y_train_t),
        batch_size=batch_size, shuffle=True
    )
    val_loader = DataLoader(
        TensorDataset(X_val_t, y_val_t),
        batch_size=batch_size, shuffle=False
    )
    return scaler, train_loader, val_loader, X_val_t, y_val_t

1.4 Define MLP Architecture

A simple feed-forward MLP:

- **Input layer** dimension = `lookback`.
- **Hidden layers** with ReLU activation and dropout for regularisation.
- **Output layer** dimension = `HORIZON = 4`, with no activation (regression).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class PriceMLP(nn.Module):
    def __init__(self, input_dim, hidden_layers, dropout, output_dim):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_layers:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, output_dim))   # linear output for regression
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

1.5 Training Function

The model is trained with:

- **MSE loss** on the transformed RRP scale.
- **Adam optimiser**.
- **Mini-batch training** on shuffled training batches.
- **Early stopping** based on validation loss with a configurable `patience`.
- **Best-weights restoration**: when training stops, the model is reverted to the epoch with the lowest validation loss.

In [ ]:
def train_mlp_model(config):
    # config keys: lookback, hidden_layers, learning_rate, dropout,
    #              batch_size, max_epochs, patience
    lookback      = config['lookback']
    hidden_layers = config['hidden_layers']
    learning_rate = config['learning_rate']
    dropout       = config['dropout']
    batch_size    = config['batch_size']
    max_epochs    = config['max_epochs']
    patience      = config['patience']

    # Reset seeds inside each experiment so different configs are comparable
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    # 1) Lagged windows for this lookback
    X_train, y_train, ts_train, X_val, y_val, ts_val = \
        build_train_val_windows(lookback, HORIZON)

    # 2) Scale and build loaders
    scaler, train_loader, val_loader, X_val_t, y_val_t = make_dataloaders(
        X_train, y_train, X_val, y_val, batch_size
    )

    # 3) Model, loss, optimiser
    model     = PriceMLP(lookback, hidden_layers, dropout, HORIZON).to(device)
    loss_fn   = nn.MSELoss()
    optimiser = torch.optim.Adam(model.parameters(), lr=learning_rate)

    # 4) Training loop with early stopping
    history          = {'epoch': [], 'train_loss': [], 'val_loss': []}
    best_val_loss    = float('inf')
    best_state_dict  = None
    epochs_no_improve = 0

    for epoch in range(1, max_epochs + 1):
        # ---- train ----
        model.train()
        train_losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimiser.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            optimiser.step()
            train_losses.append(loss.item() * xb.size(0))
        train_loss = np.sum(train_losses) / len(train_loader.dataset)

        # validate
        model.eval()
        with torch.no_grad():
            val_losses = []
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                pred = model(xb)
                val_losses.append(loss_fn(pred, yb).item() * xb.size(0))
            val_loss = np.sum(val_losses) / len(val_loader.dataset)

        history['epoch'].append(epoch)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)

        # early stopping
        if val_loss < best_val_loss - 1e-9:
            best_val_loss     = val_loss
            best_state_dict   = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break

    # Restore best weights
    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    # Validation predictions on transformed scale
    model.eval()
    with torch.no_grad():
        val_pred = model(X_val_t.to(device)).cpu().numpy()

    return {
        'model'           : model,
        'scaler'          : scaler,
        'history'         : pd.DataFrame(history),
        'val_pred'        : val_pred,         # (n_val_windows, HORIZON), transformed scale
        'y_val_actual'    : y_val,            # (n_val_windows, HORIZON), transformed scale
        'val_target_ts'   : ts_val,           # (n_val_windows, HORIZON), datetime64
        'best_val_loss'   : best_val_loss,
        'epochs_run'      : len(history['epoch']),
    }

1.6 Hyperparameter Tuning

We explore six configurations that each vary one factor relative to the baseline (Experiment 2) to make the effect of each hyperparameter visible:

| Experiment | Varied factor | lookback | hidden layers | learning rate | dropout | batch size |
|---|---|---|---|---|---|---|
| 1 | shorter lookback | 48  | [64, 32]  | 0.001  | 0.1 | 32 |
| 2 | **baseline**     | 96  | [64, 32]  | 0.001  | 0.1 | 32 |
| 3 | longer lookback  | 192 | [64, 32]  | 0.001  | 0.1 | 32 |
| 4 | wider network    | 96  | [128, 64] | 0.001  | 0.1 | 32 |
| 5 | smaller lr       | 96  | [64, 32]  | 0.0005 | 0.1 | 32 |
| 6 | higher dropout   | 96  | [64, 32]  | 0.001  | 0.2 | 32 |

For each experiment we compute the **validation RMSE on the original RRP scale** by inverse-transforming the predictions. Model selection uses this validation RMSE (not test data).

In [ ]:
experiment_grid = [
    {'name': 'Exp1 lookback48',  'lookback':  48, 'hidden_layers':[64, 32],  'learning_rate':0.001,  'dropout':0.1, 'batch_size':32},
    {'name': 'Exp2 baseline',    'lookback':  96, 'hidden_layers':[64, 32],  'learning_rate':0.001,  'dropout':0.1, 'batch_size':32},
    {'name': 'Exp3 lookback192', 'lookback': 192, 'hidden_layers':[64, 32],  'learning_rate':0.001,  'dropout':0.1, 'batch_size':32},
    {'name': 'Exp4 wider',       'lookback':  96, 'hidden_layers':[128, 64], 'learning_rate':0.001,  'dropout':0.1, 'batch_size':32},
    {'name': 'Exp5 smaller lr',  'lookback':  96, 'hidden_layers':[64, 32],  'learning_rate':0.0005, 'dropout':0.1, 'batch_size':32},
    {'name': 'Exp6 dropout 0.2', 'lookback':  96, 'hidden_layers':[64, 32],  'learning_rate':0.001,  'dropout':0.2, 'batch_size':32},
]

# Common training budget for every experiment
COMMON = {'max_epochs': 100, 'patience': 10}

tuning_records = []
tuning_results_models = {}   # keep models for the post-tuning inspection in C7

for cfg in experiment_grid:
    full_cfg = {**cfg, **COMMON}
    print(f"Training {cfg['name']} ...")
    result = train_mlp_model(full_cfg)

    # Inverse transform val predictions / actuals to original RRP for RMSE
    val_pred_rrp   = inverse_price_transform(result['val_pred'])
    val_actual_rrp = inverse_price_transform(result['y_val_actual'])
    val_rmse_rrp   = rmse(val_actual_rrp.flatten(), val_pred_rrp.flatten())

    tuning_records.append({
        'experiment'        : cfg['name'],
        'lookback'          : cfg['lookback'],
        'hidden_layers'     : str(cfg['hidden_layers']),
        'learning_rate'     : cfg['learning_rate'],
        'dropout'           : cfg['dropout'],
        'batch_size'        : cfg['batch_size'],
        'best_val_loss (transformed MSE)': result['best_val_loss'],
        'val_RMSE (RRP $/MWh)'           : val_rmse_rrp,
        'epochs_run'                     : result['epochs_run'],
    })
    tuning_results_models[cfg['name']] = result

tuning_results = (pd.DataFrame(tuning_records)
                    .sort_values('val_RMSE (RRP $/MWh)')
                    .reset_index(drop=True))
print("\nC6: Hyperparameter tuning results (sorted by validation RMSE)")
tuning_results.round(4)

1.7 Model Validation Evaluation

In [ ]:
# Select best experiment
best_row  = tuning_results.iloc[0]
best_name = best_row['experiment']
best      = tuning_results_models[best_name]
print(f"Best experiment: {best_name}")
print(f"Best validation RMSE on RRP scale ($/MWh): {best_row['val_RMSE (RRP $/MWh)']:.4f}")

# Multi-output predictions as a DataFrame with one row per validation window
val_pred_df = pd.DataFrame(
    best['val_pred'],
    columns=[f'horizon_{h+1}' for h in range(HORIZON)]
)
val_actual_df = pd.DataFrame(
    best['y_val_actual'],
    columns=[f'horizon_{h+1}' for h in range(HORIZON)]
)

# Inverse transform to original RRP
val_pred_rrp_df   = pd.DataFrame(
    inverse_price_transform(best['val_pred']),
    columns=val_pred_df.columns
)
val_actual_rrp_df = pd.DataFrame(
    inverse_price_transform(best['y_val_actual']),
    columns=val_actual_df.columns
)

# Overall validation RMSE across all (window, horizon) pairs
overall_rmse = rmse(val_actual_rrp_df.values.flatten(),
                    val_pred_rrp_df.values.flatten())
print(f"Overall validation RMSE (RRP $/MWh, all horizons flattened): {overall_rmse:.4f}")

# Optional spike / non-spike RMSE using Task B masks if available
spike_rmse = non_spike_rmse = None
if 'spike_mask_val' in globals() and 'non_spike_mask_val' in globals():
    # Flatten predictions, actuals, and corresponding timestamps
    flat_pred   = val_pred_rrp_df.values.flatten()
    flat_actual = val_actual_rrp_df.values.flatten()
    flat_ts     = pd.DatetimeIndex(best['val_target_ts'].flatten())

    # Look up each forecasted timestamp's mask value
    spike_flat     = spike_mask_val.reindex(flat_ts).fillna(False).values.astype(bool)
    non_spike_flat = non_spike_mask_val.reindex(flat_ts).fillna(False).values.astype(bool)

    if spike_flat.any():
        spike_rmse = rmse(flat_actual[spike_flat], flat_pred[spike_flat])
    if non_spike_flat.any():
        non_spike_rmse = rmse(flat_actual[non_spike_flat], flat_pred[non_spike_flat])

    print(f"Spike RMSE     (RRP $/MWh): "
          f"{spike_rmse:.4f}" if spike_rmse is not None else "Spike RMSE     : N/A")
    print(f"Non-spike RMSE (RRP $/MWh): "
          f"{non_spike_rmse:.4f}" if non_spike_rmse is not None else "Non-spike RMSE : N/A")
else:
    print("spike_mask_val / non_spike_mask_val not found — skipping spike/non-spike RMSE.")

# Summary table for the best model
best_summary = pd.DataFrame([{
    'best experiment'      : best_name,
    'lookback'             : best_row['lookback'],
    'hidden_layers'        : best_row['hidden_layers'],
    'learning_rate'        : best_row['learning_rate'],
    'dropout'              : best_row['dropout'],
    'batch_size'           : best_row['batch_size'],
    'Overall val RMSE'     : overall_rmse,
    'Spike val RMSE'       : spike_rmse,
    'Non-spike val RMSE'   : non_spike_rmse,
}])
display(best_summary.round(4))




# Training RMSE on original RRP scale (in-sample diagnostic only)
# Uses the SAME trained best model and its SAME training-only scaler.
# No re-training, no re-tuning, no test-set access. Training X / y are
# rebuilt for the best lookback by reusing build_train_val_windows.
best_model   = best['model']
best_scaler  = best['scaler']
best_lookback = int(best_row['lookback'])

# Rebuild only the training windows for the best lookback (validation windows
# from build_train_val_windows are ignored here; validation predictions are
# already in `best['val_pred']`).
X_train_raw_best, y_train_raw_best, _ts_train_best, _, _, _ = \
    build_train_val_windows(best_lookback, HORIZON)

# Scale with the training-only scaler that was fit during training
X_train_scaled_best = best_scaler.transform(X_train_raw_best).astype(np.float32)

# Predict on training windows with the best model
best_model.eval()
with torch.no_grad():
    train_pred_transformed = best_model(
        torch.from_numpy(X_train_scaled_best).to(device)
    ).cpu().numpy()

# Inverse-transform to original RRP scale and flatten across all 4 horizons
train_pred_rrp_arr   = inverse_price_transform(train_pred_transformed)
train_actual_rrp_arr = inverse_price_transform(y_train_raw_best)
train_overall_rmse   = rmse(train_actual_rrp_arr.flatten(),
                            train_pred_rrp_arr.flatten())




# Training spike / non-spike masks (reuse Task B training thresholds).
# The mask is defined on the training actual RRP value at each predicted
# (window, horizon) pair, so every prediction is classified by its own actual.
training_actual_rrp_flat = train_actual_rrp_arr.flatten()
training_pred_rrp_flat   = train_pred_rrp_arr.flatten()

training_spike_mask_flat = (
    (training_actual_rrp_flat < lower_spike_threshold) |
    (training_actual_rrp_flat > upper_spike_threshold)
)
training_non_spike_mask_flat = ~training_spike_mask_flat

train_spike_count     = int(training_spike_mask_flat.sum())
train_non_spike_count = int(training_non_spike_mask_flat.sum())

train_spike_rmse = (
    rmse(training_actual_rrp_flat[training_spike_mask_flat],
         training_pred_rrp_flat[training_spike_mask_flat])
    if train_spike_count > 0 else float('nan')
)
train_non_spike_rmse = (
    rmse(training_actual_rrp_flat[training_non_spike_mask_flat],
         training_pred_rrp_flat[training_non_spike_mask_flat])
    if train_non_spike_count > 0 else float('nan')
)




# Validation spike / non-spike counts and RMSE for Table 2.
# Reuses the per-(window, horizon) spike_flat / non_spike_flat masks already
# computed above (if available) or, as a fallback, defines them directly from
# the validation actuals against the same training thresholds (so the spike
# definition is consistent across training and validation).
val_actual_rrp_flat = val_actual_rrp_df.values.flatten()
val_pred_rrp_flat   = val_pred_rrp_df.values.flatten()

if 'spike_flat' in dir() and 'non_spike_flat' in dir():
    val_spike_mask_flat     = spike_flat
    val_non_spike_mask_flat = non_spike_flat
else:
    val_spike_mask_flat = (
        (val_actual_rrp_flat < lower_spike_threshold) |
        (val_actual_rrp_flat > upper_spike_threshold)
    )
    val_non_spike_mask_flat = ~val_spike_mask_flat

val_spike_count     = int(val_spike_mask_flat.sum())
val_non_spike_count = int(val_non_spike_mask_flat.sum())

# Reuse the existing validation Spike/Non-spike RMSE computed earlier in this
# cell. They may be None if the corresponding subset was empty; coerce to NaN.
val_spike_rmse_for_table     = spike_rmse     if spike_rmse     is not None else float('nan')
val_non_spike_rmse_for_table = non_spike_rmse if non_spike_rmse is not None else float('nan')




# Table 1: Training RMSE diagnostic for the best Price MLP
training_rmse_table_c7 = pd.DataFrame([{
    'Dataset'        : 'Training',
    'Overall RMSE'   : train_overall_rmse,
    'Spike RMSE'     : train_spike_rmse,
    'Non-spike RMSE' : train_non_spike_rmse,
    'Spike count'    : train_spike_count,
    'Non-spike count': train_non_spike_count,
}])
print("Table 1: Training RMSE diagnostic for the best Price MLP (original RRP, $/MWh)")
display(training_rmse_table_c7.round(4))



# Table 2: Validation RMSE evaluation for the best Price MLP
validation_rmse_table_c7 = pd.DataFrame([{
    'Dataset'        : 'Validation',
    'Overall RMSE'   : overall_rmse,
    'Spike RMSE'     : val_spike_rmse_for_table,
    'Non-spike RMSE' : val_non_spike_rmse_for_table,
    'Spike count'    : val_spike_count,
    'Non-spike count': val_non_spike_count,
}])
print("Table 2: Validation RMSE evaluation for the best Price MLP (original RRP, $/MWh)")
display(validation_rmse_table_c7.round(4))



# Residual diagnostic plots (validation only)
# These are additional diagnostics; they do not affect any RMSE numbers and
# do not use the test set.
val_resid_flat = val_actual_rrp_flat - val_pred_rrp_flat

# Plot 1: Residual vs Predicted (validation)
plt.figure(figsize=(10, 5))
plt.scatter(val_pred_rrp_flat, val_resid_flat, s=10, alpha=0.5)
plt.axhline(0, color='black', linewidth=0.8)
plt.title('Residual Plot for Best Price MLP (Validation)')
plt.xlabel('Predicted RRP ($/MWh)')
plt.ylabel('Residual = Actual - Predicted ($/MWh)')
plt.tight_layout()
plt.show()

# Plot 2: Residual over validation target time
val_resid_ts = pd.DatetimeIndex(best['val_target_ts'].flatten())
plt.figure(figsize=(14, 4))
plt.plot(val_resid_ts, val_resid_flat, color='gray', linewidth=0.8)
plt.axhline(0, color='black', linewidth=0.8)
plt.title('Validation Residuals over Time for Best Price MLP')
plt.xlabel('Validation target time')
plt.ylabel('Residual ($/MWh)')
plt.tight_layout()
plt.show()


# Plot training and validation loss curves for the best model
hist = best['history']
plt.figure(figsize=(10, 4))
plt.plot(hist['epoch'], hist['train_loss'], label='Training loss')
plt.plot(hist['epoch'], hist['val_loss'],   label='Validation loss')
plt.title('Training and Validation Loss for Best Price MLP')
plt.xlabel('Epoch')
plt.ylabel('MSE loss (transformed RRP scale)')
plt.legend()
plt.tight_layout()
plt.show()


# Validation forecast visualisation on a readable window: first 2-3 val days.
# We plot horizon-1 (one-step-ahead from each window) so each timestamp has a
# single visible forecast value; this is for visual inspection only and does
# not affect the RMSE numbers above, which use all four horizons.
PLOT_VAL_DAYS = 2  # readable window
ts_first      = pd.DatetimeIndex(best['val_target_ts'][:, 0])  # horizon-1 timestamp per window
pred_h1_rrp   = val_pred_rrp_df['horizon_1'].values
actual_h1_rrp = val_actual_rrp_df['horizon_1'].values

cutoff = yc_price_val.index[0] + pd.Timedelta(days=PLOT_VAL_DAYS)
mask   = ts_first < cutoff

plt.figure(figsize=(14, 5))
plt.plot(yc_price_val.index[:PLOT_VAL_DAYS * STEPS_PER_DAY],
         inverse_price_transform(yc_price_val.iloc[:PLOT_VAL_DAYS * STEPS_PER_DAY].values),
         color='black', linewidth=1, label='Validation actual RRP ($/MWh)')
plt.plot(ts_first[mask], pred_h1_rrp[mask],
         color='tab:purple', linewidth=1.2, marker='.',
         label='MLP horizon-1 forecast (RRP $/MWh)')
plt.title('MLP with Lagged Inputs — Validation Forecast (first 2 validation days, RRP scale)')
plt.xlabel('Time')
plt.ylabel('RRP ($/MWh)')
plt.legend()
plt.tight_layout()
plt.show()

lstm demend

Demand LSTM — v3 (targeted fix for residual structure)
   + add lag1, lag4, diff1
  + HuberLoss → MSELoss

  + dropout 0.3

In [ ]:

# ---- Config ----------------------------------------------------------------
RUN_MODE = 'default_multi_seed'
SEEDS    = [42, 7, 123]
EPOCHS   = 80
PATIENCE = 12
BATCH    = 64
LR       = 1e-3
WEIGHT_DECAY = 1e-4

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

M = 96
H = HORIZON

# ---- Features (9 channels) -------------------------------------------------
all_demand = pd.concat([yc_demand_train, yc_demand_val, yc_demand_test])

demand_mean = float(yc_demand_train.mean())
demand_std  = float(yc_demand_train.std())
demand_z    = ((all_demand.values - demand_mean) / demand_std).astype(np.float32)

slot     = all_demand.index.hour * 4 + all_demand.index.minute // 15
sin_slot = np.sin(2 * np.pi * slot.values / M).astype(np.float32)
cos_slot = np.cos(2 * np.pi * slot.values / M).astype(np.float32)
is_weekend = all_demand.index.dayofweek.isin([5, 6]).astype(np.float32)

def make_lag(arr, k):
    out = arr.copy()
    out[k:] = arr[:-k]
    out[:k] = arr[:k]
    return out.astype(np.float32)

lag1   = make_lag(demand_z, 1)
lag4   = make_lag(demand_z, 4)
lag96  = make_lag(demand_z, 96)
lag672 = make_lag(demand_z, 672)
diff1  = (demand_z - lag1).astype(np.float32)

features = np.column_stack([demand_z, sin_slot, cos_slot, is_weekend,
                            lag1, lag4, lag96, lag672, diff1]).astype(np.float32)
N_FEATURES = features.shape[1]
print(f'feature matrix shape: {features.shape}  (channels: {N_FEATURES})')


def make_windows(t_lo, t_hi, L):
    Xs, ys = [], []
    for t in range(t_lo, t_hi + 1):
        Xs.append(features[t - L + 1: t + 1])
        ys.append(demand_z[t + 1: t + H + 1])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)


class DemandLSTM(nn.Module):
    def __init__(self, input_size=N_FEATURES, hidden_size=64, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=dropout)
        self.head = nn.Linear(hidden_size, H)
        for name, p in self.lstm.named_parameters():
            if 'weight' in name:
                nn.init.xavier_uniform_(p)
            elif 'bias_ih' in name:
                p.data.fill_(0)
                n = p.size(0)
                p.data[n // 4: n // 2].fill_(1.0)
            elif 'bias_hh' in name:
                p.data.fill_(0)
        nn.init.xavier_uniform_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :])


def rmse_mw(pred_z, actual_z):
    pred_mw   = pred_z   * demand_std + demand_mean
    actual_mw = actual_z * demand_std + demand_mean
    return float(np.sqrt(((pred_mw - actual_mw) ** 2).mean(axis=1)).mean())


def _set_seed(s):
    os.environ['PYTHONHASHSEED'] = str(s)
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def _train_once(L, hidden, dropout, seed):
    _set_seed(seed)
    X_tr, y_tr = make_windows(L - 1, TRAIN_END - H - 1, L)
    X_va, y_va = make_windows(TRAIN_END, VAL_END - H - 1, L)

    X_tr_t = torch.from_numpy(X_tr); y_tr_t = torch.from_numpy(y_tr)
    X_va_t = torch.from_numpy(X_va).to(device)
    gen = torch.Generator().manual_seed(seed)
    loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=BATCH,
                        shuffle=True, generator=gen, drop_last=False)

    model = DemandLSTM(N_FEATURES, hidden, 2, dropout).to(device)
    opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=3)
    loss_fn = nn.MSELoss()

    X_tr_full = X_tr_t.to(device)
    best_rmse, best_state, bad, hist_v, hist_t = float('inf'), None, 0, [], []
    for ep in range(EPOCHS):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        model.eval()
        with torch.no_grad():
            pred_va = model(X_va_t).cpu().numpy()
            pred_tr = model(X_tr_full).cpu().numpy()
        v = rmse_mw(pred_va, y_va)
        t = rmse_mw(pred_tr, y_tr)
        hist_v.append(v); hist_t.append(t)
        sched.step(v)
        if v < best_rmse:
            best_rmse = v
            best_state = {k: vv.detach().cpu().clone() for k, vv in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= PATIENCE: break
    return best_rmse, best_state, (hist_t, hist_v)


def train_multi_seed(L, hidden, dropout, seeds=SEEDS):
    rs, states, hists = [], [], []
    for s in seeds:
        t0 = time.time()
        r, st, (ht, hv) = _train_once(L, hidden, dropout, s)
        rs.append(r); states.append(st); hists.append((ht, hv))
        gap = hv[int(np.argmin(hv))] - ht[int(np.argmin(hv))]
        print(f'    seed={s:>3}: val RMSE={r:7.2f} | train@best_val={ht[int(np.argmin(hv))]:7.2f} | gap={gap:+7.2f} | ep={len(hv):2d}  ({time.time()-t0:5.1f}s)')
    return float(np.mean(rs)), float(np.std(rs)), rs, states, hists


def test_eval(L, hidden, dropout, state):
    model = DemandLSTM(N_FEATURES, hidden, 2, dropout).to(device)
    model.load_state_dict(state)
    model.eval()
    X_te, y_te = make_windows(VAL_END, len(all_demand) - H - 1, L)
    X_te_t = torch.from_numpy(X_te).to(device)
    with torch.no_grad():
        pred_te = model(X_te_t).cpu().numpy()
    return rmse_mw(pred_te, y_te)


# ---- Run -------------------------------------------------------------------
DEFAULT = dict(L=192, hidden=64, dropout=0.3)
t_start = time.time()

print(f'\n=== v3 DEFAULT cfg × {len(SEEDS)} seeds  (L={DEFAULT["L"]}, hidden={DEFAULT["hidden"]}, dropout={DEFAULT["dropout"]}, wd={WEIGHT_DECAY}, MSE loss) ===')
mean_v, std_v, all_v, states, _ = train_multi_seed(**DEFAULT)
print(f'  val RMSE: {mean_v:.2f} ± {std_v:.2f} MW   (per seed: {[round(x,2) for x in all_v]})')

test_rmses = [test_eval(DEFAULT['L'], DEFAULT['hidden'], DEFAULT['dropout'], st) for st in states]
print(f'  test RMSE: {np.mean(test_rmses):.2f} ± {np.std(test_rmses):.2f} MW   (per seed: {[round(x,2) for x in test_rmses]})')

winner_cfg = DEFAULT
winner_state = states[int(np.argmin(all_v))]

print(f'\nTotal runtime: {time.time()-t_start:.1f}s')

In [ ]:
# ============================================================
# Use the winner model to predict on VAL and TEST, then plot
# ============================================================
L_w = winner_cfg['L']

# ---- Rebuild winner model ----
_lstm = DemandLSTM(N_FEATURES, winner_cfg['hidden'], 2, winner_cfg['dropout']).to(device)
_lstm.load_state_dict(winner_state)
_lstm.eval()

# ---- Helper: given a window range, return (h-step-ahead prediction Series, overall RMSE) ----
def predict_series(t_lo, t_hi, ref_series, L):
    """
    t_lo, t_hi: window range passed to make_windows (must match training)
    ref_series: the ground-truth series used to align the index (yc_demand_val / yc_demand_test)
    returns: (Series of the h=H step prediction aligned to the time axis, multi-step RMSE in MW)
    """
    X, y = make_windows(t_lo, t_hi, L)
    with torch.no_grad():
        pred = _lstm(torch.from_numpy(X).to(device)).cpu().numpy()

    # Overall multi-step RMSE (same as rmse_mw used in training)
    rmse = rmse_mw(pred, y)

    # Align the H-th step prediction back onto the time axis
    out = pd.Series(index=ref_series.index, dtype=float)
    # Window at t predicts t+1..t+H; we take the H-th step -> lands at t+H
    pos = (np.arange(t_lo, t_hi + 1) + H) - ref_series_index_start(ref_series)
    ok = (pos >= 0) & (pos < len(out))
    vals = (pred[:, H - 1] * demand_std + demand_mean)
    out.iloc[pos[ok]] = vals[ok]
    return out, rmse


def ref_series_index_start(ref_series):
    """Integer start position of ref_series within all_demand"""
    return all_demand.index.get_loc(ref_series.index[0])


# ---- VAL: window range matches the validation set used in training ----
val_pred, val_rmse = predict_series(TRAIN_END, VAL_END - H - 1, yc_demand_val, L_w)

# ---- TEST: window range matches the one in test_eval (VAL_END -> len(all_demand)-H-1) ----
test_pred, test_rmse = predict_series(VAL_END, len(all_demand) - H - 1, yc_demand_test, L_w)

print(f'VAL  RMSE = {val_rmse:.1f} MW')
print(f'TEST RMSE = {test_rmse:.1f} MW')

# ---- Plot ----
fig, axes = plt.subplots(2, 1, figsize=(13, 8))

axes[0].plot(yc_demand_val.index, yc_demand_val.values,
             color='black', linewidth=1.2, label='Actual')
axes[0].plot(val_pred.index, val_pred.values,
             color='green', linewidth=1.2, linestyle='--', label=f'LSTM h={H}')
axes[0].set_title(f'Validation  —  LSTM h={H}  (val RMSE = {val_rmse:.0f} MW)')
axes[0].set_ylabel('Demand (MW)'); axes[0].grid(alpha=0.3); axes[0].legend()

axes[1].plot(yc_demand_test.index, yc_demand_test.values,
             color='black', linewidth=1.2, label='Actual')
axes[1].plot(test_pred.index, test_pred.values,
             color='red', linewidth=1.2, linestyle='--', label=f'LSTM h={H}')
axes[1].set_title(f'Test  —  LSTM h={H}  (test RMSE = {test_rmse:.0f} MW)')
axes[1].set_xlabel('Time'); axes[1].set_ylabel('Demand (MW)')
axes[1].grid(alpha=0.3); axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# LSTM competition forecast: 2026-02-10 00:00-00:45

L_w = winner_cfg['L']
_lstm = DemandLSTM(N_FEATURES, winner_cfg['hidden'], 2, winner_cfg['dropout']).to(device)
_lstm.load_state_dict(winner_state)
_lstm.eval()

last_window = features[-L_w:].copy()
X_future    = torch.from_numpy(last_window[None, :, :]).to(device)
with torch.no_grad():
    lstm_future_z = _lstm(X_future).cpu().numpy().flatten()
lstm_future = lstm_future_z * demand_std + demand_mean

future_idx = pd.date_range(yc_demand_test.index[-1] + pd.Timedelta(minutes=15),
                            periods=H, freq='15min')
forecast_df = pd.DataFrame({
    'time': future_idx,
    'LSTM forecast (MW)': np.round(lstm_future, 2),
})
print(forecast_df)


**TASK C: MLP (Demand)**

In [ ]:
#fix random seed to ensure reproducibility

tf.random.set_seed(42)
np.random.seed(42)

In [ ]:
#transform the inputs into a bounded range for stability

demand_train_raw = yc_demand_train.values.astype('float32')
demand_val_raw   = yc_demand_val.values.astype('float32')
demand_test_raw  = yc_demand_test.values.astype('float32')

scaler_demand = MinMaxScaler(feature_range=(0, 1))
scaler_demand.fit(demand_train_raw.reshape(-1, 1))

demand_train_scaled = scaler_demand.transform(
    demand_train_raw.reshape(-1, 1)).flatten()
demand_val_scaled   = scaler_demand.transform(
    demand_val_raw.reshape(-1, 1)).flatten()
demand_test_scaled  = scaler_demand.transform(
    demand_test_raw.reshape(-1, 1)).flatten()

print(f'Train scaled size: {len(demand_train_scaled)}')
print(f'Validation scaled size: {len(demand_val_scaled)}')
print(f'Test scaled size: {len(demand_test_scaled)}')

In [ ]:
#define a funtion to build the lagged dataset

def make_lagged_dataset(series, time_window, horizon=4):
    """
    X[i] = [y[i-time_window], ..., y[i-1]]   (lagged inputs)
    Y[i] = [y[i], y[i+1], y[i+2], y[i+3]]   (4-step ahead targets)
    """
    X, Y = [], []
    for i in range(time_window, len(series) - horizon + 1):
        X.append(series[i - time_window:i])
        Y.append(series[i:i + horizon])
    return np.array(X), np.array(Y)

In [ ]:
#hyperparameter tuning using grid search

time_window_options   = [48, 96, 192]
hidden_units_options  = [32, 64, 128]
learning_rate_options = [0.001, 0.01]

combined_tv = np.concatenate([demand_train_scaled, demand_val_scaled])

tuning_results = []

print('\n--- Hyperparameter Tuning ---')
print(f'{"time_window":>12} {"hidden_units":>12} {"lr":>8} {"Val RMSE":>10}')
print('-' * 50)

for time_window, hidden_units, lr in itertools.product(
        time_window_options, hidden_units_options, learning_rate_options):

    X_train, Y_train = make_lagged_dataset(demand_train_scaled, time_window)
    X_all, Y_all = make_lagged_dataset(combined_tv, time_window)
    X_val = X_all[len(X_train):]
    Y_val = Y_all[len(Y_train):]

    tf.random.set_seed(42)
    model = Sequential([
        Input(shape=(time_window,)),
        Dense(hidden_units, activation='relu'),
        Dense(hidden_units // 2, activation='relu'),
        Dense(4)
    ])
    model.compile(loss='mean_squared_error', optimizer=Adam(learning_rate=lr))

    es = EarlyStopping(monitor='val_loss', patience=10,
                       restore_best_weights=True, verbose=0)

    model.fit(
        X_train, Y_train,
        epochs=100,
        batch_size=32,
        validation_data=(X_val, Y_val),
        callbacks=[es],
        verbose=0
    )

    Y_val_pred = scaler_demand.inverse_transform(model.predict(X_val, verbose=0))
    Y_val_true = scaler_demand.inverse_transform(Y_val)

    per_origin_mse = np.mean((Y_val_pred - Y_val_true) ** 2, axis=1)
    val_rmse = math.sqrt(np.mean(per_origin_mse))

    tuning_results.append({
        'time_window':  time_window,
        'hidden_units': hidden_units,
        'lr':           lr,
        'val_rmse':     val_rmse,
        'model':        model
    })

    print(f'{time_window:>12} {hidden_units:>12} {lr:>8.3f} {val_rmse:>10.2f}')

In [ ]:
#select the best configuration

tuning_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'model'}
                           for r in tuning_results])
tuning_df = tuning_df.sort_values('val_rmse').reset_index(drop=True)

print('\nTuning results (sorted by Val RMSE):')
print(tuning_df.to_string(index=False))

best_row         = tuning_df.iloc[0]
best_time_window = int(best_row['time_window'])
best_hidden      = int(best_row['hidden_units'])
best_lr          = float(best_row['lr'])

print(f'\nBest config: time_window={best_time_window}, '
      f'hidden_units={best_hidden}, lr={best_lr}')
print(f'Best Val RMSE: {best_row["val_rmse"]:.2f} MW')

In [ ]:
#refit the final model using the best hyperparameters

X_train_best, Y_train_best = make_lagged_dataset(
    demand_train_scaled, best_time_window)
X_all_tv, Y_all_tv = make_lagged_dataset(combined_tv, best_time_window)
X_val_best = X_all_tv[len(X_train_best):]
Y_val_best = Y_all_tv[len(Y_train_best):]

tf.random.set_seed(42)
final_model = Sequential([
    Input(shape=(best_time_window,)),
    Dense(best_hidden,      activation='relu'),
    Dense(best_hidden // 2, activation='relu'),
    Dense(4)
])
final_model.compile(
    loss='mean_squared_error',
    optimizer=Adam(learning_rate=best_lr)
)

es_final = EarlyStopping(monitor='val_loss', patience=15,
                          restore_best_weights=True, verbose=0)

history = final_model.fit(
    X_train_best, Y_train_best,
    epochs=200,
    batch_size=32,
    validation_data=(X_val_best, Y_val_best),
    callbacks=[es_final],
    verbose=1
)

print(f'\nTraining stopped at epoch {len(history.history["loss"])}')

In [ ]:
#plot the training loss curve
#the losses of both curve decrease and converge

plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'],     label='Training Loss',   color='tab:blue')
plt.plot(history.history['val_loss'], label='Validation Loss', color='tab:orange')
plt.title(f'MLP Training Loss  |  '
          f'time_window={best_time_window}, hidden={best_hidden}, lr={best_lr}')
plt.xlabel('Epoch')
plt.ylabel('MSE (scaled)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('output/figures/mlp_demand_training_loss.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
#forecast on the validation set and compute the RMSE

context_val = np.concatenate([
    demand_train_scaled[-best_time_window:],
    demand_val_scaled
])

X_val_full, Y_val_full = make_lagged_dataset(context_val, best_time_window)

Y_val_pred_full = scaler_demand.inverse_transform(
    final_model.predict(X_val_full, verbose=0))
Y_val_true_full = scaler_demand.inverse_transform(Y_val_full)

per_origin_mse_val = np.mean((Y_val_pred_full - Y_val_true_full) ** 2, axis=1)
val_rmse_final = math.sqrt(np.mean(per_origin_mse_val))
print(f'\nFinal Model — Validation RMSE: {val_rmse_final:.2f} MW')

In [ ]:
#plot the training set and validation set as well as the forecasts on the validation set for visual assessment

val_pred_h4       = Y_val_pred_full[:, -1]
val_timestamps_h4 = yc_demand_val.index[3: 3 + len(val_pred_h4)]

plt.figure(figsize=(15, 6))
plt.plot(yc_demand_train, label='Training Demand',
         linewidth=2, color='tab:blue')
plt.plot(yc_demand_val,   label='Validation Demand (Actual)',
         linewidth=2, color='tab:orange')
plt.plot(val_timestamps_h4, val_pred_h4, label='4-step Forecast (MLP)',
         linestyle='--', linewidth=1.5, color='tab:green')
plt.title(f'Training + Validation Demand with 4-Step-Ahead MLP Forecast  '
          f'|  Val RMSE: {val_rmse_final:.2f} MW')
plt.xlabel('Time')
plt.ylabel('Demand (MW)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('output/figures/mlp_demand_forecast_full.png',
            dpi=150, bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
#plot the valdiation set only with the foreasts on the valdiation set for clearer comparison

plt.figure(figsize=(15, 6))
plt.plot(yc_demand_val,                  label='Actual Demand',
         linewidth=2, color='tab:orange')
plt.plot(val_timestamps_h4, val_pred_h4, label='4-step Forecast (MLP)',
         linestyle='--', linewidth=1.5, color='tab:green')
plt.title(f'Validation Period: Actual vs MLP 4-Step-Ahead Forecast  '
          f'|  RMSE: {val_rmse_final:.2f} MW')
plt.xlabel('Time')
plt.ylabel('Demand (MW)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('output/figures/mlp_demand_forecast_val.png',
            dpi=150, bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
#evaluate on the test set

context_test = np.concatenate([
    demand_val_scaled[-best_time_window:],
    demand_test_scaled
])

X_test_full, Y_test_full = make_lagged_dataset(context_test, best_time_window)

Y_test_pred_full = scaler_demand.inverse_transform(
    final_model.predict(X_test_full, verbose=0))
Y_test_true_full = scaler_demand.inverse_transform(Y_test_full)

per_origin_mse_test = np.mean((Y_test_pred_full - Y_test_true_full) ** 2, axis=1)
test_rmse = math.sqrt(np.mean(per_origin_mse_test))

print(f'\nFinal Model Performance:')
print(f'  Validation RMSE: {val_rmse_final:.2f} MW')
print(f'  Test RMSE:       {test_rmse:.2f} MW')

In [ ]:
#plot the test set and the forecasts on the test set

test_pred_h4       = Y_test_pred_full[:, -1]
test_true_h4       = Y_test_true_full[:, -1]
test_timestamps_h4 = yc_demand_test.index[3: 3 + len(test_pred_h4)]

plt.figure(figsize=(15, 6))
plt.plot(yc_demand_test,
         label='Actual Demand (Test)', linewidth=2, color='tab:orange')
plt.plot(test_timestamps_h4, test_pred_h4,
         label='4-step Forecast (MLP)', linewidth=1.5,
         linestyle='--', color='tab:green')
plt.title(f'Test Set: Actual vs MLP 4-Step-Ahead Forecast  '
          f'|  Test RMSE: {test_rmse:.2f} MW')
plt.xlabel('Time')
plt.ylabel('Demand (MW)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('output/figures/mlp_demand_forecast_test.png',
            dpi=150, bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
#MLP performance summary table

summary = pd.DataFrame({
    'Split':     ['Validation', 'Test'],
    'Days':      [VAL_DAYS, TEST_DAYS],
    'Steps':     [len(yc_demand_val), len(yc_demand_test)],
    'RMSE (MW)': [round(val_rmse_final, 2), round(test_rmse, 2)],
})
print('\n── MLP Final Performance Summary ────────────────────────────────────')
print(summary.to_string(index=False))

In [ ]:
#4-step-ahead forecast using MLP

all_demand_scaled = scaler_demand.transform(
    np.concatenate([demand_train_raw,
                    demand_val_raw,
                    demand_test_raw]).reshape(-1, 1)
).flatten()

last_window  = all_demand_scaled[-best_time_window:].reshape(1, -1)
comp_pred_sc = final_model.predict(last_window, verbose=0)
comp_pred_mw = scaler_demand.inverse_transform(comp_pred_sc).flatten()

comp_timestamps = pd.date_range(
    start='2026-02-10 00:00:00',
    periods=4,
    freq='15min'
)

competition_df = pd.DataFrame({
    'SETTLEMENTDATE': comp_timestamps,
    'TOTALDEMAND':    np.round(comp_pred_mw, 2)
})

print('\nCompetition Forecast (Demand):')
print(competition_df.to_string(index=False))

RNN/LSTM/GRU for RRP

In [ ]:

# ============================================================
# RNN / LSTM / GRU for RRP forecasting using Keras
# ============================================================
#
# Assumed existing variables:
#   y_price_train       : original-scale RRP training series, pandas Series/DataFrame with DatetimeIndex
#   y_price_val         : original-scale RRP validation series, pandas Series/DataFrame with DatetimeIndex
#   yc_demand_train     : TOTALDEMAND training series, pandas Series/DataFrame with DatetimeIndex
#   yc_demand_val       : TOTALDEMAND validation series, pandas Series/DataFrame with DatetimeIndex
#
# Optional existing variables:
#   spike_mask_val      : Boolean Series indexed by validation target timestamps
#   non_spike_mask_val  : Boolean Series indexed by validation target timestamps
#
# Task:
#   Forecast RRP 4 steps ahead, i.e. 1 hour ahead for 15-minute data.
#
# Model inputs at each time step in the 96-step window:
#   1. past transformed RRP
#   2. past demand
#   3. sin_time_of_day
#   4. cos_time_of_day
#   5. day_of_week_sin
#   6. day_of_week_cos
#   7. is_weekend
#   8. spike_dummy
#
# Target:
#   transformed RRP at t+4, where transformed RRP = asinh(RRP / 100)
#
# Scaling:
#   All continuous inputs/target are scaled to [-1,1] using training data only.
#   sin/cos features are naturally in [-1,1], so no change is needed.
#
# Notes linked to LeCun et al. training advice:
#   - Focus on validation performance, not training loss.
#   - Use early stopping to reduce overfitting.
#   - Use mini-batches rather than full-batch training.
#   - Treat learning rate, model size, and regularisation as key hyperparameters.
# ============================================================


# ============================================================
# 0. Imports
# ============================================================

import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, SimpleRNN, LSTM, GRU, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


# ============================================================
# 1. Reproducibility
# ============================================================

SEED = 42

np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)


# ============================================================
# 2. Tunable hyperparameters
# ============================================================
#
# Ranked by likely importance for RMSE:
#
# 1. WINDOW_SIZE:
#      Main value = 96 because m = 96 daily seasonality.
#      Sensitivity values: 48, 96, 192.
#
# 2. HIDDEN_UNITS:
#      Main value = 32.
#      Try 16 if overfitting or 64 if underfitting.
#
# 3. LEARNING_RATE:
#      Main value = 0.001.
#      Try 0.0005 or 0.0001 if validation loss is unstable.
#
# 4. DROPOUT_RATE:
#      Main value = 0.1.
#      Try 0.0 if underfitting; 0.2 if overfitting.
#
# 5. BATCH_SIZE:
#      Main value = 32.
#      Try 16 for noisier/smaller updates; 64 for faster but smoother updates.
#
# 6. DENSE_UNITS:
#      Main value = 16.
#      Try 0 by removing dense hidden layer if model overfits.
#
# 7. EPOCHS / EARLY_STOPPING_PATIENCE:
#      Use a high max epoch with early stopping.
#      Early stopping decides when to stop based on validation loss.
#
# Recommended first run:
#   WINDOW_SIZE = 96
#   HIDDEN_UNITS = 32
#   LEARNING_RATE = 0.001
#   DROPOUT_RATE = 0.1
#   BATCH_SIZE = 32
# ============================================================

FORECAST_HORIZON = 4
WINDOW_SIZE = 96

ASINH_SCALE = 100
IQR_MULTIPLIER = 1.5

HIDDEN_UNITS = 32
DENSE_UNITS = 16
DROPOUT_RATE = 0.1

LEARNING_RATE = 0.001
BATCH_SIZE = 32
EPOCHS = 200

EARLY_STOPPING_PATIENCE = 15
REDUCE_LR_PATIENCE = 7
REDUCE_LR_FACTOR = 0.5
MIN_LEARNING_RATE = 1e-5

VALIDATION_VERBOSE = 1


# ============================================================
# 3. Basic helper functions
# ============================================================

def to_1d_series(x, name):
    """
    Convert pandas Series/DataFrame to clean 1D float Series.
    """
    if isinstance(x, pd.DataFrame):
        x = x.squeeze()

    x = pd.to_numeric(
        x.sort_index(),
        errors="coerce"
    ).astype(float).dropna()

    x.name = name
    return x


def inverse_asinh(y_asinh):
    """
    Convert transformed RRP back to original RRP scale.
    """
    return ASINH_SCALE * np.sinh(y_asinh)


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


# ============================================================
# 4. Prepare train/validation series
# ============================================================

y_price_train = to_1d_series(y_price_train, "RRP")
y_price_val = to_1d_series(y_price_val, "RRP")

yc_demand_train = to_1d_series(yc_demand_train, "TOTALDEMAND")
yc_demand_val = to_1d_series(yc_demand_val, "TOTALDEMAND")

train_end_time = y_price_train.index.max()
val_start_time = y_price_val.index.min()
val_end_time = y_price_val.index.max()

# Combine train + validation for rolling-origin style validation.
# Validation samples may use lagged / windowed values from earlier validation timestamps,
# because those observations would be known at the forecast origin.
rrp_full_raw = pd.concat([y_price_train, y_price_val]).sort_index()
demand_full_raw = pd.concat([yc_demand_train, yc_demand_val]).sort_index()

# Align demand to RRP timestamp index.
# If both are already identical, this does nothing.
demand_full_raw = demand_full_raw.reindex(rrp_full_raw.index).interpolate().ffill().bfill()

# Transform RRP.
rrp_full_asinh = np.arcsinh(rrp_full_raw / ASINH_SCALE)
rrp_train_asinh = np.arcsinh(y_price_train / ASINH_SCALE)


# ============================================================
# 5. Spike dummy
# ============================================================

# Spike threshold estimated from training data only.
Q1_price = y_price_train.quantile(0.25)
Q3_price = y_price_train.quantile(0.75)
IQR_price = Q3_price - Q1_price

lower_spike_threshold = Q1_price - IQR_MULTIPLIER * IQR_price
upper_spike_threshold = Q3_price + IQR_MULTIPLIER * IQR_price

spike_dummy_full = (
    (rrp_full_raw < lower_spike_threshold) |
    (rrp_full_raw > upper_spike_threshold)
).astype(int)

# Target spike mask for validation RMSE split.
if "spike_mask_val" not in globals() or "non_spike_mask_val" not in globals():
    spike_mask_val = (
        (y_price_val < lower_spike_threshold) |
        (y_price_val > upper_spike_threshold)
    )
    non_spike_mask_val = ~spike_mask_val
else:
    if isinstance(spike_mask_val, pd.DataFrame):
        spike_mask_val = spike_mask_val.squeeze()
    if isinstance(non_spike_mask_val, pd.DataFrame):
        non_spike_mask_val = non_spike_mask_val.squeeze()

    spike_mask_val = spike_mask_val.reindex(y_price_val.index).fillna(False).astype(bool)
    non_spike_mask_val = non_spike_mask_val.reindex(y_price_val.index).fillna(False).astype(bool)

print("Spike threshold lower:", round(lower_spike_threshold, 3))
print("Spike threshold upper:", round(upper_spike_threshold, 3))
print("Validation spike count:", int(spike_mask_val.sum()))
print("Validation non-spike count:", int(non_spike_mask_val.sum()))


# ============================================================
# 6. Fit scalers on training data only
# ============================================================

rrp_scaler = MinMaxScaler(feature_range=(-1, 1))
demand_scaler = MinMaxScaler(feature_range=(-1, 1))

# Fit RRP scaler on transformed training RRP.
rrp_scaler.fit(
    rrp_train_asinh.values.reshape(-1, 1)
)

# Fit demand scaler on training demand only.
demand_scaler.fit(
    yc_demand_train.values.reshape(-1, 1)
)


# ============================================================
# 7. Build feature DataFrame
# ============================================================

df = pd.DataFrame(index=rrp_full_raw.index)

# Continuous features scaled to [-1,1].
df["rrp_asinh_scaled"] = rrp_scaler.transform(
    rrp_full_asinh.values.reshape(-1, 1)
)

df["demand_scaled"] = demand_scaler.transform(
    demand_full_raw.values.reshape(-1, 1)
)

# Calendar features based on current timestamp in the input window.
timestamp = df.index

# Time of day as fraction in [0,1).
minutes_in_day = timestamp.hour * 60 + timestamp.minute
time_of_day_fraction = minutes_in_day / (24 * 60)

sin_time = np.sin(2 * np.pi * time_of_day_fraction)
cos_time = np.cos(2 * np.pi * time_of_day_fraction)

df["sin_time_of_day"] = sin_time
df["cos_time_of_day"] = cos_time

# Day of week cycle. Monday=0, Sunday=6.
day_of_week = timestamp.dayofweek
day_fraction = day_of_week / 7

dow_sin = np.sin(2 * np.pi * day_fraction)
dow_cos = np.cos(2 * np.pi * day_fraction)

df["day_of_week_sin"] = dow_sin
df["day_of_week_cos"] = dow_cos

df["is_weekend"] = pd.Series(day_of_week, index=df.index).isin([5, 6]).astype(int)

# Historical spike dummy in the input window.
# This is known at each input time and is not the future target spike status.
df["spike_dummy"] = spike_dummy_full.astype(int)

# Target at t+FORECAST_HORIZON.
df["target_rrp_asinh"] = rrp_full_asinh.shift(-FORECAST_HORIZON)
df["target_rrp_raw"] = rrp_full_raw.shift(-FORECAST_HORIZON)

target_timestamp = pd.Series(df.index, index=df.index).shift(-FORECAST_HORIZON)
df["target_timestamp"] = target_timestamp

# Scaled target.
# Use the same RRP scaler fitted on transformed training RRP.
df["target_rrp_scaled"] = rrp_scaler.transform(
    df["target_rrp_asinh"].values.reshape(-1, 1)
)

# Remove rows with missing target caused by shift.
df = df.dropna()


FEATURE_COLS = [
    "rrp_asinh_scaled",
    "demand_scaled",
    "sin_time_of_day",
    "cos_time_of_day",
    "day_of_week_sin",
    "day_of_week_cos",
    "is_weekend",
    "spike_dummy"
]

print("Feature columns:")
print(FEATURE_COLS)


# ============================================================
# 8. Create supervised sequence windows
# ============================================================

def create_sequence_data(df, feature_cols, window_size):
    """
    Create sequence windows.

    Each sample:
      X = features from [t-window_size+1, ..., t]
      y = target RRP at t + FORECAST_HORIZON

    The sample is assigned to train/validation according to target_timestamp,
    not according to the last input timestamp.
    """
    X_list = []
    y_scaled_list = []
    y_raw_list = []
    target_time_list = []

    feature_values = df[feature_cols].values.astype(float)
    target_scaled_values = df["target_rrp_scaled"].values.astype(float)
    target_raw_values = df["target_rrp_raw"].values.astype(float)
    target_times = pd.to_datetime(df["target_timestamp"])

    for end_pos in range(window_size - 1, len(df)):
        start_pos = end_pos - window_size + 1

        X_window = feature_values[start_pos:end_pos + 1]
        y_scaled = target_scaled_values[end_pos]
        y_raw = target_raw_values[end_pos]
        target_time = target_times.iloc[end_pos]

        X_list.append(X_window)
        y_scaled_list.append(y_scaled)
        y_raw_list.append(y_raw)
        target_time_list.append(target_time)

    X = np.asarray(X_list, dtype=float)
    y_scaled = np.asarray(y_scaled_list, dtype=float)
    y_raw = np.asarray(y_raw_list, dtype=float)
    target_times = pd.DatetimeIndex(target_time_list)

    return X, y_scaled, y_raw, target_times


X_all, y_all_scaled, y_all_raw, target_times_all = create_sequence_data(
    df=df,
    feature_cols=FEATURE_COLS,
    window_size=WINDOW_SIZE
)

train_sample_mask = target_times_all <= train_end_time

val_sample_mask = (
    (target_times_all >= val_start_time) &
    (target_times_all <= val_end_time)
)

X_train = X_all[train_sample_mask]
y_train_scaled = y_all_scaled[train_sample_mask]
y_train_raw = y_all_raw[train_sample_mask]
train_target_times = target_times_all[train_sample_mask]

X_val = X_all[val_sample_mask]
y_val_scaled = y_all_scaled[val_sample_mask]
y_val_raw = y_all_raw[val_sample_mask]
val_target_times = target_times_all[val_sample_mask]

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("Number of input features:", X_train.shape[2])


# ============================================================
# 9. Build Keras models
# ============================================================

def build_recurrent_model(model_type):
    """
    Build one of:
      - SimpleRNN
      - LSTM
      - GRU

    Model type controls the recurrent layer only.
    """
    model_type = model_type.upper()

    if model_type == "RNN":
        recurrent_layer = SimpleRNN(
            HIDDEN_UNITS,
            return_sequences=False
        )
    elif model_type == "LSTM":
        recurrent_layer = LSTM(
            HIDDEN_UNITS,
            return_sequences=False
        )
    elif model_type == "GRU":
        recurrent_layer = GRU(
            HIDDEN_UNITS,
            return_sequences=False
        )
    else:
        raise ValueError("model_type must be 'RNN', 'LSTM', or 'GRU'.")

    model = Sequential()
    model.add(Input(shape=(WINDOW_SIZE, len(FEATURE_COLS))))
    model.add(recurrent_layer)

    if DROPOUT_RATE > 0:
        model.add(Dropout(DROPOUT_RATE))

    if DENSE_UNITS is not None and DENSE_UNITS > 0:
        model.add(Dense(DENSE_UNITS, activation="relu"))

    model.add(Dense(1))

    optimizer = Adam(learning_rate=LEARNING_RATE)

    model.compile(
        optimizer=optimizer,
        loss="mse"
    )

    return model


callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=EARLY_STOPPING_PATIENCE,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=REDUCE_LR_FACTOR,
        patience=REDUCE_LR_PATIENCE,
        min_lr=MIN_LEARNING_RATE
    )
]


# ============================================================
# 10. Train and evaluate RNN / LSTM / GRU
# ============================================================

def inverse_scale_and_asinh(y_scaled):
    """
    Convert scaled transformed RRP back to original RRP scale.
    """
    y_scaled = np.asarray(y_scaled).reshape(-1, 1)
    y_asinh = rrp_scaler.inverse_transform(y_scaled).ravel()
    y_raw = inverse_asinh(y_asinh)
    return y_raw


def evaluate_predictions(y_true_raw, y_pred_raw, target_times):
    """
    Calculate overall validation RMSE and spike/non-spike RMSE.
    """
    overall_rmse = rmse(y_true_raw, y_pred_raw)

    # Align target spike mask to prediction target timestamps.
    aligned_spike_mask = spike_mask_val.reindex(target_times).fillna(False).astype(bool).values
    aligned_non_spike_mask = ~aligned_spike_mask

    if aligned_spike_mask.sum() > 0:
        spike_rmse = rmse(
            y_true_raw[aligned_spike_mask],
            y_pred_raw[aligned_spike_mask]
        )
    else:
        spike_rmse = np.nan

    if aligned_non_spike_mask.sum() > 0:
        non_spike_rmse = rmse(
            y_true_raw[aligned_non_spike_mask],
            y_pred_raw[aligned_non_spike_mask]
        )
    else:
        non_spike_rmse = np.nan

    return overall_rmse, spike_rmse, non_spike_rmse


def train_evaluate_model(model_type):
    """
    Train and evaluate a recurrent model.
    """
    print("\n" + "=" * 90)
    print(f"Training {model_type}")
    print("=" * 90)

    model = build_recurrent_model(model_type)

    history = model.fit(
        X_train,
        y_train_scaled,
        validation_data=(X_val, y_val_scaled),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        shuffle=False,
        verbose=VALIDATION_VERBOSE
    )

    # Predictions are on scaled transformed-RRP scale.
    train_pred_scaled = model.predict(X_train, batch_size=BATCH_SIZE).ravel()
    val_pred_scaled = model.predict(X_val, batch_size=BATCH_SIZE).ravel()

    # Convert back to original RRP scale.
    train_pred_raw = inverse_scale_and_asinh(train_pred_scaled)
    val_pred_raw = inverse_scale_and_asinh(val_pred_scaled)

    train_rmse = rmse(y_train_raw, train_pred_raw)
    val_rmse, spike_rmse, non_spike_rmse = evaluate_predictions(
        y_true_raw=y_val_raw,
        y_pred_raw=val_pred_raw,
        target_times=val_target_times
    )

    metrics = {
        "Model": model_type,
        "Window Size": WINDOW_SIZE,
        "Hidden Units": HIDDEN_UNITS,
        "Dense Units": DENSE_UNITS,
        "Dropout": DROPOUT_RATE,
        "Learning Rate": LEARNING_RATE,
        "Batch Size": BATCH_SIZE,
        "Epochs Run": len(history.history["loss"]),
        "Best Val Loss": min(history.history["val_loss"]),
        "Training RMSE": train_rmse,
        "Validation RMSE": val_rmse,
        "Spike RMSE": spike_rmse,
        "Non-spike RMSE": non_spike_rmse
    }

    train_results = pd.DataFrame({
        "actual_rrp_raw": y_train_raw,
        "predicted_rrp_raw": train_pred_raw,
        "error_raw": y_train_raw - train_pred_raw
    }, index=train_target_times)

    val_results = pd.DataFrame({
        "actual_rrp_raw": y_val_raw,
        "predicted_rrp_raw": val_pred_raw,
        "error_raw": y_val_raw - val_pred_raw
    }, index=val_target_times)

    return model, history, metrics, train_results, val_results


model_types = ["RNN", "LSTM", "GRU"]

trained_models = {}
histories = {}
metrics_list = []
train_results_dict = {}
val_results_dict = {}

for model_type in model_types:
    model, history, metrics, train_results, val_results = train_evaluate_model(model_type)

    trained_models[model_type] = model
    histories[model_type] = history
    metrics_list.append(metrics)
    train_results_dict[model_type] = train_results
    val_results_dict[model_type] = val_results


# ============================================================
# 11. Compare model performance
# ============================================================

nn_comparison_table = pd.DataFrame(metrics_list)
nn_comparison_table = nn_comparison_table.sort_values("Validation RMSE").reset_index(drop=True)

print("\n" + "=" * 100)
print("RNN / LSTM / GRU PERFORMANCE COMPARISON")
print("=" * 100)

display(nn_comparison_table.round(4))

best_model_type = nn_comparison_table.loc[0, "Model"]

print("Best model by validation RMSE:", best_model_type)


# ============================================================
# 12. Plot training curves
# ============================================================

for model_type in model_types:
    history = histories[model_type]

    plt.figure(figsize=(10, 4))
    plt.plot(history.history["loss"], label="Training loss")
    plt.plot(history.history["val_loss"], label="Validation loss")
    plt.title(f"{model_type}: Training and validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("MSE loss on scaled transformed RRP")
    plt.legend()
    plt.tight_layout()
    plt.show()


# ============================================================
# 13. Plot validation actual vs predicted RRP
# ============================================================

plt.figure(figsize=(16, 6))

plt.plot(
    y_price_val.index,
    y_price_val,
    label="Validation actual RRP",
    linewidth=1.5
)

for model_type in model_types:
    plt.plot(
        val_results_dict[model_type].index,
        val_results_dict[model_type]["predicted_rrp_raw"],
        label=f"{model_type} prediction",
        linestyle="--",
        alpha=0.8
    )

plt.title("RNN / LSTM / GRU Validation Forecasts: Actual vs Predicted RRP")
plt.xlabel("Time")
plt.ylabel("RRP ($/MWh)")
plt.legend()
plt.tight_layout()
plt.show()


# Optional symlog plot for readability under price spikes.
plt.figure(figsize=(16, 6))

plt.plot(
    y_price_val.index,
    y_price_val,
    label="Validation actual RRP",
    linewidth=1.5
)

for model_type in model_types:
    plt.plot(
        val_results_dict[model_type].index,
        val_results_dict[model_type]["predicted_rrp_raw"],
        label=f"{model_type} prediction",
        linestyle="--",
        alpha=0.8
    )

plt.yscale("symlog", linthresh=100)
plt.title("RNN / LSTM / GRU Validation Forecasts: Actual vs Predicted RRP, Symlog Scale")
plt.xlabel("Time")
plt.ylabel("RRP ($/MWh), symlog scale")
plt.legend()
plt.tight_layout()
plt.show()


# ============================================================
# 14. Residual/error plot for best model
# ============================================================

best_val_results = val_results_dict[best_model_type]

plt.figure(figsize=(14, 4))
plt.plot(best_val_results.index, best_val_results["error_raw"])
plt.axhline(0, linewidth=1)
plt.title(f"{best_model_type}: Validation forecast errors on original RRP scale")
plt.xlabel("Time")
plt.ylabel("Forecast error ($/MWh)")
plt.tight_layout()
plt.show()


plt.figure(figsize=(10, 5))
plt.scatter(
    best_val_results["predicted_rrp_raw"],
    best_val_results["error_raw"],
    alpha=0.5
)
plt.axhline(0, linewidth=1)
plt.title(f"{best_model_type}: Predicted RRP vs forecast error")
plt.xlabel("Predicted RRP ($/MWh)")
plt.ylabel("Forecast error ($/MWh)")
plt.tight_layout()
plt.show()


# ============================================================
# 15. Suggested report wording
# ============================================================
#
# The recurrent neural-network models used a 96-step input window corresponding
# to the daily seasonal period identified in the RRP series. Inputs included
# transformed RRP, demand, cyclical calendar features, weekend status, and a
# spike indicator. The RRP target was transformed using asinh(RRP/100), and
# continuous variables were scaled to [0,1] using training data only.
#
# Following the neural-network training principles discussed by LeCun et al.,
# model selection focused on validation performance rather than training loss.
# Early stopping was used to reduce overfitting, mini-batch stochastic
# optimisation was used for training, and learning rate/model size/dropout were
# treated as important hyperparameters.
#
# Final forecasts were inverse-scaled and inverse-transformed back to the
# original RRP scale before calculating RMSE, including separate spike and
# non-spike validation RMSE.
# ============================================================


In [ ]:

# ============================================================
# RNN / GRU hyperparameter tuning for RRP forecasting
# Validation selection + final test evaluation
# ============================================================
#
# Required existing variables:
#   y_price_train, y_price_val, y_price_test        : original-scale RRP
#   yc_demand_train, yc_demand_val, yc_demand_test  : TOTALDEMAND
#
# Optional:
#   yc_price_test : transformed RRP test series. This code does not require it;
#                   it consistently uses asinh(y_price_test / 100).
#
# Target:
#   asinh(RRP / 100)
#
# Inputs per time step:
#   transformed RRP, demand, sin/cos time of day, sin/cos day of week,
#   is_weekend, spike_dummy
#
# Scaling:
#   continuous variables scaled to [-1,1]; sine/cosine already [-1,1];
#   dummy variables remain 0/1.
#
# Workflow:
#   1. Tune RNN and GRU on validation RMSE.
#   2. Select the lowest validation RMSE model.
#   3. Refit selected model on train+validation.
#   4. Evaluate once on test.
# ============================================================

import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, SimpleRNN, GRU, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras import backend as K


# ============================================================
# 1. Reproducibility and core parameters
# ============================================================

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

FORECAST_HORIZON = 4
WINDOW_SIZE = 96
ASINH_SCALE = 100
IQR_MULTIPLIER = 1.5

EPOCHS = 200
EARLY_STOPPING_PATIENCE = 15
REDUCE_LR_PATIENCE = 7
REDUCE_LR_FACTOR = 0.5
MIN_LEARNING_RATE = 1e-5

VERBOSE = 0
DENSE_UNITS = 16

# Epochs are not tuned directly because early stopping selects the effective
# training duration using validation loss.


# ============================================================
# 2. Hyperparameter grids
# ============================================================

RNN_GRID = {
    "model_type": ["RNN"],
    "hidden_units": [16, 32, 64],
    "dropout_rate": [0.0, 0.1, 0.2],
    "learning_rate": [0.001, 0.0005],
    "batch_size": [16, 32],
}

GRU_GRID = {
    "model_type": ["GRU"],
    "hidden_units": [16, 32],
    "dropout_rate": [0.1],
    "learning_rate": [0.001, 0.0005],
    "batch_size": [32],
}


# ============================================================
# 3. Helper functions
# ============================================================

def to_1d_series(x, name):
    if isinstance(x, pd.DataFrame):
        x = x.squeeze()
    x = pd.to_numeric(x.sort_index(), errors="coerce").astype(float).dropna()
    x.name = name
    return x

def inverse_asinh(y_asinh):
    return ASINH_SCALE * np.sinh(y_asinh)

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def expand_grid(grid_dict):
    import itertools
    keys = list(grid_dict.keys())
    values = [grid_dict[k] for k in keys]
    return [dict(zip(keys, combo)) for combo in itertools.product(*values)]

def inverse_scale_and_asinh(y_scaled, rrp_scaler):
    y_scaled = np.asarray(y_scaled).reshape(-1, 1)
    y_asinh = rrp_scaler.inverse_transform(y_scaled).ravel()
    return inverse_asinh(y_asinh)


# ============================================================
# 4. Prepare input series
# ============================================================

y_price_train = to_1d_series(y_price_train, "RRP")
y_price_val = to_1d_series(y_price_val, "RRP")
y_price_test = to_1d_series(y_price_test, "RRP")

yc_demand_train = to_1d_series(yc_demand_train, "TOTALDEMAND")
yc_demand_val = to_1d_series(yc_demand_val, "TOTALDEMAND")
yc_demand_test = to_1d_series(yc_demand_test, "TOTALDEMAND")

train_end_time = y_price_train.index.max()
val_start_time = y_price_val.index.min()
val_end_time = y_price_val.index.max()
test_start_time = y_price_test.index.min()
test_end_time = y_price_test.index.max()

if "yc_price_test" in globals():
    print("yc_price_test exists, but this code uses asinh(y_price_test / 100) for consistency.")


# ============================================================
# 5. Feature construction
# ============================================================

FEATURE_COLS = [
    "rrp_asinh_scaled",
    "demand_scaled",
    "sin_time_of_day",
    "cos_time_of_day",
    "day_of_week_sin",
    "day_of_week_cos",
    "is_weekend",
    "spike_dummy"
]

def build_feature_dataframe(
    rrp_raw_full,
    demand_raw_full,
    scaler_fit_rrp_raw,
    scaler_fit_demand_raw,
    spike_fit_rrp_raw
):
    rrp_raw_full = rrp_raw_full.sort_index().astype(float)
    demand_raw_full = demand_raw_full.sort_index().astype(float)

    demand_raw_full = demand_raw_full.reindex(rrp_raw_full.index).interpolate().ffill().bfill()

    rrp_full_asinh = np.arcsinh(rrp_raw_full / ASINH_SCALE)
    scaler_fit_rrp_asinh = np.arcsinh(scaler_fit_rrp_raw / ASINH_SCALE)

    rrp_scaler = MinMaxScaler(feature_range=(-1, 1))
    demand_scaler = MinMaxScaler(feature_range=(-1, 1))

    rrp_scaler.fit(scaler_fit_rrp_asinh.values.reshape(-1, 1))
    demand_scaler.fit(scaler_fit_demand_raw.values.reshape(-1, 1))

    q1 = spike_fit_rrp_raw.quantile(0.25)
    q3 = spike_fit_rrp_raw.quantile(0.75)
    iqr = q3 - q1
    lower_spike_threshold = q1 - IQR_MULTIPLIER * iqr
    upper_spike_threshold = q3 + IQR_MULTIPLIER * iqr

    spike_dummy_full = (
        (rrp_raw_full < lower_spike_threshold) |
        (rrp_raw_full > upper_spike_threshold)
    ).astype(int)

    df = pd.DataFrame(index=rrp_raw_full.index)

    df["rrp_asinh_scaled"] = rrp_scaler.transform(rrp_full_asinh.values.reshape(-1, 1))
    df["demand_scaled"] = demand_scaler.transform(demand_raw_full.values.reshape(-1, 1))

    timestamp = df.index

    minutes_in_day = timestamp.hour * 60 + timestamp.minute
    time_of_day_fraction = minutes_in_day / (24 * 60)

    df["sin_time_of_day"] = np.sin(2 * np.pi * time_of_day_fraction)
    df["cos_time_of_day"] = np.cos(2 * np.pi * time_of_day_fraction)

    day_of_week = timestamp.dayofweek
    day_fraction = day_of_week / 7

    df["day_of_week_sin"] = np.sin(2 * np.pi * day_fraction)
    df["day_of_week_cos"] = np.cos(2 * np.pi * day_fraction)

    df["is_weekend"] = pd.Series(day_of_week, index=df.index).isin([5, 6]).astype(int)
    df["spike_dummy"] = spike_dummy_full.astype(int)

    df["target_rrp_asinh"] = rrp_full_asinh.shift(-FORECAST_HORIZON)
    df["target_rrp_raw"] = rrp_raw_full.shift(-FORECAST_HORIZON)
    df["target_timestamp"] = pd.Series(df.index, index=df.index).shift(-FORECAST_HORIZON)
    df["target_rrp_scaled"] = rrp_scaler.transform(df["target_rrp_asinh"].values.reshape(-1, 1))

    df = df.dropna()

    thresholds = {
        "lower": lower_spike_threshold,
        "upper": upper_spike_threshold
    }

    return df, rrp_scaler, demand_scaler, thresholds


def create_sequence_data(df, feature_cols, window_size):
    X_list, y_scaled_list, y_raw_list, target_time_list = [], [], [], []

    feature_values = df[feature_cols].values.astype(float)
    target_scaled_values = df["target_rrp_scaled"].values.astype(float)
    target_raw_values = df["target_rrp_raw"].values.astype(float)
    target_times = pd.to_datetime(df["target_timestamp"])

    for end_pos in range(window_size - 1, len(df)):
        start_pos = end_pos - window_size + 1

        X_list.append(feature_values[start_pos:end_pos + 1])
        y_scaled_list.append(target_scaled_values[end_pos])
        y_raw_list.append(target_raw_values[end_pos])
        target_time_list.append(target_times.iloc[end_pos])

    return (
        np.asarray(X_list, dtype=float),
        np.asarray(y_scaled_list, dtype=float),
        np.asarray(y_raw_list, dtype=float),
        pd.DatetimeIndex(target_time_list)
    )


def make_spike_mask_for_targets(target_raw_values, fit_rrp_raw):
    q1 = fit_rrp_raw.quantile(0.25)
    q3 = fit_rrp_raw.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - IQR_MULTIPLIER * iqr
    upper = q3 + IQR_MULTIPLIER * iqr

    target_raw_values = np.asarray(target_raw_values, dtype=float)
    return (target_raw_values < lower) | (target_raw_values > upper)


# ============================================================
# 6. Keras model construction
# ============================================================

def build_recurrent_model(model_type, hidden_units, dropout_rate, learning_rate):
    K.clear_session()
    tf.random.set_seed(SEED)

    model = Sequential()
    model.add(Input(shape=(WINDOW_SIZE, len(FEATURE_COLS))))

    if model_type.upper() == "RNN":
        model.add(SimpleRNN(hidden_units, return_sequences=False))
    elif model_type.upper() == "GRU":
        model.add(GRU(hidden_units, return_sequences=False))
    else:
        raise ValueError("model_type must be 'RNN' or 'GRU'.")

    if dropout_rate > 0:
        model.add(Dropout(dropout_rate))

    if DENSE_UNITS is not None and DENSE_UNITS > 0:
        model.add(Dense(DENSE_UNITS, activation="relu"))

    model.add(Dense(1))

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="mse"
    )

    return model


def make_callbacks():
    return [
        EarlyStopping(
            monitor="val_loss",
            patience=EARLY_STOPPING_PATIENCE,
            restore_best_weights=True
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=REDUCE_LR_FACTOR,
            patience=REDUCE_LR_PATIENCE,
            min_lr=MIN_LEARNING_RATE
        )
    ]


# ============================================================
# 7. Build validation tuning data
# ============================================================

rrp_train_val_raw_for_tuning = pd.concat([y_price_train, y_price_val]).sort_index()
demand_train_val_raw_for_tuning = pd.concat([yc_demand_train, yc_demand_val]).sort_index()

df_tune, rrp_scaler_tune, demand_scaler_tune, tune_thresholds = build_feature_dataframe(
    rrp_raw_full=rrp_train_val_raw_for_tuning,
    demand_raw_full=demand_train_val_raw_for_tuning,
    scaler_fit_rrp_raw=y_price_train,
    scaler_fit_demand_raw=yc_demand_train,
    spike_fit_rrp_raw=y_price_train
)

X_all_tune, y_all_scaled_tune, y_all_raw_tune, target_times_all_tune = create_sequence_data(
    df=df_tune,
    feature_cols=FEATURE_COLS,
    window_size=WINDOW_SIZE
)

train_sample_mask = target_times_all_tune <= train_end_time
val_sample_mask = (
    (target_times_all_tune >= val_start_time) &
    (target_times_all_tune <= val_end_time)
)

X_train = X_all_tune[train_sample_mask]
y_train_scaled = y_all_scaled_tune[train_sample_mask]
y_train_raw = y_all_raw_tune[train_sample_mask]

X_val = X_all_tune[val_sample_mask]
y_val_scaled = y_all_scaled_tune[val_sample_mask]
y_val_raw = y_all_raw_tune[val_sample_mask]
val_target_times = target_times_all_tune[val_sample_mask]

val_spike_mask = make_spike_mask_for_targets(
    target_raw_values=y_val_raw,
    fit_rrp_raw=y_price_train
)

print("Feature columns:", FEATURE_COLS)
print("Tuning X_train shape:", X_train.shape)
print("Tuning X_val shape:", X_val.shape)
print("Validation spike count:", int(val_spike_mask.sum()))
print("Validation non-spike count:", int((~val_spike_mask).sum()))


# ============================================================
# 8. Training and evaluation helpers
# ============================================================

def evaluate_raw_predictions(y_true_raw, y_pred_raw, spike_mask):
    overall = rmse(y_true_raw, y_pred_raw)

    if spike_mask.sum() > 0:
        spike = rmse(y_true_raw[spike_mask], y_pred_raw[spike_mask])
    else:
        spike = np.nan

    non_spike_mask = ~spike_mask

    if non_spike_mask.sum() > 0:
        non_spike = rmse(y_true_raw[non_spike_mask], y_pred_raw[non_spike_mask])
    else:
        non_spike = np.nan

    return overall, spike, non_spike


def train_one_combo(params):
    model = build_recurrent_model(
        model_type=params["model_type"],
        hidden_units=params["hidden_units"],
        dropout_rate=params["dropout_rate"],
        learning_rate=params["learning_rate"]
    )

    history = model.fit(
        X_train,
        y_train_scaled,
        validation_data=(X_val, y_val_scaled),
        epochs=EPOCHS,
        batch_size=params["batch_size"],
        callbacks=make_callbacks(),
        shuffle=False,
        verbose=VERBOSE
    )

    train_pred_scaled = model.predict(X_train, batch_size=params["batch_size"], verbose=0).ravel()
    val_pred_scaled = model.predict(X_val, batch_size=params["batch_size"], verbose=0).ravel()

    train_pred_raw = inverse_scale_and_asinh(train_pred_scaled, rrp_scaler_tune)
    val_pred_raw = inverse_scale_and_asinh(val_pred_scaled, rrp_scaler_tune)

    train_rmse = rmse(y_train_raw, train_pred_raw)

    val_rmse, val_spike_rmse, val_non_spike_rmse = evaluate_raw_predictions(
        y_true_raw=y_val_raw,
        y_pred_raw=val_pred_raw,
        spike_mask=val_spike_mask
    )

    metrics = {
        "Model": params["model_type"],
        "Hidden Units": params["hidden_units"],
        "Dropout": params["dropout_rate"],
        "Learning Rate": params["learning_rate"],
        "Batch Size": params["batch_size"],
        "Window Size": WINDOW_SIZE,
        "Dense Units": DENSE_UNITS,
        "Epochs Run": len(history.history["loss"]),
        "Best Val Loss": min(history.history["val_loss"]),
        "Training RMSE": train_rmse,
        "Validation RMSE": val_rmse,
        "Validation Spike RMSE": val_spike_rmse,
        "Validation Non-spike RMSE": val_non_spike_rmse
    }

    return model, history, metrics, val_pred_raw


# ============================================================
# 9. Run RNN and GRU hyperparameter tuning
# ============================================================

tuning_combos = expand_grid(RNN_GRID) + expand_grid(GRU_GRID)
print("Number of tuning combinations:", len(tuning_combos))

tuning_metrics = []

best_validation_rmse = np.inf
best_model = None
best_history = None
best_params = None
best_val_pred_raw = None

for i, params in enumerate(tuning_combos, start=1):
    print("\n" + "=" * 100)
    print(f"Running combination {i}/{len(tuning_combos)}")
    print(params)
    print("=" * 100)

    model, history, metrics, val_pred_raw = train_one_combo(params)
    tuning_metrics.append(metrics)

    if metrics["Validation RMSE"] < best_validation_rmse:
        best_validation_rmse = metrics["Validation RMSE"]
        best_model = model
        best_history = history
        best_params = params.copy()
        best_val_pred_raw = val_pred_raw.copy()

    print("Validation RMSE:", round(metrics["Validation RMSE"], 4))
    print("Validation Spike RMSE:", round(metrics["Validation Spike RMSE"], 4))
    print("Validation Non-spike RMSE:", round(metrics["Validation Non-spike RMSE"], 4))


tuning_results_table = pd.DataFrame(tuning_metrics)
tuning_results_table = tuning_results_table.sort_values("Validation RMSE").reset_index(drop=True)

print("\n" + "=" * 120)
print("RNN / GRU HYPERPARAMETER TUNING RESULTS")
print("=" * 120)

display(tuning_results_table.round(4))

print("\nBest hyperparameters by validation RMSE:")
print(best_params)


# ============================================================
# 10. Plot best validation prediction vs actual
# ============================================================

best_val_results = pd.DataFrame({
    "actual_rrp_raw": y_val_raw,
    "predicted_rrp_raw": best_val_pred_raw,
    "error_raw": y_val_raw - best_val_pred_raw
}, index=val_target_times)

plt.figure(figsize=(16, 6))
plt.plot(best_val_results.index, best_val_results["actual_rrp_raw"], label="Validation actual RRP")
plt.plot(best_val_results.index, best_val_results["predicted_rrp_raw"], label="Best NN validation prediction", linestyle="--")
plt.title("Best RNN/GRU model: Validation actual vs predicted RRP")
plt.xlabel("Time")
plt.ylabel("RRP ($/MWh)")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(16, 6))
plt.plot(best_val_results.index, best_val_results["actual_rrp_raw"], label="Validation actual RRP")
plt.plot(best_val_results.index, best_val_results["predicted_rrp_raw"], label="Best NN validation prediction", linestyle="--")
plt.yscale("symlog", linthresh=100)
plt.title("Best RNN/GRU model: Validation actual vs predicted RRP, symlog scale")
plt.xlabel("Time")
plt.ylabel("RRP ($/MWh), symlog scale")
plt.legend()
plt.tight_layout()
plt.show()


# ============================================================
# 11. Refit best model on train + validation, then evaluate test
# ============================================================

y_price_trainval = pd.concat([y_price_train, y_price_val]).sort_index()
yc_demand_trainval = pd.concat([yc_demand_train, yc_demand_val]).sort_index()

rrp_full_for_test = pd.concat([y_price_train, y_price_val, y_price_test]).sort_index()
demand_full_for_test = pd.concat([yc_demand_train, yc_demand_val, yc_demand_test]).sort_index()

df_test_full, rrp_scaler_final, demand_scaler_final, final_thresholds = build_feature_dataframe(
    rrp_raw_full=rrp_full_for_test,
    demand_raw_full=demand_full_for_test,
    scaler_fit_rrp_raw=y_price_trainval,
    scaler_fit_demand_raw=yc_demand_trainval,
    spike_fit_rrp_raw=y_price_trainval
)

X_all_final, y_all_scaled_final, y_all_raw_final, target_times_all_final = create_sequence_data(
    df=df_test_full,
    feature_cols=FEATURE_COLS,
    window_size=WINDOW_SIZE
)

trainval_sample_mask = target_times_all_final <= val_end_time
test_sample_mask = (
    (target_times_all_final >= test_start_time) &
    (target_times_all_final <= test_end_time)
)

X_trainval = X_all_final[trainval_sample_mask]
y_trainval_scaled = y_all_scaled_final[trainval_sample_mask]
y_trainval_raw = y_all_raw_final[trainval_sample_mask]

X_test = X_all_final[test_sample_mask]
y_test_scaled = y_all_scaled_final[test_sample_mask]
y_test_raw = y_all_raw_final[test_sample_mask]
test_target_times = target_times_all_final[test_sample_mask]

test_spike_mask = make_spike_mask_for_targets(
    target_raw_values=y_test_raw,
    fit_rrp_raw=y_price_trainval
)

print("\nFinal train+validation shape:", X_trainval.shape)
print("Test shape:", X_test.shape)
print("Test spike count:", int(test_spike_mask.sum()))
print("Test non-spike count:", int((~test_spike_mask).sum()))

final_model = build_recurrent_model(
    model_type=best_params["model_type"],
    hidden_units=best_params["hidden_units"],
    dropout_rate=best_params["dropout_rate"],
    learning_rate=best_params["learning_rate"]
)

final_history = final_model.fit(
    X_trainval,
    y_trainval_scaled,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=best_params["batch_size"],
    callbacks=make_callbacks(),
    shuffle=False,
    verbose=VERBOSE
)

test_pred_scaled = final_model.predict(
    X_test,
    batch_size=best_params["batch_size"],
    verbose=0
).ravel()

test_pred_raw = inverse_scale_and_asinh(test_pred_scaled, rrp_scaler_final)

test_rmse, test_spike_rmse, test_non_spike_rmse = evaluate_raw_predictions(
    y_true_raw=y_test_raw,
    y_pred_raw=test_pred_raw,
    spike_mask=test_spike_mask
)

test_summary = {
    "Selected Model": best_params["model_type"],
    "Hidden Units": best_params["hidden_units"],
    "Dropout": best_params["dropout_rate"],
    "Learning Rate": best_params["learning_rate"],
    "Batch Size": best_params["batch_size"],
    "Window Size": WINDOW_SIZE,
    "Final Epochs Run": len(final_history.history["loss"]),
    "Test RMSE": test_rmse,
    "Test Spike RMSE": test_spike_rmse,
    "Test Non-spike RMSE": test_non_spike_rmse
}

test_summary_table = pd.DataFrame([test_summary])

print("\n" + "=" * 100)
print("FINAL TEST PERFORMANCE OF SELECTED RNN/GRU MODEL")
print("=" * 100)

display(test_summary_table.round(4))

test_results = pd.DataFrame({
    "actual_rrp_raw": y_test_raw,
    "predicted_rrp_raw": test_pred_raw,
    "error_raw": y_test_raw - test_pred_raw,
    "is_spike": test_spike_mask
}, index=test_target_times)


# ============================================================
# 12. Plot final test actual vs predicted
# ============================================================

plt.figure(figsize=(16, 6))
plt.plot(test_results.index, test_results["actual_rrp_raw"], label="Test actual RRP")
plt.plot(test_results.index, test_results["predicted_rrp_raw"], label="Selected NN test prediction", linestyle="--")
plt.title("Selected RNN/GRU model: Test actual vs predicted RRP")
plt.xlabel("Time")
plt.ylabel("RRP ($/MWh)")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(16, 6))
plt.plot(test_results.index, test_results["actual_rrp_raw"], label="Test actual RRP")
plt.plot(test_results.index, test_results["predicted_rrp_raw"], label="Selected NN test prediction", linestyle="--")
plt.yscale("symlog", linthresh=100)
plt.title("Selected RNN/GRU model: Test actual vs predicted RRP, symlog scale")
plt.xlabel("Time")
plt.ylabel("RRP ($/MWh), symlog scale")
plt.legend()
plt.tight_layout()
plt.show()


# ============================================================
# 13. Optional: save tables to CSV
# ============================================================

tuning_results_table.to_csv("nn_rrp_tuning_results.csv", index=False)
test_summary_table.to_csv("nn_rrp_test_summary.csv", index=False)
best_val_results.to_csv("nn_rrp_best_validation_predictions.csv")
test_results.to_csv("nn_rrp_test_predictions.csv")

print("Saved output CSV files:")
print("- nn_rrp_tuning_results.csv")
print("- nn_rrp_test_summary.csv")
print("- nn_rrp_best_validation_predictions.csv")
print("- nn_rrp_test_predictions.csv")


# ============================================================
# Suggested report wording
# ============================================================
#
# "RNN and GRU models were tuned using validation RMSE. The RNN received a
# larger tuning grid because it performed best in the baseline architecture
# comparison, while GRU was retained as a sensitivity comparison. The number
# of epochs was not tuned directly because early stopping selected the effective
# training duration based on validation loss. After the best hyperparameter
# combination was selected, the model was refitted using the combined training
# and validation data, and final performance was evaluated once on the test set."
# ============================================================


Competition RRP

In [ ]:

# ============================================================
# RRP RNN: Forecast next four timestamps after the test period
# ============================================================
#
# Required existing variables:
#   y_price_train, y_price_val, y_price_test
#       Original-scale RRP series with DatetimeIndex.
#
#   yc_demand_train, yc_demand_val, yc_demand_test
#       TOTALDEMAND series with DatetimeIndex.
#
# Purpose:
#   Fit the selected RNN model using all available observed data
#   (train + validation + test), then forecast the next four RRP
#   timestamps after the test period:
#
#       2026-02-10 00:00:00
#       2026-02-10 00:15:00
#       2026-02-10 00:30:00
#       2026-02-10 00:45:00
#
# Important modelling note:
#   The selected RNN was trained as a single-output 4-step-ahead model.
#   Therefore, the four competition timestamps are generated from the
#   four most recent eligible forecast origins:
#
#       origin T-3 -> forecast T+1
#       origin T-2 -> forecast T+2
#       origin T-1 -> forecast T+3
#       origin T   -> forecast T+4
#
#   This matches the 4-step-ahead training design used earlier.
#
# Selected RNN hyperparameters:
#   Hidden units  = 32
#   Dropout       = 0.0
#   Learning rate = 0.0005
#   Batch size    = 16
#   Window size   = 96
#   Dense units   = 16
# ============================================================

import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, SimpleRNN, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras import backend as K


# ============================================================
# 1. Fixed selected hyperparameters
# ============================================================

SEED = 42

FORECAST_HORIZON = 4
WINDOW_SIZE = 96

ASINH_SCALE = 100
IQR_MULTIPLIER = 1.5

HIDDEN_UNITS = 32
DROPOUT_RATE = 0.0
LEARNING_RATE = 0.0005
BATCH_SIZE = 16
DENSE_UNITS = 16

EPOCHS = 200
EARLY_STOPPING_PATIENCE = 15

REDUCE_LR_PATIENCE = 7
REDUCE_LR_FACTOR = 0.5
MIN_LEARNING_RATE = 1e-5

VERBOSE = 1
# Change to 0 if you do not want epoch-level output.

EXPECTED_FORECAST_INDEX = pd.DatetimeIndex([
    "2026-02-10 00:00:00",
    "2026-02-10 00:15:00",
    "2026-02-10 00:30:00",
    "2026-02-10 00:45:00"
])

OUTPUT_RRP_CSV = "rrp_rnn_next4_forecasts.csv"

OUTPUT_MODEL_FILE = "best_rrp.keras"


# ============================================================
# 2. Reproducibility
# ============================================================

np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)


# ============================================================
# 3. Helper functions
# ============================================================

def to_1d_series(x, name):
    """
    Convert a pandas Series/DataFrame to clean 1D float Series.
    """
    if isinstance(x, pd.DataFrame):
        x = x.squeeze()

    x = pd.to_numeric(
        x.sort_index(),
        errors="coerce"
    ).astype(float).dropna()

    x.name = name
    return x


def inverse_asinh(y_asinh):
    """
    Convert transformed RRP back to original RRP scale.
    """
    return ASINH_SCALE * np.sinh(y_asinh)


def inverse_scale_and_asinh(y_scaled, rrp_scaler):
    """
    Convert scaled transformed RRP prediction back to original RRP scale.
    """
    y_scaled = np.asarray(y_scaled).reshape(-1, 1)
    y_asinh = rrp_scaler.inverse_transform(y_scaled).ravel()
    y_raw = inverse_asinh(y_asinh)
    return y_raw


# ============================================================
# 4. Prepare full observed data
# ============================================================

y_price_train = to_1d_series(y_price_train, "RRP")
y_price_val = to_1d_series(y_price_val, "RRP")
y_price_test = to_1d_series(y_price_test, "RRP")

yc_demand_train = to_1d_series(yc_demand_train, "TOTALDEMAND")
yc_demand_val = to_1d_series(yc_demand_val, "TOTALDEMAND")
yc_demand_test = to_1d_series(yc_demand_test, "TOTALDEMAND")

# Use all available observed data for the final competition model.
rrp_full_raw = pd.concat(
    [y_price_train, y_price_val, y_price_test]
).sort_index()

demand_full_raw = pd.concat(
    [yc_demand_train, yc_demand_val, yc_demand_test]
).sort_index()

# Align demand to the RRP timestamp index.
demand_full_raw = (
    demand_full_raw
    .reindex(rrp_full_raw.index)
    .interpolate()
    .ffill()
    .bfill()
)

last_observed_time = rrp_full_raw.index.max()

forecast_index = pd.date_range(
    start=last_observed_time + pd.Timedelta(minutes=15),
    periods=4,
    freq="15min"
)

print("Last observed timestamp:", last_observed_time)
print("Forecast timestamps:")
print(forecast_index)

if not forecast_index.equals(EXPECTED_FORECAST_INDEX):
    print("\nWARNING:")
    print("Forecast timestamps do not match the expected competition timestamps.")
    print("Expected:")
    print(EXPECTED_FORECAST_INDEX)
    print("Actual:")
    print(forecast_index)


# ============================================================
# 5. Fit scalers and build feature dataframe
# ============================================================

FEATURE_COLS = [
    "rrp_asinh_scaled",
    "demand_scaled",
    "sin_time_of_day",
    "cos_time_of_day",
    "day_of_week_sin",
    "day_of_week_cos",
    "is_weekend",
    "spike_dummy"
]

# Transform RRP.
rrp_full_asinh = np.arcsinh(rrp_full_raw / ASINH_SCALE)

# Fit scalers using all observed data.
rrp_scaler = MinMaxScaler(feature_range=(-1, 1))
demand_scaler = MinMaxScaler(feature_range=(-1, 1))

rrp_scaler.fit(rrp_full_asinh.values.reshape(-1, 1))
demand_scaler.fit(demand_full_raw.values.reshape(-1, 1))

# Spike threshold estimated from all observed RRP.
q1 = rrp_full_raw.quantile(0.25)
q3 = rrp_full_raw.quantile(0.75)
iqr = q3 - q1

lower_spike_threshold = q1 - IQR_MULTIPLIER * iqr
upper_spike_threshold = q3 + IQR_MULTIPLIER * iqr

spike_dummy_full = (
    (rrp_full_raw < lower_spike_threshold) |
    (rrp_full_raw > upper_spike_threshold)
).astype(int)

print("\nSpike thresholds based on all observed data:")
print("Lower:", round(lower_spike_threshold, 3))
print("Upper:", round(upper_spike_threshold, 3))

# Feature dataframe over observed timestamps only.
df_features = pd.DataFrame(index=rrp_full_raw.index)

df_features["rrp_asinh_scaled"] = rrp_scaler.transform(
    rrp_full_asinh.values.reshape(-1, 1)
)

df_features["demand_scaled"] = demand_scaler.transform(
    demand_full_raw.values.reshape(-1, 1)
)

timestamp = df_features.index

# Time-of-day cyclical features.
minutes_in_day = timestamp.hour * 60 + timestamp.minute
time_of_day_fraction = minutes_in_day / (24 * 60)

df_features["sin_time_of_day"] = np.sin(2 * np.pi * time_of_day_fraction)
df_features["cos_time_of_day"] = np.cos(2 * np.pi * time_of_day_fraction)

# Day-of-week cyclical features.
day_of_week = timestamp.dayofweek
day_fraction = day_of_week / 7

df_features["day_of_week_sin"] = np.sin(2 * np.pi * day_fraction)
df_features["day_of_week_cos"] = np.cos(2 * np.pi * day_fraction)

df_features["is_weekend"] = pd.Series(day_of_week, index=df_features.index).isin([5, 6]).astype(int)
df_features["spike_dummy"] = spike_dummy_full.astype(int)


# ============================================================
# 6. Create training sequences using all observed data
# ============================================================

df_train = df_features.copy()

# Target = transformed RRP at t+4.
df_train["target_rrp_asinh"] = rrp_full_asinh.shift(-FORECAST_HORIZON)
df_train["target_rrp_scaled"] = rrp_scaler.transform(
    df_train["target_rrp_asinh"].values.reshape(-1, 1)
)

df_train = df_train.dropna()

X_train_list = []
y_train_list = []

feature_values = df_train[FEATURE_COLS].values.astype(float)
target_values = df_train["target_rrp_scaled"].values.astype(float)

for end_pos in range(WINDOW_SIZE - 1, len(df_train)):
    start_pos = end_pos - WINDOW_SIZE + 1

    X_train_list.append(feature_values[start_pos:end_pos + 1])
    y_train_list.append(target_values[end_pos])

X_train_final = np.asarray(X_train_list, dtype=float)
y_train_final = np.asarray(y_train_list, dtype=float)

print("\nFinal training data shape:")
print("X_train_final:", X_train_final.shape)
print("y_train_final:", y_train_final.shape)


# ============================================================
# 7. Build and train selected RNN model
# ============================================================

def build_selected_rnn():
    K.clear_session()
    tf.random.set_seed(SEED)

    model = Sequential()
    model.add(Input(shape=(WINDOW_SIZE, len(FEATURE_COLS))))

    model.add(
        SimpleRNN(
            HIDDEN_UNITS,
            return_sequences=False
        )
    )

    if DROPOUT_RATE > 0:
        model.add(Dropout(DROPOUT_RATE))

    if DENSE_UNITS is not None and DENSE_UNITS > 0:
        model.add(Dense(DENSE_UNITS, activation="relu"))

    model.add(Dense(1))

    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss="mse"
    )

    return model


callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=EARLY_STOPPING_PATIENCE,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=REDUCE_LR_FACTOR,
        patience=REDUCE_LR_PATIENCE,
        min_lr=MIN_LEARNING_RATE
    )
]

selected_rnn_model = build_selected_rnn()

history = selected_rnn_model.fit(
    X_train_final,
    y_train_final,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    shuffle=False,
    verbose=VERBOSE
)

print("\nTraining complete.")
print("Epochs run:", len(history.history["loss"]))
print("Best validation loss:", round(min(history.history["val_loss"]), 6))


# ============================================================
# 8. Create the four prediction windows
# ============================================================

# Because the model is trained for target t+4, the next four timestamps
# use the four most recent eligible origins:
#   end_pos = N-4, N-3, N-2, N-1
# where N is the number of observed timestamps.

feature_values_all = df_features[FEATURE_COLS].values.astype(float)
N = len(df_features)

prediction_end_positions = [
    N - FORECAST_HORIZON + i
    for i in range(FORECAST_HORIZON)
]

X_pred_list = []

for end_pos in prediction_end_positions:
    start_pos = end_pos - WINDOW_SIZE + 1

    if start_pos < 0:
        raise ValueError("Not enough observations to create prediction window.")

    X_window = feature_values_all[start_pos:end_pos + 1]
    X_pred_list.append(X_window)

X_pred = np.asarray(X_pred_list, dtype=float)

print("\nPrediction input shape:", X_pred.shape)


# ============================================================
# 9. Forecast next four RRP values
# ============================================================

forecast_scaled = selected_rnn_model.predict(
    X_pred,
    batch_size=BATCH_SIZE,
    verbose=0
).ravel()

forecast_rrp = inverse_scale_and_asinh(
    y_scaled=forecast_scaled,
    rrp_scaler=rrp_scaler
)

rrp_forecast_df = pd.DataFrame({
    "SETTLEMENTDATE": forecast_index,
    "RRP": forecast_rrp
})

# Competition forecasts should be reported to at least two decimals.
rrp_forecast_df["RRP"] = rrp_forecast_df["RRP"].round(3)

print("\nNext-four RRP forecasts:")
display(rrp_forecast_df)

rrp_forecast_df.to_csv(
    OUTPUT_RRP_CSV,
    index=False
)

selected_rnn_model.save(OUTPUT_MODEL_FILE)

print("\nSaved files:")
print(OUTPUT_RRP_CSV)
print(OUTPUT_MODEL_FILE)


# ============================================================
# 10. Optional: if you already have demand forecasts, merge manually
# ============================================================
#
# The final competition CSV must contain exactly:
#   SETTLEMENTDATE, TOTALDEMAND, RRP
#
# This file only produces the RRP column because demand is generated by the
# selected MLP model.
#
# After obtaining demand forecasts, merge them with rrp_forecast_df using
# SETTLEMENTDATE, then save:
#
#   next_hour_forecasts.csv
#
# ============================================================


Competition demand

In [ ]:
# =============================================================================
# MLP DEMAND — FULL CODE
# =============================================================================

import math
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Fix seeds
tf.random.set_seed(42)
np.random.seed(42)

# ── Scale demand ───────────────────────────────────────────────────────────────
demand_train_raw = yc_demand_train.values.astype('float32')
demand_val_raw   = yc_demand_val.values.astype('float32')
demand_test_raw  = yc_demand_test.values.astype('float32')

scaler_demand = MinMaxScaler(feature_range=(0, 1))
scaler_demand.fit(demand_train_raw.reshape(-1, 1))

demand_train_scaled = scaler_demand.transform(
    demand_train_raw.reshape(-1, 1)).flatten()
demand_val_scaled   = scaler_demand.transform(
    demand_val_raw.reshape(-1, 1)).flatten()
demand_test_scaled  = scaler_demand.transform(
    demand_test_raw.reshape(-1, 1)).flatten()

print(f'Train scaled size:      {len(demand_train_scaled)}')
print(f'Validation scaled size: {len(demand_val_scaled)}')
print(f'Test scaled size:       {len(demand_test_scaled)}')

# ── Lagged dataset builder ─────────────────────────────────────────────────────
def make_lagged_dataset(series, time_window, horizon=4):
    X, Y = [], []
    for i in range(time_window, len(series) - horizon + 1):
        X.append(series[i - time_window:i])
        Y.append(series[i:i + horizon])
    return np.array(X), np.array(Y)

# ── Hyperparameter tuning ──────────────────────────────────────────────────────
time_window_options   = [48, 96, 192]
hidden_units_options  = [32, 64, 128]
learning_rate_options = [0.001, 0.01]

combined_tv = np.concatenate([demand_train_scaled, demand_val_scaled])

tuning_results = []

print('\n--- Hyperparameter Tuning ---')
print(f'{"time_window":>12} {"hidden_units":>12} {"lr":>8} {"Val RMSE":>10}')
print('-' * 50)

for time_window, hidden_units, lr in itertools.product(
        time_window_options, hidden_units_options, learning_rate_options):

    X_train, Y_train = make_lagged_dataset(demand_train_scaled, time_window)
    X_all,   Y_all   = make_lagged_dataset(combined_tv, time_window)
    X_val = X_all[len(X_train):]
    Y_val = Y_all[len(Y_train):]

    tf.random.set_seed(42)
    model = Sequential([
        Input(shape=(time_window,)),
        Dense(hidden_units,      activation='relu'),
        Dense(hidden_units // 2, activation='relu'),
        Dense(4)
    ])
    model.compile(loss='mean_squared_error', optimizer=Adam(learning_rate=lr))

    es = EarlyStopping(monitor='val_loss', patience=10,
                       restore_best_weights=True, verbose=0)

    model.fit(
        X_train, Y_train,
        epochs=100,
        batch_size=32,
        validation_data=(X_val, Y_val),
        callbacks=[es],
        verbose=0
    )

    Y_val_pred = scaler_demand.inverse_transform(model.predict(X_val, verbose=0))
    Y_val_true = scaler_demand.inverse_transform(Y_val)

    per_origin_mse = np.mean((Y_val_pred - Y_val_true) ** 2, axis=1)
    val_rmse = math.sqrt(np.mean(per_origin_mse))

    tuning_results.append({
        'time_window':  time_window,
        'hidden_units': hidden_units,
        'lr':           lr,
        'val_rmse':     val_rmse,
        'model':        model
    })

    print(f'{time_window:>12} {hidden_units:>12} {lr:>8.3f} {val_rmse:>10.2f}')

# ── Select best configuration ──────────────────────────────────────────────────
tuning_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'model'}
                           for r in tuning_results])
tuning_df = tuning_df.sort_values('val_rmse').reset_index(drop=True)

print('\nTuning results (sorted by Val RMSE):')
print(tuning_df.to_string(index=False))

best_row         = tuning_df.iloc[0]
best_time_window = int(best_row['time_window'])
best_hidden      = int(best_row['hidden_units'])
best_lr          = float(best_row['lr'])

print(f'\nBest config: time_window={best_time_window}, '
      f'hidden_units={best_hidden}, lr={best_lr}')
print(f'Best Val RMSE: {best_row["val_rmse"]:.2f} MW')

# ── Refit on training set only (for val/test evaluation) ──────────────────────
X_train_best, Y_train_best = make_lagged_dataset(
    demand_train_scaled, best_time_window)
X_all_tv, Y_all_tv = make_lagged_dataset(combined_tv, best_time_window)
X_val_best = X_all_tv[len(X_train_best):]
Y_val_best = Y_all_tv[len(Y_train_best):]

tf.random.set_seed(42)
final_model = Sequential([
    Input(shape=(best_time_window,)),
    Dense(best_hidden,      activation='relu'),
    Dense(best_hidden // 2, activation='relu'),
    Dense(4)
])
final_model.compile(
    loss='mean_squared_error',
    optimizer=Adam(learning_rate=best_lr)
)

es_final = EarlyStopping(monitor='val_loss', patience=15,
                          restore_best_weights=True, verbose=0)

history = final_model.fit(
    X_train_best, Y_train_best,
    epochs=200,
    batch_size=32,
    validation_data=(X_val_best, Y_val_best),
    callbacks=[es_final],
    verbose=1
)

print(f'\nTraining stopped at epoch {len(history.history["loss"])}')

# ── Training loss curve ────────────────────────────────────────────────────────
plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'],     label='Training Loss',   color='tab:blue')
plt.plot(history.history['val_loss'], label='Validation Loss', color='tab:orange')
plt.title(f'MLP Training Loss  |  '
          f'time_window={best_time_window}, hidden={best_hidden}, lr={best_lr}')
plt.xlabel('Epoch')
plt.ylabel('MSE (scaled)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('output/figures/mlp_demand_training_loss.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()

# ── Validation forecast and RMSE ───────────────────────────────────────────────
context_val = np.concatenate([
    demand_train_scaled[-best_time_window:],
    demand_val_scaled
])

X_val_full, Y_val_full = make_lagged_dataset(context_val, best_time_window)

Y_val_pred_full = scaler_demand.inverse_transform(
    final_model.predict(X_val_full, verbose=0))
Y_val_true_full = scaler_demand.inverse_transform(Y_val_full)

per_origin_mse_val = np.mean((Y_val_pred_full - Y_val_true_full) ** 2, axis=1)
val_rmse_final = math.sqrt(np.mean(per_origin_mse_val))
print(f'\nFinal Model — Validation RMSE: {val_rmse_final:.2f} MW')

# ── Validation forecast plots ──────────────────────────────────────────────────
val_pred_h4       = Y_val_pred_full[:, -1]
val_timestamps_h4 = yc_demand_val.index[3: 3 + len(val_pred_h4)]

plt.figure(figsize=(15, 6))
plt.plot(yc_demand_train, label='Training Demand',
         linewidth=2, color='tab:blue')
plt.plot(yc_demand_val,   label='Validation Demand (Actual)',
         linewidth=2, color='tab:orange')
plt.plot(val_timestamps_h4, val_pred_h4, label='4-step Forecast (MLP)',
         linestyle='--', linewidth=1.5, color='tab:green')
plt.title(f'Training + Validation Demand with 4-Step-Ahead MLP Forecast  '
          f'|  Val RMSE: {val_rmse_final:.2f} MW')
plt.xlabel('Time')
plt.ylabel('Demand (MW)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('output/figures/mlp_demand_forecast_full.png',
            dpi=150, bbox_inches='tight')
plt.show()
plt.close()

plt.figure(figsize=(15, 6))
plt.plot(yc_demand_val,                  label='Actual Demand',
         linewidth=2, color='tab:orange')
plt.plot(val_timestamps_h4, val_pred_h4, label='4-step Forecast (MLP)',
         linestyle='--', linewidth=1.5, color='tab:green')
plt.title(f'Validation Period: Actual vs MLP 4-Step-Ahead Forecast  '
          f'|  RMSE: {val_rmse_final:.2f} MW')
plt.xlabel('Time')
plt.ylabel('Demand (MW)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('output/figures/mlp_demand_forecast_val.png',
            dpi=150, bbox_inches='tight')
plt.show()
plt.close()

# ── Residual diagnostics (validation set) ─────────────────────────────────────
val_true_h4   = Y_val_true_full[:, -1]
residuals_val = pd.Series(
    val_true_h4 - val_pred_h4,
    index=val_timestamps_h4
)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

ax = axes[0, 0]
ax.plot(residuals_val.index, residuals_val.values,
        color='tab:red', linewidth=0.8, alpha=0.8)
ax.axhline(0, color='black', linestyle='--', linewidth=1)
ax.set_title('Residuals over Time')
ax.set_xlabel('Time')
ax.set_ylabel('Residual (MW)')
ax.grid(True, alpha=0.3)
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

ax = axes[0, 1]
ax.hist(residuals_val.values, bins=50,
        color='tab:blue', edgecolor='white', alpha=0.8)
ax.axvline(0, color='black', linestyle='--', linewidth=1)
ax.axvline(residuals_val.mean(), color='tab:red', linestyle='--',
           linewidth=1.5, label=f'Mean = {residuals_val.mean():.1f} MW')
ax.set_title('Distribution of Residuals')
ax.set_xlabel('Residual (MW)')
ax.set_ylabel('Frequency')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
ax.scatter(val_pred_h4, val_true_h4,
           alpha=0.3, s=8, color='tab:purple')
min_v = min(val_pred_h4.min(), val_true_h4.min())
max_v = max(val_pred_h4.max(), val_true_h4.max())
ax.plot([min_v, max_v], [min_v, max_v],
        'k--', linewidth=1.5, label='Perfect forecast')
ax.set_title('Predicted vs Actual (Validation)')
ax.set_xlabel('Predicted Demand (MW)')
ax.set_ylabel('Actual Demand (MW)')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
plot_acf(residuals_val.values, lags=96, ax=ax, alpha=0.05)
ax.set_title('ACF of Residuals')
ax.set_xlabel('Lag')
ax.set_ylabel('ACF')
ax.grid(True, alpha=0.3)

plt.suptitle('MLP Residual Diagnostics — Validation Set', fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('output/figures/mlp_residual_diagnostics_val.png',
            dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

# ── Test set evaluation ────────────────────────────────────────────────────────
context_test = np.concatenate([
    demand_val_scaled[-best_time_window:],
    demand_test_scaled
])

X_test_full, Y_test_full = make_lagged_dataset(context_test, best_time_window)

Y_test_pred_full = scaler_demand.inverse_transform(
    final_model.predict(X_test_full, verbose=0))
Y_test_true_full = scaler_demand.inverse_transform(Y_test_full)

per_origin_mse_test = np.mean((Y_test_pred_full - Y_test_true_full) ** 2, axis=1)
test_rmse = math.sqrt(np.mean(per_origin_mse_test))

print(f'\nFinal Model Performance:')
print(f'  Validation RMSE: {val_rmse_final:.2f} MW')
print(f'  Test RMSE:       {test_rmse:.2f} MW')

# ── Refit on train + val for competition forecast ──────────────────────────────
combined_tv_full = np.concatenate([demand_train_scaled, demand_val_scaled])
X_full, Y_full   = make_lagged_dataset(combined_tv_full, best_time_window)

tf.random.set_seed(42)
final_model_comp = Sequential([
    Input(shape=(best_time_window,)),
    Dense(best_hidden,      activation='relu'),
    Dense(best_hidden // 2, activation='relu'),
    Dense(4)
])
final_model_comp.compile(
    loss='mean_squared_error',
    optimizer=Adam(learning_rate=best_lr)
)

es_comp = EarlyStopping(monitor='loss', patience=15,
                         restore_best_weights=True, verbose=0)

final_model_comp.fit(
    X_full, Y_full,
    epochs=200,
    batch_size=32,
    callbacks=[es_comp],
    verbose=1
)


final_model_comp.save("best_demand.keras")

# ── Competition forecast (T+1 to T+4) ─────────────────────────────────────────
all_demand_scaled = scaler_demand.transform(
    np.concatenate([demand_train_raw,
                    demand_val_raw,
                    demand_test_raw]).reshape(-1, 1)
).flatten()

last_window  = all_demand_scaled[-best_time_window:].reshape(1, -1)
comp_pred_sc = final_model_comp.predict(last_window, verbose=0)
comp_pred_mw = scaler_demand.inverse_transform(comp_pred_sc).flatten()

comp_timestamps = pd.date_range(
    start='2026-02-10 00:00:00',
    periods=4,
    freq='15min'
)

competition_df = pd.DataFrame({
    'SETTLEMENTDATE': comp_timestamps,
    'TOTALDEMAND':    np.round(comp_pred_mw, 2)
})

print('\nCompetition Forecast (Demand):')
print(competition_df.to_string(index=False))


# ── Test set forecast plot ─────────────────────────────────────────────────────
test_pred_h4       = Y_test_pred_full[:, -1]
test_true_h4       = Y_test_true_full[:, -1]
test_timestamps_h4 = yc_demand_test.index[3: 3 + len(test_pred_h4)]

plt.figure(figsize=(15, 6))
plt.plot(yc_demand_test,              label='Actual Demand (Test)',
         linewidth=2, color='tab:orange')
plt.plot(test_timestamps_h4, test_pred_h4, label='4-step Forecast (MLP)',
         linestyle='--', linewidth=1.5, color='tab:green')
plt.title(f'Test Period: Actual vs MLP 4-Step-Ahead Forecast  '
          f'|  Test RMSE: {test_rmse:.2f} MW')
plt.xlabel('Time')
plt.ylabel('Demand (MW)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('output/figures/mlp_demand_forecast_test.png',
            dpi=150, bbox_inches='tight')
plt.show()
plt.close()



In [ ]:

final_competition_df = competition_df.merge(
    rrp_forecast_df,
    on="SETTLEMENTDATE",
    how="inner"
)

final_competition_df = final_competition_df[
    ["SETTLEMENTDATE", "TOTALDEMAND", "RRP"]
]

# Force exact required timestamps
required_timestamps = pd.to_datetime([
    "2026-02-10 00:00:00",
    "2026-02-10 00:15:00",
    "2026-02-10 00:30:00",
    "2026-02-10 00:45:00"
])

final_competition_df["SETTLEMENTDATE"] = pd.to_datetime(
    final_competition_df["SETTLEMENTDATE"]
)

final_competition_df = final_competition_df.set_index("SETTLEMENTDATE")
final_competition_df = final_competition_df.loc[required_timestamps]
final_competition_df = final_competition_df.reset_index()

# Rename index column back
final_competition_df = final_competition_df.rename(
    columns={"index": "SETTLEMENTDATE"}
)

# Round forecasts
final_competition_df["TOTALDEMAND"] = final_competition_df["TOTALDEMAND"].round(2)
final_competition_df["RRP"] = final_competition_df["RRP"].round(2)

# Safety checks
assert list(final_competition_df.columns) == ["SETTLEMENTDATE", "TOTALDEMAND", "RRP"]
assert final_competition_df.shape == (4, 3)
assert final_competition_df["SETTLEMENTDATE"].tolist() == list(required_timestamps)

output_csv = "next_hour_forecasts.csv"

final_competition_df.to_csv(output_csv, index=False)

print(final_competition_df)
print("Saved:", output_csv)